
<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; color: #000099;">
    <div style="text-align: center;">
        <span style="font-weight: bold; font-size: 18px; letter-spacing: 1px; text-transform: uppercase;">
            Techno-Economic Parameters Data Prossecing
        </span>
        <br>
        <span style="font-weight: bold; font-size: 28px; font-style: italic; letter-spacing: 5px;">
            Sector Power | Sector X (VPP Resisitive Units) <br>
            POWER PLANTS
        </span>
        <br>
        <span style="font-weight: bold; font-size: 18px;">
            Main Formatting Notebook
        </span>
    </div>
    <table style="margin: 20px auto; color: #000099; border-collapse: collapse; text-align: center;">
        <tr style="background-color: #E6E6E6;">
            <td style="font-size: 14px; padding: 10px 40px; opacity: 0.8;"><b>FROM</b></td>
            <td style="font-size: 14px; padding: 10px 40px; opacity: 0.8;"><b>TO</b></td>
        </tr>
        <tr style="background-color: #CCCCCC; font-size: 22px; font-style: italic; font-weight: bold;">
            <td style="padding: 5px 40px 15px 40px;">PYPSA | négaWatt</td>
            <td style="padding: 5px 40px 15px 40px;">DISPA-SET | Unleash</td>
        </tr>
    </table>
    <div style="border-top: 1px solid #000099; padding: 10px">
        This notebook processes techno-economic information extracted from <b>PyPSA</b> simulation outputs and converts the resulting infrastructure, operational, and demand-related datasets into formats compatible with the <b>Dispa-SET Unleash</b> framework.
    </div>
</div>

In [1]:
                            import os
                            import csv
from datetime               import datetime
                            import requests
                            import pandas                      as pd
from shutil                 import move
                            import numpy                       as np
                            import shutil
from bs4                    import BeautifulSoup
                            import re
                            import io
                            import plotly.graph_objects        as go
from typing                 import List, Dict, Tuple
                            import re
from IPython.display        import HTML
from difflib                import get_close_matches
from pathlib                import Path
                            import json
                            import difflib
from rapidfuzz              import process, fuzz
from pprint                 import pprint
                            import shutil

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: right; color: #000099;">
        <span style="text-transform: uppercase;">
        <b>Auxiliary Code</b>
        </span>
            <div style="border-top: 1px solid #000099; padding: 10px">
            This cell facilitates the creation of project directories corresponding to
            </div>
        <b>ENTSO-E member countries</b>. 
        <br>
        <span style="font-style: italic; opacity: 0.8;">
        Note: Uncomment the code below only if a fresh directory structure is required.</span>
    </div>
</div>

In [2]:
"""
# List of countries with their acronyms in parentheses
countries = [
    "Albania       (AL)", "Armenia        (AM)", "Austria          (AT)", "Azerbaijan (AZ)",
    "Belarus       (BY)", "Belgium        (BE)", "Bosnia and Herz. (BA)", "Bulgaria   (BG)",
    "Croatia       (HR)", "Cyprus         (CY)", "Czech Republic   (CZ)", "Denmark    (DK)",
    "Estonia       (EE)", "Finland        (FI)", "France           (FR)", "Georgia    (GE)",
    "Germany       (DE)", "Greece         (EL)", "Hungary          (HU)", "Iceland    (IS)",
    "Ireland       (IE)", "Italy          (IT)", "Kosovo           (XK)", "Latvia     (LV)",
    "Lithuania     (LT)", "Luxembourg     (LU)", "Malta            (MT)", "Moldova    (MD)",
    "Montenegro    (ME)", "Netherlands    (NL)", "North Macedonia  (MK)", "Norway     (NO)",
    "Poland        (PL)", "Portugal       (PT)", "Romania          (RO)", "Russia     (RU)",
    "Russia Legacy (RU)", "Serbia         (RS)", "Slovakia         (SK)", "Slovenia   (SI)",
    "Spain         (ES)", "Sweden         (SE)", "Switzerland      (CH)", "Turkey     (TR)",
    "Ukraine       (UA)", "United Kingdom (UK)"
]
# Set the path where you want to create the folders
base_path = '/home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/PowerPlants'
# Ensure the base path exists
os.makedirs(base_path, exist_ok=True)
# Loop through the list of countries
for country_string in countries:
    # Use a regular expression to find the acronym inside the parentheses
    match = re.search(r'\((.*?)\)', country_string)
    # If a match is found, extract the acronym
    if match:
        acronym = match.group(1).strip()  # Use strip() to remove any extra whitespace
        folder_path = os.path.join(base_path, acronym)
        # Check if the folder already exists to avoid errors
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)
            print(f"Created folder: {folder_path}")
        else:
            print(f"Folder already exists: {folder_path}")
    else:
        print(f"Could not extract acronym from: {country_string}")
print("\nAll folders created successfully!")
"""

<>:1: SyntaxWarning: invalid escape sequence '\('
<>:1: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_2803280/3390782925.py:1: SyntaxWarning: invalid escape sequence '\('
  """


'\n# List of countries with their acronyms in parentheses\ncountries = [\n    "Albania       (AL)", "Armenia        (AM)", "Austria          (AT)", "Azerbaijan (AZ)",\n    "Belarus       (BY)", "Belgium        (BE)", "Bosnia and Herz. (BA)", "Bulgaria   (BG)",\n    "Croatia       (HR)", "Cyprus         (CY)", "Czech Republic   (CZ)", "Denmark    (DK)",\n    "Estonia       (EE)", "Finland        (FI)", "France           (FR)", "Georgia    (GE)",\n    "Germany       (DE)", "Greece         (EL)", "Hungary          (HU)", "Iceland    (IS)",\n    "Ireland       (IE)", "Italy          (IT)", "Kosovo           (XK)", "Latvia     (LV)",\n    "Lithuania     (LT)", "Luxembourg     (LU)", "Malta            (MT)", "Moldova    (MD)",\n    "Montenegro    (ME)", "Netherlands    (NL)", "North Macedonia  (MK)", "Norway     (NO)",\n    "Poland        (PL)", "Portugal       (PT)", "Romania          (RO)", "Russia     (RU)",\n    "Russia Legacy (RU)", "Serbia         (RS)", "Slovakia         (SK)", "Slove

<div style= "background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099; " >
 <span style= "font-weight: bold; font-size: 16px; " >
Section A Overview: Configuration & Environment Setup
 </span >
 <div style= "border-top: 1px solid #000099; padding: 10px; " >
This section establishes the foundational configuration for the data processing pipeline.<br> It defines the directory structures, selects the target PyPSA scenario (e.g., <i>Sufficiency_Scenario</i>), configures the Virtual Power Plant (VPP) mode flag, and specifies the geographical zones and target year for the simulation.
 </div >
 </div >
 <div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
    A-01. Dispa-SET Unleash Folder Path
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
    This step dynamically determines the <b>zone_folder_path</b> by locating the <i>"Dispa-SET_Unleash"</i> directory relative to your current workspace. 
    </div>
</div>

In [3]:
### Get the directory two levels up from the current working directory
dispaSET_unleash_folder_path = Path.cwd().parent.parent
print("dispaSET_unleash_folder_path:", dispaSET_unleash_folder_path)

dispaSET_unleash_folder_path: /home/ray/Dispa-SET_Unleash


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        A-02. PyPSA Source Scenario
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        There are two primary scenarios available as sources for the PyPSA raw data:<br>
            <div style="margin-left: 2em; font-size: 12px;">
    <li> <b> Reference_Scenario   </b> </li>
    <li> <b> Sufficiency_Scenario </b> </li>
    </div>
</div>

In [4]:
### Set the configuration
# =============================================================================
#pypsa_scenario = "Reference_Scenario"
pypsa_scenario = "Sufficiency_Scenario"
# =============================================================================
print(f"PyPSA Chosen Scenario: {pypsa_scenario}")

PyPSA Chosen Scenario: Sufficiency_Scenario


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        A-03. VPP (Virtual Power Plant) Mode Flag
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        The <b>VPP flag</b> controls whether the model operates in <b>Virtual Power Plant mode</b>. When enabled, additional carriers (e.g., resistive heaters) are activated to model sector-coupling and demand-side flexibility.<br>
    <b>Flag Logic:</b>
    <div style="margin-left: 4em; font-size: 12px">
         <li> <b>VPP = True</b>: Activates VPP mode, appending resistive heater carriers (e.g., <b>residential rural resistive heater</b>) to the <b>power_units_carriers</b> list.<br>
         <li> <b>VPP = False</b>: Disables VPP mode, limiting the model to core generation and storage technologies.</div>
    <div style="margin-left: 2em; font-size: 12px">
        The flag is used throughout the workflow to conditionally include sector-coupling components and enable multi-energy system modeling.<br>
        Toggling this flag allows for rapid scenario switching between a pure power-sector model and a fully integrated multi-energy system with heat, transport, and industry components.
    </div>
</div>

In [5]:
# VPP FLAG
# =============================================================================
VPP = True
#VPP = False
# =============================================================================

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
            A-04. Secondary Folders Path
        </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        The internal architecture of the <i>Dispa-SET Unleash</i> directory contains several subfolders that must be mapped to specific path variables for accurate data retrieval.
    </div>
</div>

In [6]:
### Convert the string path into a Path object
dispaSET_unleash_folder_path = Path(dispaSET_unleash_folder_path)
### Base Power Plants Data
power_plants_base_data_folder_path = dispaSET_unleash_folder_path / "Database" / "PowerPlants"
print(f"{'power_plants_base_data_folder_path:':<55} {power_plants_base_data_folder_path}")
print("—" * 140)
### PyPSA Raw Data
power_plants_pypsa_raw_data_folder_path = (
    dispaSET_unleash_folder_path / "RawData_PyPSA" / pypsa_scenario / "PowerPlants"
)
print(f"{'power_plants_pypsa_raw_data_folder_path:':<55} {power_plants_pypsa_raw_data_folder_path}")
print("—" * 140)
### PyPSA Formatted Data
power_plants_pypsa_formated_data_folder_path = (
    dispaSET_unleash_folder_path / "Database_PyPSA" / pypsa_scenario / "PowerPlants"
)
print(f"{'power_plants_pypsa_formated_data_folder_path:':<55} {power_plants_pypsa_formated_data_folder_path}")
print("—" * 140)
### PyPSA Power Units Formatted Data with VPP
vpp_scenario_folder = f"{pypsa_scenario}________VPP" 
vpp_power_plants_pypsa_formated_data_folder_path = (
    dispaSET_unleash_folder_path / "Database_PyPSA" / vpp_scenario_folder / "PowerPlants"
)
print(f"{'vpp_power_plants_pypsa_formated_data_folder_path:':<55} {vpp_power_plants_pypsa_formated_data_folder_path}")
print("—" * 140)
### PyPSA Data Formatted Data with VPP
vpp_scenario_folder = f"{pypsa_scenario}________VPP" 
vpp_data_pypsa_formated_data_folder_path = (
    dispaSET_unleash_folder_path / "Database_PyPSA" / vpp_scenario_folder / "VPP_data"
)
print(f"{'vpp_data_pypsa_formated_data_folder_path:':<55} {vpp_data_pypsa_formated_data_folder_path}")
print("—" * 140)
### Base VPP Power Plants Data File
vpp_power_plants_base_data_folder_path = (
    dispaSET_unleash_folder_path / "Database" / "PowerPlants" / "BE" / "2023_VPP.csv"
)
print(f"{'vpp_power_plants_base_data_folder_path:':<55} {vpp_power_plants_base_data_folder_path}")
print("—" * 140)
### Base VPP Features Data File
vpp_features_base_data_folder_path = (
    dispaSET_unleash_folder_path / "Database" / "Boundary_Sector" / "BoundarySectorData" / "BE" / "2023.csv"
)
print(f"{'vpp_features_base_data_folder_path:':<55} {vpp_features_base_data_folder_path}")

power_plants_base_data_folder_path:                     /home/ray/Dispa-SET_Unleash/Database/PowerPlants
————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
power_plants_pypsa_raw_data_folder_path:                /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Sufficiency_Scenario/PowerPlants
————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
power_plants_pypsa_formated_data_folder_path:           /home/ray/Dispa-SET_Unleash/Database_PyPSA/Sufficiency_Scenario/PowerPlants
————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
vpp_power_plants_pypsa_formated_data_folder_path:       /home/ray/Dispa-SET_Unleash/Database_PyPSA/Sufficiency_Scenario________VPP/PowerPlants
——————————————————————————————————————————————————————————————————

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        A-05. Zone(s) Configuration
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        Define the target geographical zone(s) for data processing. This selection determines which regional datasets will be accessed and where the resulting formatted files will be stored.
        <br>
        Use <b>ISO 3166-1 alpha-2</b> standard codes for European countries (e.g., <i>AT, BE, BG, CH, DE, FR</i>). 
    </div>
</div>

In [7]:
# =============================================================================
all_zones = [
    "AL", "AM", "AT", "AZ", "BY", "BE", "BA", "BG", "HR", "CY", "CZ", "DK",
    "EE", "FI", "FR", "GE", "DE", "EL", "HU", "IS", "IE", "IT", "XK", "LV",
    "LT", "LU", "MT", "MD", "ME", "NL", "MK", "NO", "PL", "PT", "RO", "RU",
    "RS", "SK", "SI", "ES", "SE", "CH", "TR", "UA", "UK"
]
# =============================================================================
active_selection = {"BE", "FR", "DE", "NL", "UK"}
# =============================================================================
vpp_active_selection = {"BE"}
# =============================================================================
filter_zones = lambda selected: [z for z in all_zones if z in selected]
zone_names = filter_zones(active_selection)
vpp_zone_names = filter_zones(vpp_active_selection)
print("Selected Zones:", zone_names)
print("VPP Selected Zones:", vpp_zone_names)

Selected Zones: ['BE', 'FR', 'DE', 'NL', 'UK']
VPP Selected Zones: ['BE']


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: right; color: #000099;">
    <span style="text-transform: uppercase;">
    <b>International Nomenclature</b>
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px">
        The following dictionary maps <b>standardized country codes</b> to their various synonyms, aliases, or alternative naming conventions used across international databases
    <br>
    (e.g., ENTSO-E, Eurostat, or ISO variants).
    <br>
    <b>Purpose:</b> This ensures the script can correctly identify and link data even when source files use inconsistent regional identifiers (e.g., <i>UK</i> vs. <i>GB</i> or <i>EL</i> vs. <i>GR</i>).
    </div>
</div>

In [8]:
### Define a clean base dictionary (no messy padding or unnecessary lists)
# =============================================================================
raw_countries = {
    "AL": ("Albania",            ""),  "AM": ("Armenia",          ""),  "AT": ("Austria",          ""),
    "AZ": ("Azerbaijan",         ""),  "BY": ("Belarus",          ""),  "BE": ("Belgium",          ""),
    "BA": ("Bosnia and Herz.",   ""),  "BG": ("Bulgaria",         ""),  "HR": ("Croatia",          ""),
    "CY": ("Cyprus",             ""),  "CZ": ("Czech Republic",   ""),  "DK": ("Denmark",          ""),
    "EE": ("Estonia",            ""),  "FI": ("Finland",          ""),  "FR": ("France",           ""),
    "GE": ("Georgia",            ""),  "DE": ("Germany",          ""),  "EL": ("Greece",         "GR"),
    "HU": ("Hungary",            ""),  "IS": ("Iceland",          ""),  "IE": ("Ireland",          ""),
    "IT": ("Italy",              ""),  "XK": ("Kosovo",           ""),  "LV": ("Latvia",           ""),
    "LT": ("Lithuania",          ""),  "LU": ("Luxembourg",       ""),  "MT": ("Malta",            ""),
    "MD": ("Moldova",            ""),  "ME": ("Montenegro",       ""),  "NL": ("Netherlands",      ""),
    "MK": ("North Macedonia",    ""),  "NO": ("Norway",           ""),  "PL": ("Poland",           ""),
    "PT": ("Portugal",           ""),  "RO": ("Romania",          ""),  "RU": ("Russia",           ""),
    "RS": ("Serbia",             ""),  "SK": ("Slovakia",         ""),  "SI": ("Slovenia",         ""),
    "ES": ("Spain",              ""),  "SE": ("Sweden",           ""),  "CH": ("Switzerland",      ""),
    "TR": ("Turkey",             ""),  "UA": ("Ukraine",          ""),  "UK": ("United Kingdom", "GB")
}
# =============================================================================
### Constructing dicctionary
zone_names_equivalences_dict = {
    code: {"Acronym": [acronym if acronym else " "], "name": [name]}
    for code, (name, acronym) in raw_countries.items()
}
### Filter the dictionary using only the keys present in zone_names
selected_zone_names_equivalences_dict = {
    k: zone_names_equivalences_dict[k] 
    for k in zone_names 
    if k in zone_names_equivalences_dict
}
### Print only the zones that have a valid alternative acronym
for key, value in selected_zone_names_equivalences_dict.items():
    acronym = value["Acronym"][0].strip()
    if acronym:  # Evaluates to True only if the acronym is not empty or spaces
        print(f"Key: {key};    Acronym: {value['Acronym']};    Name: {value['name']}\n")

Key: UK;    Acronym: ['GB'];    Name: ['United Kingdom']



<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        A-06. Data Reference Year
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
            Define the <b>target temporal scope</b> for the processing workflow. This variable determines which yearly dataset will be filtered, formatted, and exported.
    </div>
</div>

In [9]:
### Year to which data is formatting to
# =============================================================================
data_target_year = "2030"
#data_target_year = "2040"
#data_target_year = "2050"
# =============================================================================
print(f"Selected Year: {data_target_year}")

Selected Year: 2030


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: right; color: #000099;">
    <span style="text-transform: uppercase;">
    <b>Tracking & Validation Variables</b>
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px">
        These cells serve to <b>verify and audit</b> the current configuration, confirming file names, directory paths, and scenario-specific metadata.
    <br>
    <span style="font-style: italic; opacity: 0.8;">
        By maintaining these variables, the notebook ensures consistent data flow to subsequent modules without the need for redundant manual entries.
    </span>
    </div>
</div>

In [10]:
### Print the relevant information with automatic 55-character left alignment
print(f"{'Path to the DispaSET Unleash folder:':<55} {dispaSET_unleash_folder_path}")
print("—" * 140)
print(f"{'Path to the Power Plants Base data folder:':<55} {power_plants_base_data_folder_path}")
print("—" * 140)
print(f"{'Path to the Power Plants_Pypsa Raw data folder:':<55} {power_plants_pypsa_raw_data_folder_path}")
print("—" * 140)
print(f"{'Path to the Power Plants_Pypsa Formated data folder:':<55} {power_plants_pypsa_formated_data_folder_path}")
print("—" * 140)
print(f"{'Name of the zones:':<55} {zone_names}")
print("—" * 140)
print(f"{'Name of the zone_names_equivalences_dictionary:':<55} {[v['Acronym'] for v in selected_zone_names_equivalences_dict.values()]}")
print("—" * 140)
print(f"{'Target year:':<55} {data_target_year}")

Path to the DispaSET Unleash folder:                    /home/ray/Dispa-SET_Unleash
————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
Path to the Power Plants Base data folder:              /home/ray/Dispa-SET_Unleash/Database/PowerPlants
————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
Path to the Power Plants_Pypsa Raw data folder:         /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Sufficiency_Scenario/PowerPlants
————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
Path to the Power Plants_Pypsa Formated data folder:    /home/ray/Dispa-SET_Unleash/Database_PyPSA/Sufficiency_Scenario/PowerPlants
—————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————

<div style= "background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099; " >
 <span style= "font-weight: bold; font-size: 16px; " >
Section B Overview: Nomenclature, Mapping & Data Sources
 </span >
 <div style= "border-top: 1px solid #000099; padding: 10px; " >
This section builds the critical translation layer between PyPSA and Dispa-SET.<br> It defines the equivalence dictionaries for technologies and fuels, loads country-specific fuel-technology proportion matrices, and provides a comprehensive literature review of the data sources used to calibrate the European energy system parameters.
 </div >
 </div >
<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        B-01. Technology Nomenclature Mapping
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This section defines the <b>equivalence table</b> between PyPSA and Dispa-SET technology identifiers. <br>Accurate cross-model mapping is essential to ensure that generation assets are assigned the correct techno-economic parameters.
    <div style="margin-left: 2em; font-size: 12px">
    <b>Source Reference:</b> The Dispa-SET technology identifiers are synchronized with the 
        <a href="https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py" style="text-decoration: underline; font-weight: bold;">
        official toolkit repository
        </a>.
        <br>
        <i>Note: Ensure the dictionary values match the standardized nomenclature to avoid Dispa-SET simulation errors during the data import phase.</i>
    </div>
    </div>
</div>

In [11]:
### Define all the Dispa-SET technology lists from the common.py script of Dispa-SET core scripts 
# =============================================================================
tech_master_list        =  []
tech_renewables         =  ['HROR' , 'PHOT' , 'WAVE' , 'WTOF' , 'WTON' , 'SOTH']
tech_conventional       =  ['HDAM' , 'COMC' , 'GTUR' , 'STUR' , 'BATS' , 'ICEN']
tech_batteries          =  ['BATS']
tech_storage            =  ['BATS' , 'HDAM' , 'HPHS' , 'BEVS' , 'CAES' , 'SCSP' , 'H2ST' , 'HPHSC', 'THMS']
tech_p2bs               =  ['P2GS' , 'ALKE' , 'PEME' , 'SOXE' , 'P2BS' , 'PEFC' , 'DMFC' , 'ALFC' , 'PAFC' , 'MCFC' , 'SOFC' ,
                            'REFC' , 'HDAMC', 'HRORC', 'HDLZ' , 'COMCX', 'GTURX', 'ICENX', 'STURX', 'P2HT' , 'ASHP' , 'GSHP' , 
                            'HYHP' , 'WSHP' , 'REHE']
tech_bs2p               =  ['BSPG']
tech_boundary_sector    =  ['BSPG' , 'GETH' , 'HOBO' , 'SOTH' , 'ABHP' , 'HOBOX', 'P2BS' , 'HBBS' , 'WHEN']
# =============================================================================
### Combine unique values using set union and sort them directly into a list
all_technologies_list = sorted(
    set(tech_master_list) | 
    set(tech_renewables) | 
    set(tech_conventional) | 
    set(tech_batteries) | 
    set(tech_storage) | 
    set(tech_p2bs) | 
    set(tech_bs2p) | 
    set(tech_boundary_sector)
)
### Create the DataFrame
dispaSET_tech_list = pd.DataFrame(all_technologies_list, columns=["Dispa-SET Technologies"])
print(dispaSET_tech_list.columns)
dispaSET_tech_list

Index(['Dispa-SET Technologies'], dtype='object')


,Dispa-SET Technologies
0,ABHP
1,ALFC
2,ALKE
3,ASHP
4,BATS
5,BEVS
6,BSPG
7,CAES
8,COMC
9,COMCX


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       B-02. Dispa-SET Fuel Nomenclature
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        Similar to the technology mapping, the <b>fuel nomenclature</b> must be strictly aligned with the Dispa-SET standards to ensure correct calculation of emissions and fuel costs.
        <div style="margin-left: 2em; font-size: 12px">
    <b> Source Reference:</b> Fuel identifiers are derived from the
    <a href="https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py" style="text-decoration: underline; font-weight: bold;">
        common.py configuration file
    </a> within the official repository.
    <br>
    <i>Requirement: All fuel types (e.g., Natural Gas, Biomass, Lignite) must be translated from the PyPSA input names to the corresponding Dispa-SET strings.</i>
        </div>
    </div>
</div>

In [12]:
### Define all the Dispa-SET fuel lists
# =============================================================================
fuels_list = [
    "AIR", "AMO", "BIO", "GAS", "HRD", "LIG", "NUC", "OIL", "PEA", "SUN", 
    "WAT", "WIN", "WST", "OTH", "GEO", "HYD", "WHT", "ELE", "THE", "UNK"
]
# =============================================================================
### Create a DataFrame with a separate, distinct name
dispaSET_fuel_df = pd.DataFrame(fuels_list, columns=["Dispa-SET Fuels"])
print(list(dispaSET_fuel_df.columns))
dispaSET_fuel_df

['Dispa-SET Fuels']


,Dispa-SET Fuels
0,AIR
1,AMO
2,BIO
3,GAS
4,HRD
5,LIG
6,NUC
7,OIL
8,PEA
9,SUN


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: left; color: #000099;">
    <span style="font-weight: bold;">
        B-03. PyPSA vs. Dispa-SET Nomenclature
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px">
        To ensure model interoperability, technologies from PyPSA must be mapped to their functional equivalents within Dispa-SET. This process identifies compatible assets and filters out elements that cannot be represented due to structural differences.
    </div>
    <span style="font-weight: bold;">
        B-03.1. PyPSA Energy System Flow Diagram
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
The following chart illustrates the internal structure of the PyPSA Energy sector, serving as the baseline for the conversion logic.
    </div>
    <div style="text-align: center; margin: 15px 0;">
            <img src="Images/PyPSA_multisector_figure_1.png" alt="PyPSA Multisector Flow Diagram" style="max-width:35%; height: auto;">
            <div style="margin-top: 8px; font-size: 10px; ">
                Source: 
            <a href="https://pypsa-eur.readthedocs.io/" target="_blank";"> PyPSA-Eur Documentation</a>
    </div>
    </div>
    <span style="font-weight: bold;">
        B-03.2. Equivalent Dispa-SET Flow Logic
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
A correlation is established to map PyPSA parameters into the Dispa-SET environment.
    </div>
            <img src="Images/PyPSA_sectors_as_Dispaset_Flow_work.svg" alt="Equivalent Dispa-SET Diagram" style="width: 100%; height: auto;">
        <div style="text-align: center; margin: 15px 0; margin-top: 8px; font-size: 10px; ">
        Source: Adapted for Dispa-SET_Unleash
    </div>
    <div style="padding: 10px;">
        Due to framework limitations, only compatible technologies are retained.
    </div>
            <img src="Images/PyPSA_sectors_as_Dispaset_Flow_work_Filtered_1.svg" alt="Filtered Dispa-SET Diagram" style="width: 100%; height: auto;">
    <div style="text-align: center; margin: 15px 0; margin-top: 8px; font-size: 10px; ">
        Source: Adapted for Dispa-SET_Unleash
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: left; color: #000099;">
    <span style="font-weight: bold;">
        B-03.3. Technology & Demand Correlation Table
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        The PyPSA technologies and demands nomenclature and their correlation with Dispa-SET are described below:
    </div>
    <!-- Table -->
    <div style="overflow-x: auto; margin: 15px 10px;">
        <table style="width: 100%; border-collapse: collapse; font-family: 'Times New Roman', serif; font-size: 12px; color: #000099;">
            <colgroup>
                <col style="width: 8%;">
                <col style="width: 14%;">
                <col style="width: 8%;">
                <col style="width: 28%;">
                <col style="width: 8%;">
                <col style="width: 20%;">
                <col style="width: 6%;">
                <col style="width: 8%;">
            </colgroup>
            <thead>
                <tr style="background-color: #E6E6E6;">
                    <th colspan="4" style="text-align: center; padding: 8px;">PyPSA</th>
                    <th style="width: 5%;"></th>
                    <th colspan="3" style="text-align: center; padding: 8px;">Dispa-SET</th>
                </tr>
                <tr style="background-color: #CCCCCC;">
                    <th style="padding: 8px; text-align: left; border-bottom: 1px solid #000033;">PyPSA Element</th>
                    <th style="padding: 8px; text-align: left; border-bottom: 1px solid #000033;">Technology</th>
                    <th style="padding: 8px; text-align: left; border-bottom: 1px solid #000033;">Sector Classification</th>
                    <th style="padding: 8px; text-align: left; border-bottom: 1px solid #000033;">Description</th>
                    <th style="padding: 8px; text-align: center; border-bottom: 1px solid #000033;">Sector Relation</th>
                    <th style="padding: 8px; text-align: left; border-bottom: 1px solid #000033;">Dispa-SET Element</th>
                    <th style="padding: 8px; text-align: left; border-bottom: 1px solid #000033;">Element Type</th>
                    <th style="padding: 8px; text-align: left; border-bottom: 1px solid #000033;">May Modeled?</th>
                </tr>
            </thead>
            <tbody>
                <!-- Table rows remain the same as in your original code -->
                <tr style="background-color: #E6E6E6;">
                    <td style="padding: 8px;">link</td>
                    <td style="padding: 8px;">DC</td>
                    <td style="padding: 8px;">Sector Power</td>
                    <td style="padding: 8px;">Represents the DC (HVDC) transmission network for electricity</td>
                    <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P<br>P / P</td>
                    <td style="padding: 8px;">NTC / NTC</td>
                    <td style="padding: 8px;">input / input</td>
                    <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">OCGT</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Open-Cycle Gas Turbine producing electricity from gas.</td>
                  <td style="padding: 8px;border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2(P\&X) <br> P / P</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">CCGT</td>
                  <td style="padding: 8px;">Power Sector</td>
                  <td style="padding: 8px;">Combined-Cycle Gas Turbine</td>
                  <td style="padding: 8px;border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2(P\&X) <br> P / P</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">EV charger</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Interface between the grid and electric vehicles</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P2S <br> P&S / P&S</td>
                  <td style="padding: 8px;">PowerCapacity, STOCapacity STOMaxChargingPower / --- id. ---</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">V2G</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Vehicle-to-Grid—allows EVs to discharge electricity</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">S2P <br> P&S / P&S</td>
                  <td style="padding: 8px;">STOMaxChargingPower PowerCapacity, STOCapacity / - id. -</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">battery charger</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Converts electricity to stored energy in batteries (charging link)</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P2S <br> P&S / P&S</td>
                  <td style="padding: 8px;">PowerCapacity, STOCapacity STOMaxChargingPower / --- id. ---</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">BioSNG</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Synthetic Nat. Gas. (bio-methane)-fuel synthesis process</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2(X&X) <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">DAC</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Direct Air Capture—captures CO<sub>2</sub> for storage or utilization</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">(HT&P)2X <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">Fischer-Tropsch</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Converts H<sub>2</sub> + CO<sub>2</sub> to liquid hydrocarbons; fuel synthesis</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">(X&X)2X <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">H<sub>2</sub> Electrolysis</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Electricity into hydrogen cross-sector conversion</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P2X <br> P2X / P2X</td>
                  <td style="padding: 8px;">PowerCapacity, STOCapacity, STOMaxPower / --- id. ---</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">H<sub>2</sub> Fuel Cell</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Converts hydrogen back to electricity</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2P <br> X2P / X2P</td>
                  <td style="padding: 8px;">STOMaxPower, <br> PowerCapacity, STOCapacity / - id. -</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">H<sub>2</sub> pipeline</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Transports hydrogen between regions or sectors</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> X / X</td>
                  <td style="padding: 8px;">BS_NTC / BS_NTC</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">H<sub>2</sub> pipeline retrofitted</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Existing pipelines adapted for H<sub>2</sub> transport</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> X / X</td>
                  <td style="padding: 8px;">BS_NTC / BS_NTC</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">H<sub>2</sub> turbine</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Generates electricity or heat from hydrogen—boundary technology</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2P <br> X2P / X2P</td>
                  <td style="padding: 8px;">STOMaxPower, <br> PowerCapacity, STOCapacity / - id. -</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">Haber-Bosch</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Converts H<sub>2</sub> + N<sub>2</sub> into ammonia—chemical / fertilizer Ind.</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">(P&X)2X <br> P2X / P2X</td>
                  <td style="padding: 8px;">PowerCapacity, STOCapacity, STOMaxPower / --- id. ---</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">SMR</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Steam Methane Reforming—gas to hydrogen conversion</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2(X&X) <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">SMR CC</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">SMR with Carbon Capture</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2(X&X) <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">Sabatier</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">H<sub>2</sub> + CO<sub>2</sub> → CH<sub>4</sub>—synthetic methane production</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">(X&X)2X <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">agriculture machinery oil</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Oil used in agricultural machinery—transport/fuel</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2X <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">ammonia cracker</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Converts ammonia back into hydrogen</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">(P&X)2X <br> P2X / P2X</td>
                  <td style="padding: 8px;">PowerCapacity, STOCapacity, STOMaxPower / --- id. ---</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">battery discharger</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Discharges stored electricity from batteries</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">S2P <br> P&S / P&S</td>
                  <td style="padding: 8px;">STOMaxChargingPower PowerCapacity, STOCapacity / - id. -</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">electricity distribution grid</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Represents distribution-level power flow—low voltage</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">coal</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Represents coal-based electricity generation</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2(P&X) <br> P / P</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">lignite</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Represents coal-based electricity generation or conversion node</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">S2(P&X) <br> P / P</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">nuclear</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Nuclear-to-electricity conversion within the power system.</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">S2P <br> P / P</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">oil</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Oil-fired electricity generation or conversion node</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2(P&X) <br> P / P</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">biogas to gas</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Upgrades raw biogas into pipeline-quality methane</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2(X&X) <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">biogas to gas CC</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Biogas upgrading with carbon capture—industrial fuel conversion</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2(X&X) <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">biomass to liquid</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Converts biomass into liquid fuels—synthetic fuel process.</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2(X&X) <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">CO<sub>2</sub> sequestered</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Represents captured Tons of CO<sub>2</sub> / hour transported or stored</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">gas for industry CC</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Industrial gas use with carbon capture</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">gas pipeline</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Transports natural gas—energy carrier infrastructure</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> ---- / ----</td>
                  <td style="padding: 8px;">BS_NTC</td>
                  <td style="padding: 8px;">input</td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">gas pipeline new</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Expansion of natural gas transport capacity</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> ---- / ----</td>
                  <td style="padding: 8px;">BS_NTC</td>
                  <td style="padding: 8px;">input</td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">methanolisation</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Synthesizes methanol (CO<sub>2</sub> + H<sub>2</sub> → CH<sub>3</sub>OH)</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">(P&X&X)2X <br> P2X / P2X</td>
                  <td style="padding: 8px;">PowerCapacity, STOCapacity, STOMaxPower / --- id. ---</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">process emissions</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Industrial process CO<sub>2</sub> emissions. Tons of CO<sub>2</sub> / hour</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> P / P</td>
                  <td style="padding: 8px;">OutputEmissions / OutputEmissions</td>
                  <td style="padding: 8px;">output / output</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">process emissions CC</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Industrial CO<sub>2</sub> emissions with capture — Tons of CO<sub>2</sub> / hour</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> P / P</td>
                  <td style="padding: 8px;">OutputEmissions / OutputEmissions</td>
                  <td style="padding: 8px;">output / output</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">residential rural air heat pump</td>
                  <td style="padding: 8px;">Sector Heat</td>
                  <td style="padding: 8px;">Converts electricity to heat for rural homes</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P2HT <br> P2HT / P2X</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">residential rural ground heat pump</td>
                  <td style="padding: 8px;">Sector Heat</td>
                  <td style="padding: 8px;">Electric heat using stable ground temperature</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P2HT <br> P2HT / P2X</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">residential rural biomass boiler</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Converts biomass to heat—residential fuel use</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">residential rural gas boiler</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Burns gas for heating—non-electric final energy use</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">residential rural oil boiler</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Burns oil for household heating—outside power generation</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">residential rural resistive heater</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Uses electricity directly for heating—part of demand side</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P2HT <br> P2HT / P2X</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">residential rural water tanks discharger</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Discharges stored heat — part of residential heating loop</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / OutputSectorXStorageInput</td>
                  <td style="padding: 8px;">---- / output</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">residential urban decentral air heat pump</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Decentralized air-source heat pump for urban buildings</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P2HT <br> P2HT / P2X</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">residential urban decentral biomass boiler</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Biomass-to-heat conversion for urban residential areas</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">residential urban decentral gas boiler</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Urban gas boilers—distributed thermal devices</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">residential urban decentral oil boiler</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Decentralized oil boilers for urban homes</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">residential urban decentral resistive heater</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Electric heating devices—boundary heat sector</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P2HT <br> P2HT / P2X</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">residential urban decentral water tanks charger</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Transfers electric energy to thermal storage (heat)</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / OutputSectorXStorageInput</td>
                  <td style="padding: 8px;">---- / output</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">residential urban decentral gas boiler</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Urban gas boilers—distributed thermal devices</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">residential urban decentral water tanks discharger</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Discharges heat from thermal storage</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / OutputSectorXStorageInput</td>
                  <td style="padding: 8px;">---- / output</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">services rural air heat pump</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Service sector rural building heat pump</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P2HT <br> P2HT / P2X</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">services rural biomass boiler</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Provides space or process heat for service buildings</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">services rural gas boiler</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Burns gas for heating—final energy use</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">services rural ground heat pump</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Electricity-to-heat for service buildings—heat sector</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P2HT <br> P2HT / P2X</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">services rural oil boiler</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Oil heating in rural service buildings.</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">services rural resistive heater</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Electric heating for service buildings—final energy use</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P2HT <br> P2HT / P2X</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">services rural water tanks charger</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Converts electricity to heat for service‐sector thermal storage</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / OutputSectorXStorageInput</td>
                  <td style="padding: 8px;">---- / output</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">services rural water tanks discharger</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Discharges stored heat to buildings—part of the heat sector</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / OutputSectorXStorageInput</td>
                  <td style="padding: 8px;">---- / output</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">residential urban decentral gas boiler</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Urban gas boilers—distributed thermal devices</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">residential urban decentral water tanks discharger</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Discharges heat from thermal storage</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / OutputSectorXStorageInput</td>
                  <td style="padding: 8px;">---- / output</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">services rural air heat pump</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Service sector rural building heat pump</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P2HT <br> P2HT / P2X</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">services rural biomass boiler</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Provides space or process heat for service buildings</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">services rural gas boiler</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Burns gas for heating—final energy use</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">services rural ground heat pump</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Electricity-to-heat for service buildings—heat sector</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P2HT <br> P2HT / P2X</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">services rural oil boiler</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Oil heating in rural service buildings.</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">services rural resistive heater</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Electric heating for service buildings—final energy use</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P2HT <br> P2HT / P2X</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">services rural water tanks charger</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Converts electricity to heat for service‐sector thermal storage</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / OutputSectorXStorageInput</td>
                  <td style="padding: 8px;">---- / output</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">services rural water tanks discharger</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Discharges stored heat to buildings—part of the heat sector</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / OutputSectorXStorageInput</td>
                  <td style="padding: 8px;">---- / output</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">solid biomass for industry CC</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Same as above but with carbon capture</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">solid biomass transport</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Represents biomass logistics between regions/sectors</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">urban central air heat pump</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Centralized district heat pump—heat sector interfac.</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P2HT <br> P2HT / P2X</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">urban central gas CHP</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Combined heat + power supplying district heating</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2CHP <br> CHP / CHP</td>
                  <td style="padding: 8px;">PowerCapacity, CHPType, CHPMaxHeat / -- id. --</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">urban central gas CHP CC</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Gas CHP with carbon capture—district heating system</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2CHP <br> CHP / CHP</td>
                  <td style="padding: 8px;">PowerCapacity, CHPType, CHPMaxHeat / -- id. --</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">urban central gas boiler</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Centralized gas heating for urban networks</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">urban central resistive heater</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Electric boiler for district heating—end-use conversion</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P2HT <br> P2HT / P2X</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity, STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">urban central solid biomass CHP</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Biomass combined heat + power—heat boundary process</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2CHP <br> CHP / CHP</td>
                  <td style="padding: 8px;">PowerCapacity, CHPType, CHPMaxHeat / -- id. --</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">urban central solid biomass CHP CC</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Same with carbon capture—boundary sector</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X2CHP <br> CHP / CHP</td>
                  <td style="padding: 8px;">PowerCapacity, CHPType, CHPMaxHeat / -- id. --</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">urban central water tanks charger</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Converts electricity to heat in district storage</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / OutputSectorXStorageInput</td>
                  <td style="padding: 8px;">---- / output</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">link</td>
                  <td style="padding: 8px;">urban central water tanks discharger</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Discharges stored heat to the district network</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / OutputSectorXStorageInput</td>
                  <td style="padding: 8px;">---- / output</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6; border-top: 1px solid #000033;">
                  <td style="padding: 8px;">generator</td>
                  <td style="padding: 8px;">coal</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Fossil fuel-based electricity generation</td>
                  <td style="padding: 8px; border-top: 1px solid #CCCCCC; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> ---- / ----</td>
                  <td style="padding: 8px;">OutputPower / OutputPower</td>
                  <td style="padding: 8px;">output / output</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">generator</td>
                  <td style="padding: 8px;">gas</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Natural gas–fired power generation</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> ---- / ----</td>
                  <td style="padding: 8px;">OutputPower / OutputPower</td>
                  <td style="padding: 8px;">output / output</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">generator</td>
                  <td style="padding: 8px;">lignite</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Coal variant used for power generation</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> ---- / ----</td>
                  <td style="padding: 8px;">OutputPower / OutputPower</td>
                  <td style="padding: 8px;">output / output</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">generator</td>
                  <td style="padding: 8px;">oil</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Oil-fired electricity generation</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> ---- / ----</td>
                  <td style="padding: 8px;">OutputPower / OutputPower</td>
                  <td style="padding: 8px;">output / output</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">generator</td>
                  <td style="padding: 8px;">onwind</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Onshore wind turbine generation</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P <br> P / P</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">generator</td>
                  <td style="padding: 8px;">offwind</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Offshore wind turbine generation</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P <br> P / P</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">generator</td>
                  <td style="padding: 8px;">offwind-ac</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Offshore wind with AC connection</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P <br> P / P</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">generator</td>
                  <td style="padding: 8px;">offwind-dc</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Offshore wind with DC connection</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P <br> P / P</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">generator</td>
                  <td style="padding: 8px;">solar</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Utility-scale photovoltaic generation</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P <br> P / P</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">generator</td>
                  <td style="padding: 8px;">solar rooftop</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Distributed PV connected to power grid</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P <br> P / P</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">generator</td>
                  <td style="padding: 8px;">ror</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Run-of-river hydro power plant</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P <br> P / P</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">generator</td>
                  <td style="padding: 8px;">uranium</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Nuclear fuel input for electricity generation</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P <br> ---- / ----</td>
                  <td style="padding: 8px;">OutputPower / OutputPower</td>
                  <td style="padding: 8px;">output / output</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">generator</td>
                  <td style="padding: 8px;">residential rural solar thermal</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Produces heat for households (not electricity)</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / OutputSectorXStorageInput</td>
                  <td style="padding: 8px;">---- / output</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">generator</td>
                  <td style="padding: 8px;">residential urban decentral solar thermal</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Decentralized solar heating for buildings</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / OutputSectorXStorageInput</td>
                  <td style="padding: 8px;">---- / output</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">generator</td>
                  <td style="padding: 8px;">services rural solar thermal</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Solar thermal for service-sector heat demand</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / OutputSectorXStorageInput</td>
                  <td style="padding: 8px;">---- / output</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">generator</td>
                  <td style="padding: 8px;">services urban decentral solar thermal</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Urban service-sector solar heating</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / OutputSectorXStorageInput</td>
                  <td style="padding: 8px;">---- / output</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">generator</td>
                  <td style="padding: 8px;">urban central solar thermal</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Centralized solar thermal for district heating</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> ---- / P2X</td>
                  <td style="padding: 8px;">------------------- / OutputSectorXStorageInput</td>
                  <td style="padding: 8px;">---- / output</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">generator</td>
                  <td style="padding: 8px;">load</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Represents total shredding energy</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P <br> P / P</td>
                  <td style="padding: 8px;">Load Shedding</td>
                  <td style="padding: 8px;">output</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6; border-top: 1px solid #000033;">
                  <td style="padding: 8px;">storage_units</td>
                  <td style="padding: 8px;">PHS</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Pumped Hydro Storage, a grid-scale electricity storage technology</td>
                  <td style="padding: 8px; border-top: 1px solid #CCCCCC; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P&S <br> P&S / P&S</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">storage_units</td>
                  <td style="padding: 8px;">hydro</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Conventional hydro reservoir</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P&S <br> P&S / P&S</td>
                  <td style="padding: 8px;">PowerCapacity / PowerCapacity</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6; border-top: 1px solid #000033;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">battery</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Electrical energy storage — directly coupled with the grid.</td>
                  <td style="padding: 8px; border-top: 1px solid #CCCCCC; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P&S <br> P&S / P&S</td>
                  <td style="padding: 8px;">STOCapacity, STOMaxChargingPower / -- id --</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">uranium</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Nuclear fuel stock for electricity generation</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P&S <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">H<sub>2</sub></td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Hydrogen storage—chemical energy carrier</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X&S <br> X&S / X&S</td>
                  <td style="padding: 8px;">STOCapacity, STOMaxPower / STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">NH<sub>3</sub></td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Ammonia chemical/fertilizer storage or fuel vector.</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X&S <br> X&S / X&S</td>
                  <td style="padding: 8px;">STOCapacity, STOMaxPower / STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">biogas</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Biomethane stock for heating or industry</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X&S <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">CO<sub>2</sub></td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Captured CO<sub>2</sub> pool—used in synthesis or stored</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> P / P</td>
                  <td style="padding: 8px;">OutputEmissions / OutputEmissions</td>
                  <td style="padding: 8px;">output / output</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">CO<sub>2</sub> sequestered</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Permanent CO<sub>2</sub> storage—Tons of CO<sub>2</sub></td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X&S <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">CO<sub>2</sub> stored</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Intermediate or final CO<sub>2</sub> reservoir—Tons of CO<sub>2</sub></td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X&S <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">coal</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Coal stock for industrial/fuel processes</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X&S <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">gas</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Natural gas (CH<sub>4</sub>) stock — cross-sector energy carrier</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X&S <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">lignite</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Fuel storage for thermal use — outside grid operations</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P&S <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">methanol</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Liquid fuel stock — used in transport or synthesis chains</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X&S <br> X&S / X&S</td>
                  <td style="padding: 8px;">STOCapacity, STOMaxPower / STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">oil</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Oil (synthetic hydrocarbons) stock for transport/industrial fuels</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X&S <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">solid biomass</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Biomass stock for heating/industrial use</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X&S <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">residential rural water tanks</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Thermal storage for rural households — heat sector</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT&S <br> ---- / X&S</td>
                  <td style="padding: 8px;">------------------- / STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">---- / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">residential urban decentral water tanks</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Distributed heat storage in urban residences</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT&S <br> ---- / X&S</td>
                  <td style="padding: 8px;">------------------- / STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">---- / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">services rural water tanks</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Heat storage for rural service buildings</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT&S <br> ---- / X&S</td>
                  <td style="padding: 8px;">------------------- / STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">---- / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">services urban decentral water tanks</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Thermal storage for urban service buildings</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT&S <br> ---- / X&S</td>
                  <td style="padding: 8px;">------------------- / STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">---- / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">stores</td>
                  <td style="padding: 8px;">urban central water tanks</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">District heating storage — boundary heat network</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT&S <br> ---- / X&S</td>
                  <td style="padding: 8px;">------------------- / STOCapacity, STOMaxPower</td>
                  <td style="padding: 8px;">---- / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC; border-top: 1px solid #000033;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">agriculture electricity</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Agro-activities consumption; i.e. irrigation, lighting, ventilation, etc.</td>
                  <td style="padding: 8px; border-top: 1px solid #333333; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P <br> P / P</td>
                  <td style="padding: 8px;">Demand / Demand</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">agriculture heat</td>
                  <td style="padding: 8px;">Sector Heat</td>
                  <td style="padding: 8px;">Crop drying, greenhouse heating, others processes thermal demand</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> ---- / X</td>
                  <td style="padding: 8px;">------------------- / SectorXDemand</td>
                  <td style="padding: 8px;">---- / input</td>
                  <td style="padding: 8px;">N / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">agriculture machinery oil</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Agro-machinery and vehicles diesel and lubricants consumption</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">Electricity demand residential and tertiary</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Residential and service sector appliance and lighting load</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P <br> P / P</td>
                  <td style="padding: 8px;">Demand / Demand</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">industry electricity</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Electrified industrial processes and production demand</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P <br> P / P</td>
                  <td style="padding: 8px;">Demand / Demand</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">gas for industry</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Heat and feedstock industrial natural and synthetic gas demand</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">land transport EV</td>
                  <td style="padding: 8px;">Sector Power</td>
                  <td style="padding: 8px;">Battery electric vehicles in road transport consumption</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">P <br> P / P</td>
                  <td style="padding: 8px;">Demand / Demand</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">land transport oil</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Gasoline and diesel vehicles conventional land transport demand</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">residential rural heat</td>
                  <td style="padding: 8px;">Sector Heat</td>
                  <td style="padding: 8px;">Decentralized space/water heating rural residential buildings demand</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> HT / X</td>
                  <td style="padding: 8px;">HeatDemand / SectorXDemand</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">services rural heat</td>
                  <td style="padding: 8px;">Sector Heat</td>
                  <td style="padding: 8px;">Rural service-sector buildings space/water heating demand</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> HT / X</td>
                  <td style="padding: 8px;">HeatDemand / SectorXDemand</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">residential urban decentral heat</td>
                  <td style="padding: 8px;">Sector Heat</td>
                  <td style="padding: 8px;">Urban residential buildings decentralized space/water demand</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> HT / X</td>
                  <td style="padding: 8px;">HeatDemand / SectorXDemand</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">services urban decentral heat</td>
                  <td style="padding: 8px;">Sector Heat</td>
                  <td style="padding: 8px;">Decentralized heat demand in urban service-sector buildings</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> HT / X</td>
                  <td style="padding: 8px;">HeatDemand / SectorXDemand</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">urban central heat</td>
                  <td style="padding: 8px;">Sector Heat</td>
                  <td style="padding: 8px;">Urban district heating residential and service demand</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> HT / X</td>
                  <td style="padding: 8px;">HeatDemand / SectorXDemand</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">solid biomass for industry</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Industrial solid biomass heat and material Use</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">H<sub>2</sub> for industry</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Steel making, refining, and chemicals hydrogen demand</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> X / X</td>
                  <td style="padding: 8px;">SectorXDemand / SectorXDemand</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">H<sub>2</sub> for shipping</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Fuel-cell and combustion maritime hydrogen demand</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> X / X</td>
                  <td style="padding: 8px;">SectorXDemand / SectorXDemand</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">shipping methanol</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Methanol demand as a maritime fuel for shipping applications</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> X / X</td>
                  <td style="padding: 8px;">SectorXDemand / SectorXDemand</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">shipping oil</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Maritime transport conventional marine oil fuel demand</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">naphtha for industry</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Industrial naphtha non-energy feedstock</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">kerosene for aviation</td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Jet fuel (kerosene) demand for aviation transport</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> ---- / ----</td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;"></td>
                  <td style="padding: 8px;">N / N</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">low-temperature heat for industry</td>
                  <td style="padding: 8px;">Sector Heat</td>
                  <td style="padding: 8px;">Boilers, heat pumps, and waste heat low-temp industrial heat demand</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">HT <br> HT / X</td>
                  <td style="padding: 8px;">HeatDemand / SectorXDemand</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #E6E6E6;">
                  <td style="padding: 8px;">loads</td>
                  <td style="padding: 8px;">NH<sub>3</sub></td>
                  <td style="padding: 8px;">Sector X</td>
                  <td style="padding: 8px;">Fertilizers and industrial/energy use Ammonia demand</td>
                  <td style="padding: 8px; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> X / X</td>
                  <td style="padding: 8px;">SectorXDemand / SectorXDemand</td>
                  <td style="padding: 8px;">input / input</td>
                  <td style="padding: 8px;">Y / Y</td>
                </tr>
                <tr style="background-color: #CCCCCC;">
                  <td style="padding: 8px; border-bottom: 2px solid #000033;">loads</td>
                  <td style="padding: 8px; border-bottom: 2px solid #000033;">coal for industry</td>
                  <td style="padding: 8px; border-bottom: 2px solid #000033;">Sector X</td>
                  <td style="padding: 8px; border-bottom: 2px solid #000033;">Industrial heat, power generation, or metallurgical processes demand</td>
                  <td style="padding: 8px; border-bottom: 2px solid #000033; border-right: 1px dashed #000033; border-left: 1px dashed #000033;">X <br> ---- / ----</td>
                  <td style="padding: 8px; border-bottom: 2px solid #000033;"></td>
                  <td style="padding: 8px; border-bottom: 2px solid #000033;"></td>
                  <td style="padding: 8px; border-bottom: 2px solid #000033;">N / N</td>
                </tr>
             </tbody>
        </table>
    </div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        B-04. Equivalences Dictionary
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        <b>Important Notes:</b> Keep dictionary keys identical to the <code>dispaSET_tech_list</code>.
        <br>
        If the Dispa-SET <code>common.py</code> is updated, these mappings must be resynchronized.
</div>

In [13]:
### Dictionary mapping Dispa-SET acronyms to PyPSA tech names
# =============================================================================
tech_equivalences_dict = {
    ### Renewable Power Units ### ---------------------------------------------
    "offwind-ac"                                    :  {"tech": ["WTOF"                      ] ,   "fuel": ["WIN"             ]}  ,
    "offwind-dc"                                    :  {"tech": ["WTOF"                      ] ,   "fuel": ["WIN"             ]}  ,    
    "onwind"                                        :  {"tech": ["WTON"                      ] ,   "fuel": ["WIN"             ]}  ,
    "solar"                                         :  {"tech": ["PHOT"                      ] ,   "fuel": ["SUN"             ]}  ,
    "solar rooftop"                                 :  {"tech": ["PHOT"                      ] ,   "fuel": ["SUN"             ]}  ,
    "ror"                                           :  {"tech": ["HROR"                      ] ,   "fuel": ["WAT"             ]}  ,
    "hydro"                                         :  {"tech": ["HDAM"                      ] ,   "fuel": ["WAT"             ]}  ,
    "PHS"                                           :  {"tech": ["HPHS"                      ] ,   "fuel": ["WAT"             ]}  ,
    ### Conventional Power Units ### -------------------------------------------
    "lignite"                                       :  {"tech": ["STUR"                      ] ,   "fuel": ["LIG"             ]}  ,
    "nuclear"                                       :  {"tech": ["STUR"                      ] ,   "fuel": ["NUC"             ]}  ,
    "CCGT"                                          :  {"tech": ["COMC"                      ] ,   "fuel": ["GAS"             ]}  ,
    "OCGT"                                          :  {"tech": ["GTUR"                      ] ,   "fuel": ["GAS"             ]}  ,
    "oil"                                           :  {"tech": ["STUR"  , "GTUR"  , "ICEN"  ] ,   "fuel": ["OIL"             ]}  ,
    "coal"                                          :  {"tech": ["STUR"  , "COMC"  , "GTUR"  ] ,   "fuel": ["HRD" , "PEA"     ]}  ,
    ### Combined Heat and Power Units ### ---------------------------------------
    "urban central gas CHP CC"                      :  {"tech": ["STUR"  , "ICEN"            ] ,   "fuel": ["GAS"             ]}  ,
    "urban central gas CHP"                         :  {"tech": ["STUR"  , "ICEN"            ] ,   "fuel": ["GAS"             ]}  ,
    "urban central solid biomass CHP CC"            :  {"tech": ["STUR"                      ] ,   "fuel": ["BIO"             ]}  ,
    "urban central solid biomass CHP"               :  {"tech": ["STUR"                      ] ,   "fuel": ["BIO"             ]}  ,
    ### Sector X Units ### ------------------------------------------------------
    #"EV charger"                                    :  {"tech": ["BEVS"                      ] ,   "fuel": ["ELE"             ]}  ,
    #"V2G"                                           :  {"tech": ["BEVS"                      ] ,   "fuel": ["ELE"             ]}  ,
    #"battery charger"                               :  {"tech": ["BATS"                      ] ,   "fuel": ["ELE"             ]}  ,
    #"battery discharger"                            :  {"tech": ["BATS"                      ] ,   "fuel": ["ELE"             ]}  ,    
    #"H2 Electrolysis"                               :  {"tech": ["P2GS"                      ] ,   "fuel": ["ELE"             ]}  ,
    "H2 Fuel Cell"                                  :  {"tech": ["GTUR"  , "ICEN"            ] ,   "fuel": ["HYD"             ]}  ,
    "H2 turbine"                                    :  {"tech": ["GTUR"                      ] ,   "fuel": ["HYD"             ]}  ,
    #"Haber-Bosch"                                   :  {"tech": ["P2GS"                      ] ,   "fuel": ["ELE"             ]}  ,
    #"methanolisation"                               :  {"tech": ["P2BS"                      ] ,   "fuel": ["ELE"             ]}  ,
    #"DAC"                                           :  {"tech": ["P2GS"                      ] ,   "fuel": ["ELE"             ]}  ,
    "urban central resistive heater"                :  {"tech": ["P2BS"                      ] ,   "fuel": ["ELE"             ]}  ,
    "residential rural resistive heater"            :  {"tech": ["P2BS"                      ] ,   "fuel": ["ELE"             ]}  ,
    "residential urban decentral resistive heater"  :  {"tech": ["P2BS"                      ] ,   "fuel": ["ELE"             ]}  ,
    "services urban decentral resistive heater"     :  {"tech": ["P2BS"                      ] ,   "fuel": ["ELE"             ]}  ,
    "services rural resistive heater"               :  {"tech": ["P2BS"                      ] ,   "fuel": ["ELE"             ]}  ,
    #"urban central air heat pump"                   :  {"tech": ["ASHP"                      ] ,   "fuel": ["ELE"             ]}  ,
    #"residential rural air heat pump"               :  {"tech": ["ASHP"                      ] ,   "fuel": ["ELE"             ]}  ,
    #"residential urban decentral air heat pump"     :  {"tech": ["ASHP"                      ] ,   "fuel": ["ELE"             ]}  ,
    #"services urban decentral air heat pump"        :  {"tech": ["ASHP"                      ] ,   "fuel": ["ELE"             ]}  ,
    #"services rural air heat pump"                  :  {"tech": ["ASHP"                      ] ,   "fuel": ["ELE"             ]}  ,
    #"residential rural ground heat pump"            :  {"tech": ["P2BS"                      ] ,   "fuel": ["ELE"             ]}  ,
    #"services rural ground heat pump"               :  {"tech": ["P2BS"                      ] ,   "fuel": ["ELE"             ]}  ,
}
# =============================================================================
print(f"Number of mappings: {len(tech_equivalences_dict)}")
for key, value in tech_equivalences_dict.items():
    print(
        f"{key:<45} -> "
        f"tech={','.join(value['tech'])} | "
        f"fuel={','.join(value['fuel'])}"
    )

Number of mappings: 25
offwind-ac                                    -> tech=WTOF | fuel=WIN
offwind-dc                                    -> tech=WTOF | fuel=WIN
onwind                                        -> tech=WTON | fuel=WIN
solar                                         -> tech=PHOT | fuel=SUN
solar rooftop                                 -> tech=PHOT | fuel=SUN
ror                                           -> tech=HROR | fuel=WAT
hydro                                         -> tech=HDAM | fuel=WAT
PHS                                           -> tech=HPHS | fuel=WAT
lignite                                       -> tech=STUR | fuel=LIG
nuclear                                       -> tech=STUR | fuel=NUC
CCGT                                          -> tech=COMC | fuel=GAS
OCGT                                          -> tech=GTUR | fuel=GAS
oil                                           -> tech=STUR,GTUR,ICEN | fuel=OIL
coal                                          -> tech=STU

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       B-05. Loading Fuel–Technology Mapping Data
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process loads the default and country-specific <b>fuel–technology mapping</b> files. <br>These CSV files define the proportion of each fuel type used by each technology, enabling accurate fuel consumption and emission calculations in Dispa-SET.
    </div>
</div>

In [14]:
### Fuel–Technology Mapping Data
### 1. Fuel mapping folder
fuel_technology_mapping_folder_path = (
    Path(dispaSET_unleash_folder_path)
    / "scripts"
    / "Unleash_PyPSA_DispaSET_Raw_Data_Processing"
    / "Coefficients_Sources"
    / "Fuels"
)
print(f"fuel_technology_mapping_folder_path: {fuel_technology_mapping_folder_path}")
print("*" * 140)
### 2. Load the default fuel–technology mapping
overall_fuel_technologies_match_df = pd.read_csv(
    fuel_technology_mapping_folder_path / "DEFAULT.csv",
    index_col="Technology"
)
### 3. Load all country-specific fuel–technology mappings
fuel_technologies_match_dict = {}
for csv_file in fuel_technology_mapping_folder_path.glob("*.csv"):
    # DEFAULT.csv is loaded separately as the fallback mapping
    if csv_file.stem == "DEFAULT":
        continue
    ### Country code from the filename
    ### Example: BE.csv → BE
    country = csv_file.stem
    fuel_technologies_match_dict[country] = pd.read_csv(
        csv_file,
        index_col="Technology"
    )
### 4. Report loaded mappings
print(
    f"✅ Default mapping loaded: "
    f"{overall_fuel_technologies_match_df.shape[0]} technologies × "
    f"{overall_fuel_technologies_match_df.shape[1]} fuels"
)
print(
    f"✅ Country-specific mappings loaded: "
    f"{len(fuel_technologies_match_dict)}"
)
print(
    f"   Countries: "
    f"{', '.join(sorted(fuel_technologies_match_dict.keys()))}"
)
print("*" * 140)

fuel_technology_mapping_folder_path: /home/ray/Dispa-SET_Unleash/scripts/Unleash_PyPSA_DispaSET_Raw_Data_Processing/Coefficients_Sources/Fuels
********************************************************************************************************************************************
✅ Default mapping loaded: 51 technologies × 19 fuels
✅ Country-specific mappings loaded: 45
   Countries: AL, AM, AT, AZ, BA, BE, BG, BY, CH, CY, CZ, DE, DK, EE, EL, ES, FI, FR, GE, HR, HU, IE, IS, IT, LT, LU, LV, MD, ME, MK, MT, NL, NO, PL, PT, RO, RS, RU, SE, SI, SK, TR, UA, UK, XK
********************************************************************************************************************************************


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        B-06. Dispa-SET Technology - Fuel Matching</span>
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        In the Dispa-SET framework, every power unit is defined by a <b>(Technology, Fuel)</b> tuple.<br> The following matrix provides an updated <b>Fuel-Type Proportion Table</b>, mapping the fractional contribution (0.0000 to 1.0000) of each resource to its respective technology.
    <div style="margin-left: 2em; font-size: 12px">
        <b>Data Synthesis:</b> These mappings are derived from the <code>commons.py</code> core scripts and further refined using the 2024-2025 statistical Compendium.
    </div>
    </div>
    <span style="font-weight: bold; font-size: 12px">
        <b>Overall Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px; font-size: 10px">
<b>I. Frameworks, Policy & Primary Databases</b>
        <ol>
            <li>Dispa-SET Data Documentation: <a href="https://www.dispaset.eu/en/latest/data.html" style="color: #000033;">Link</a></li>
            <li>JRC European Energy Storage Inventory (2025): <a href="https://joint-research-centre.ec.europa.eu/jrc-news-and-updates/new-tool-maps-europes-real-time-sustainable-energy-storage-data-2025-03-20_en" style="color: #000033;">Link</a></li>
            <li>IEA Energy Statistics Data Browser: <a href="https://www.iea.org/data-and-statistics/data-tools/energy-statistics-data-browser" style="color: #000033;">Link</a></li>
            <li>EU Energy Statistical Pocketbook: <a href="https://energy.ec.europa.eu/data-and-analysis/eu-energy-statistical-pocketbook-and-country-datasheets_en" style="color: #000033;">Link</a></li>
            <li>Ember European Electricity Review (2025): <a href="https://www.ember-climate.org/publications/european-electricity-review-2025/" style="color: #000033;">Link</a></li>
            <li>ETSAP-TIMES Model Generator: <a href="https://iea-etsap.org/index.php/etsap-tools/model-generator/times" style="color: #000033;">Link</a></li>
            <li>Dispa-SET Technical Documentation: <a href="https://www.dispaset.eu/en/latest/" style="color: #000033;">Link</a></li>
            <li>IEA Hydrogen Roadmap (2023): <a href="https://www.iea.org/reports/technology-roadmap-hydrogen-and-fuel-cells" style="color: #000033;">Link</a></li>
            <li>IRENA Electrolyzers (2023): <a href="https://www.irena.org/publications" style="color: #000033;">Link</a></li>
            <li>US DOE Electrolysis: <a href="https://www.energy.gov/eere/fuelcells/hydrogen-production-electrolysis" style="color: #000033;">Link</a></li>
            <li>EC Hydrogen System: <a href="https://energy.ec.europa.eu/topics/eus-energy-system/hydrogen_en" style="color: #000033;">Link</a></li>
            <li>US DOE Air-Source Heat Pumps (2024): <a href="https://www.energy.gov/energysaver/air-source-heat-pumps" style="color: #000033;">Link</a></li>
            <li>Minnesota ASHP Report (2011): <a href="https://www.leg.mn.gov/docs/2014/other/141021.pdf" style="color: #000033;">Link</a></li>
            <li>North NJ HVAC Thermal Energy: <a href="https://northnjhvac.com/common-sources-thermal-energy-heat-pumps-explained/" style="color: #000033;">Link</a></li>
            <li>US DOE Battery Storage (2024): <a href="https://www.energy.gov/sites/default/files/2025-01/BESSIE_supply-chain-battery-report_111124_OPENRELEASE_SJ_1.pdf" style="color: #000033;">Link</a></li>
            <li>EIA Battery Storage (2024): <a href="https://www.eia.gov/todayinenergy/detail.php?id=63025" style="color: #000033;">Link</a></li>
            <li>IEA Batteries & Transitions (2024): <a href="https://www.iea.org/reports/batteries-and-secure-energy-transitions/executive-summary" style="color: #000033;">Link</a></li>
            <li>Vehicle-to-Grid (2025): <a href="https://www.mdpi.com/2032-6653/16/3/142" style="color: #000033;">Link</a></li>
            <li>EIA Transportation Energy (2024): <a href="https://www.eia.gov/energyexplained/use-of-energy/transportation-in-depth.php" style="color: #000033;">Link</a></li>
            <li>Solar PV Systems (2023): <a href="https://www.frontiersin.org/articles/10.3389/fenrg.2023.1164494/full" style="color: #000033;">Link</a></li>
        </ol>
<b>II. Specialized Conversion & Industrial Systems</b>
        <ol start="21">
            <li>US DOE Solar Building Integration (2024): <a href="https://www.energy.gov/eere/solar/articles/expanding-solar-energy-opportunities-rooftops-building-integration" style="color: #000033;">Link</a></li>
            <li>US DOE Compressed Air Storage (2023): <a href="https://www.energy.gov/sites/default/files/2023-07/Technology%20Strategy%20Assessment%20-%20Compressed%20Air%20Energy%20Storage_0.pdf" style="color: #000033;">Link</a></li>
            <li>Urbanao CAES Overview (2025): <a href="https://urbanao.com/post/compressed-air-energy-storage-caes-a-comprehensive-2025-overview" style="color: #000033;">Link</a></li>
            <li>EPA CHP Emissions Methodology (2021): <a href="https://www.epa.gov/sites/production/files/2015-07/documents/fuel_and_carbon_dioxide_emissions_savings_calculation_methodology_for_combined_heat_and_power_systems.pdf" style="color: #000033;">Link</a></li>
            <li>US DOE CHP Guide (2015): <a href="https://www.energy.gov/sites/default/files/2019/01/f58/CHPGuide2015.pdf" style="color: #000033;">Link</a></li>
            <li>EIA CHP Technical Details: <a href="https://www.eia.gov/todayinenergy/detail.php?id=8250" style="color: #000033;">Link</a></li>
            <li>Direct Methanol Fuel Cells (Zelenay): <a href="https://www1.eere.energy.gov/hydrogenandfuelcells//pdfs/ive16_zelenay.pdf" style="color: #000033;">Link</a></li>
            <li>DMFC Review (Springer 2015): <a href="https://link.springer.com/article/10.1557/mre.2015.4" style="color: #000033;">Link</a></li>
            <li>Methanol Fuel (IntechOpen 2023): <a href="https://www.intechopen.com/chapters/1157844" style="color: #000033;">Link</a></li>
            <li>World Bank Geothermal Direct Use (2022): <a href="https://www.esmap.org/sites/default/files/esmap-files/16103-WB_ESMAP%20Direct%20Use-WEB.pdf" style="color: #000033;">Link</a></li>
            <li>UMich Geothermal Factsheet (2023): <a href="https://css.umich.edu/publications/factsheets/energy/geothermal-energy-factsheet" style="color: #000033;">Link</a></li>
            <li>ORNL Geothermal Heat Pumps Study (2024): <a href="https://www.ornl.gov/news/ornl-study-projects-geothermal-heat-pumps-impact-carbon-emissions-and-electrical-grid-2050" style="color: #000033;">Link</a></li>
            <li>US DOE Geothermal Guide (2023): <a href="https://www.energy.gov/sites/prod/files/guide_to_geothermal_heat_pumps.pdf" style="color: #000033;">Link</a></li>
            <li>EnergySage Geothermal Pros/Cons (2023): <a href="https://www.energysage.com/heat-pumps/pros-cons-geothermal-heat-pumps/" style="color: #000033;">Link</a></li>
            <li>Wikipedia Ground Source Heat Pump: <a href="https://en.wikipedia.org/wiki/Ground_source_heat_pump" style="color: #000033;">Link</a></li>
            <li>Hydrogen in Gas Turbines (Oxford 2022): <a href="https://doi.org/10.1093/ijlct/ctac025" style="color: #000033;">Link</a></li>
            <li>Fuel Composition Impact (ASME 2019): <a href="https://doi.org/10.1115/1.4044238" style="color: #000033;">Link</a></li>
            <li>US DOE Hydrogen Turbine Review (2022): <a href="https://www.netl.doe.gov/sites/default/files/publication/A-Literature-Review-of-Hydrogen-and-Natural-Gas-Turbines-081222.pdf" style="color: #000033;">Link</a></li>
            <li>Modernizing Gas District Heating (2024): <a href="https://doi.org/10.3390/su16041401" style="color: #000033;">Link</a></li>
            <li>Hydrogen Storage Review (WJET 2023): <a href="https://doi.org/10.4236/wjet.2023.113033" style="color: #000033;">Link</a></li>
        </ol>
<b>III. Renewables, Storage & Thermodynamics</b>
        <ol start="41">
            <li>Sandia Hydrogen Storage (2022): <a href="https://www.sandia.gov/app/uploads/sites/163/2022/03/ESHB_Ch11_Hydrogen_Headley.pdf" style="color: #000033;">Link</a></li>
            <li>US Billion-Ton Biomass Report (2024): <a href="https://www.energy.gov/sites/default/files/2024-03/beto-2023-billion-ton-report_2-current_0.pdf" style="color: #000033;">Link</a></li>
            <li>JRC Residential Biomass (2018): <a href="https://publications.jrc.ec.europa.eu/repository/bitstream/JRC113417/kjna29542enn.pdf" style="color: #000033;">Link</a></li>
            <li>EIA Hydropower Index (2023): <a href="https://www.eia.gov/energyexplained/hydropower/index.php" style="color: #000033;">Link</a></li>
            <li>Britannica Hydroelectric Science (2025): <a href="https://www.britannica.com/science/hydroelectric-power" style="color: #000033;">Link</a></li>
            <li>Sandia Pumped Hydro (2024): <a href="https://www.sandia.gov/app/uploads/sites/163/2024/08/ESHB_Ch9_PHS_Bera.pdf" style="color: #000033;">Link</a></li>
            <li>EPA Fuel Oil Combustion (2020): <a href="https://www.epa.gov/sites/production/files/2020-09/documents/1.3_fuel_oil_combustion.pdf" style="color: #000033;">Link</a></li>
            <li>World Bank Pumped Storage (2022): <a href="https://www.esmap.org/sites/default/files/ESP/WB_PSH_16Jun22.pdf" style="color: #000033;">Link</a></li>
            <li>Engineering Toolbox Combustion Efficiency: <a href="https://www.engineeringtoolbox.com/boiler-combustion-efficiency-d_271.html" style="color: #000033;">Link</a></li>
            <li>EIA Hydropower Generation Geography: <a href="https://www.eia.gov/energyexplained/hydropower/where-hydropower-is-generated.php" style="color: #000033;">Link</a></li>
            <li>Run-of-River Design (Elsevier 2019): <a href="https://doi.org/10.1016/j.envsoft.2018.08.018" style="color: #000033;">Link</a></li>
            <li>Columbia Hybrid Heat Pump System (2020): <a href="https://qsel.columbia.edu/assets/uploads/blog/2020/publications/cost-optimal-sizing-and-operation-of-a-hybrid-heat-pump-system-using-numerical-simulation.pdf" style="color: #000033;">Link</a></li>
            <li>EPA Biomass CHP Tech Catalog (2007): <a href="https://www.epa.gov/sites/production/files/2015-07/documents/biomass_combined_heat_and_power_catalog_of_technologies_v.1.1.pdf" style="color: #000033;">Link</a></li>
            <li>WEF Internal Combustion Engine CHP (2017): <a href="https://www.wef.org/globalassets/assets-wef/direct-download-library/public/03---resources/wsec-2017-tr-002-rbc-internal-combustion-engines---9.2017.pdf" style="color: #000033;">Link</a></li>
            <li>Molten Carbonate Fuel Cell Analysis: <a href="https://www.hydrogen.energy.gov/docs/hydrogenprogramlibraries/pdfs/progress15/ix_4_ahmed_2015.pdf" style="color: #000033;">Link</a></li>
            <li>Electrolyzer-Methanation (Applied Energy): <a href="https://doi.org/10.1016/j.apenergy.2022.120268" style="color: #000033;">Link</a></li>
            <li>US DOE Electric Resistance Heating: <a href="https://www.energy.gov/energysaver/electric-resistance-heating" style="color: #000033;">Link</a></li>
            <li>NASA Regenerative Fuel Cell TM (2020): <a href="https://ntrs.nasa.gov/api/citations/20205000357/downloads/TM-20205000357.pdf" style="color: #000033;">Link</a></li>
            <li>Reversible Fuel Cells (JPEE 2025): <a href="https://doi.org/10.4236/jpee.2025.136001" style="color: #000033;">Link</a></li>
            <li>UMich Solar PV Factsheet (2024): <a href="https://css.umich.edu/publications/factsheets/energy/solar-pv-energy-factsheet" style="color: #000033;">Link</a></li>
        </ol>
<b>IV. Emerging Tech & Advanced Thermal</b>
        <ol start="61">
            <li>US DOE PEM Electrolysis Targets: <a href="https://www.energy.gov/eere/fuelcells/technical-targets-proton-exchange-membrane-electrolysis" style="color: #000033;">Link</a></li>
            <li>Phosphoric Acid Fuel Cells (FuelCellZ): <a href="https://fuelcellz.com/phosphoric-acid-fuel-cells-pafc/" style="color: #000033;">Link</a></li>
            <li>PEM Fuel Cell Review (RSC 2022): <a href="https://pubs.rsc.org/en/content/articlelanding/2022/ee/d2ee00790h" style="color: #000033;">Link</a></li>
            <li>US DOE High-Temp Electrolysis: <a href="https://www.energy.gov/eere/fuelcells/technical-targets-high-temperature-electrolysis" style="color: #000033;">Link</a></li>
            <li>Solar Thermal Urban (Frontiers 2025): <a href="https://doi.org/10.3389/frsc.2025.1583316" style="color: #000033;">Link</a></li>
            <li>Solid Oxide Fuel Cells (MDPI 2025): <a href="https://doi.org/10.3390/pr13041145" style="color: #000033;">Link</a></li>
            <li>EnergySage Solar Thermal Guide (2023): <a href="https://www.energysage.com/about-clean-energy/solar/solar-thermal-what-you-need-to-know/" style="color: #000033;">Link</a></li>
            <li>MIT Solar Heating for Industry (2015): <a href="https://energy.mit.edu/wp-content/uploads/2015/04/MITEI-WP-2015-04.pdf" style="color: #000033;">Link</a></li>
            <li>Biomass Steam Power (Oxford 2023): <a href="https://doi.org/10.1093/ce/zkad049" style="color: #000033;">Link</a></li>
            <li>EPA Waste Heat to Power (2015): <a href="https://www.epa.gov/sites/default/files/2015-07/documents/waste_heat_to_power_systems.pdf" style="color: #000033;">Link</a></li>
            <li>Thermal Hot Water Storage (2025): <a href="https://www.energystoragenl.nl/wp-content/uploads/2025/01/Thermische-HotWater.pdf" style="color: #000033;">Link</a></li>
            <li>EIA Wave Power Explained (2024): <a href="https://www.eia.gov/energyexplained/hydropower/wave-power.php" style="color: #000033;">Link</a></li>
            <li>Trane Water-Source Heat Pumps: <a href="https://www.trane.com/content/dam/Trane/Commercial/global/products-systems/education-training/engineers-newsletters/energy-environment/admapn024en_0507.pdf" style="color: #000033;">Link</a></li>
            <li>DOE Offshore Wind Market (2023): <a href="https://www.energy.gov/sites/default/files/2023-09/doe-offshore-wind-market-report-2023-edition.pdf" style="color: #000033;">Link</a></li>
            <li>IEA Wind Tracking (2024): <a href="https://www.iea.org/energy-system/renewables/wind" style="color: #000033;">Link</a></li>
            <li>NREL Building Thermal Decarbonization (2024): <a href="https://www.nrel.gov/docs/fy24osti/87812.pdf" style="color: #000033;">Link</a></li>
        </ol>
        <div style="break-inside: avoid;">
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(AL) Albania-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; padding-top: 10px; font-size: 10px;">
        The Albanian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                KESH (Albanian Power Corporation) Annual Operations:
                <a href="https://www.kesh.al/en/energy-production-data/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ERE (Energy Regulatory Authority) Yearly Reports:
                <a href="https://www.ere.gov.al/publications" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                OST (Transmission System Operator) Grid Statistics:
                <a href="https://www.ost.al/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                INSTAT (Albanian Institute of Statistics) Energy Balance:
                <a href="http://www.instat.gov.al/en/themes/industry-trade-and-services/energy/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Infrastructure and Energy - National Strategy:
                <a href="https://www.infrastruktura.gov.al/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                TAP (Trans Adriatic Pipeline) Regional Flow Data:
                <a href="https://www.tap-ag.com/about-us/our-pipeline" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Energy Community Secretariat - Albania Profile:
                <a href="https://www.energy-community.org/implementation/Albania.html" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(AM) Armenia-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px; font-size: 10px;">
        The Armenian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                Ministry of Territorial Administration and Infrastructure:
                <a href="http://www.mtad.am/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Public Services Regulatory Commission of Armenia (PSRC):
                <a href="http://www.psrc.am/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Settlement Centre of Armenia - Power System Balance:
                <a href="https://www.sc.am/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Armenian Nuclear Power Plant (ANPP) Operations:
                <a href="http://www.armeniannpp.am/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Armenian Renewable Energy and Energy Efficiency Fund (R2E2):
                <a href="https://r2e2.am/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Statistical Committee of the Republic of Armenia (Armstat):
                <a href="https://www.armstat.am/en/?id=14" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Energy Community - Armenia Monitoring:
                <a href="https://www.energy-community.org/implementation/Armenia.html" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(AT) Austria-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Austrian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                BMK (Federal Ministry for Climate Action) - Final Updated National Energy and Climate Plan (NECP) 2021-2030:
                <a href="https://www.bmk.gv.at/en.html" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                E-Control Austria - Annual Energy Statistics and 2025/2026 Market Performance Reports:
                <a href="https://www.e-control.at/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                International Energy Agency (IEA) - Austria Energy Policy Review and 2024 Analysis:
                <a href="https://www.iea.org/reports/austria-2020" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Austrian Power Grid (APG) - Network Development Plan 2025 and 2026 Operational Reserve Strategy:
                <a href="https://www.apg.at/en/power-grid/grid-expansion/network-development-plan-2025/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Statistics Austria - Energy Balances and Monthly Electricity Statistics (Dec 2025 Update):
                <a href="https://www.statistik.at/en/statistics/energy-and-environment/energy" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Eurostat - Energy Database (European Commission Statistical Integration 2026):
                <a href="https://ec.europa.eu/eurostat/web/energy/database" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Verbund AG - Technical Parameters for the Kaprun and Malta-Reisseck Hydropower Groups:
                <a href="https://www.verbund.com/en-at" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(AZ) Azerbaijan-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px; font-size: 10px;">
        The Azerbaijani matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                Ministry of Energy of the Republic of Azerbaijan:
                <a href="https://minenergy.gov.az/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Azerenerji JSC (Main Power Producer) Statistics:
                <a href="https://azerenerji.gov.az/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                State Statistical Committee of Azerbaijan - Energy Balances:
                <a href="https://www.stat.gov.az/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                AREA (Azerbaijan Renewable Energy Agency):
                <a href="https://area.gov.az/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                SOCAR (State Oil Company) Gas Supply Data:
                <a href="https://socar.az/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                EIA Azerbaijan Country Analysis:
                <a href="https://www.eia.gov/international/analysis/country/AZE" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                International Energy Agency (IEA) Azerbaijan Profile:
                <a href="https://www.iea.org/countries/azerbaijan" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(BA) Bosnia and Herzegovina-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Bosnian and Herzegovinian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                NOSBiH (Independent System Operator) Operational Data:
                <a href="https://www.nosbih.ba/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                DERK (State Electricity Regulatory Commission):
                <a href="https://www.derk.ba/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Elektroprivreda BiH (EPBiH) Generation Statistics:
                <a href="https://www.epbih.ba/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Agency for Statistics of Bosnia and Herzegovina (BHAS):
                <a href="https://bhas.gov.ba/?lang=en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Energy Community Secretariat - Bosnia and Herzegovina Profile:
                <a href="https://www.energy-community.org/implementation/Bosnia_Herzegovina.html" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Elektroprivreda HZHB (Hydropower Operations):
                <a href="https://www.ephzhb.ba/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                USAID Energy Policy Activity (EPA) in BiH Reports:
                <a href="https://usaidepa.ba/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(BE) Belgium-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px; font-size: 10px;">
        The Belgian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                ELIA Group - Transparency Platform:
                <a href="https://www.elia.be/en/grid-data" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                International Energy Agency (IEA) - Belgium:
                <a href="https://www.iea.org/countries/belgium" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                FPS Economy - Energy Themes:
                <a href="https://economie.fgov.be/en/themes/energy" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Eurostat - Energy Balances for Belgium:
                <a href="https://ec.europa.eu/eurostat/web/energy/data" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                FEBEG (Federation of Belgian Energy Companies):
                <a href="https://www.febeg.be/en/statistics" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                CREG (Commission for Electricity and Gas Regulation):
                <a href="https://www.creg.be/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Synergrid - Belgian Grid Technical Data:
                <a href="https://www.synergrid.be/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(BG) Bulgaria-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Bulgarian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                ESO (Electricity System Operator) Real-time Data:
                <a href="https://www.eso.bg/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                EWRC (Energy and Water Regulatory Commission):
                <a href="https://www.dker.bg/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Energy of the Republic of Bulgaria:
                <a href="https://me.government.bg/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Kozloduy Nuclear Power Plant - Operational Stats:
                <a href="https://www.kznpp.org/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                IBEX (Independent Bulgarian Energy Exchange):
                <a href="https://ibex.bg/?lang=en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                National Statistical Institute (NSI) - Energy Statistics:
                <a href="https://www.nsi.bg/en/content/5012/energy" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Sustainable Energy Development Agency (SEDA):
                <a href="https://www.seea.government.bg/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(BY) Belarus-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px; font-size: 10px;">
        The Belarusian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                Ministry of Energy of the Republic of Belarus:
                <a href="https://minenergo.gov.at/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Belenergo State Production Association (GPA):
                <a href="http://www.belenergo.by/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                National Statistical Committee of the Republic of Belarus (Belstat):
                <a href="https://www.belstat.gov.by/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Belarusian Nuclear Power Plant (Ostrovets) Technical Data:
                <a href="https://belaes.by/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                IEA Belarus Energy Profile & Statistics:
                <a href="https://www.iea.org/countries/belarus" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                UNECE Belarus Sustainable Energy Data:
                <a href="https://unece.org/sustainable-energy/belarus" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Department for Energy Efficiency of the State Committee for Standardization:
                <a href="https://energoeffekt.gov.by/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(CH) Switzerland-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Swiss matrix was constructed and refined using the following localized and international sources:
        <ol>
            <li>
                BFE (Federal Office of Energy) - Schweizerische Gesamtenergiestatistik 2022/2023:
                <a href="https://www.bfe.admin.ch/bfe/en/home/supply/statistics-and-geodata/energy-statistics/overall-energy-statistics.html" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                SFOE - Swiss Energy Strategy 2050 Policy Roadmap and Monitoring Reports:
                <a href="https://www.bfe.admin.ch/bfe/en/home/policy/energy-strategy-2050.html" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                International Energy Agency (IEA) - Switzerland Energy Policy Review:
                <a href="https://www.iea.org/reports/switzerland-2023" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Swissgrid - Grid Infrastructure and Operational Data for the European Interconnected System:
                <a href="https://www.swissgrid.ch/en/home/newsroom/newsfeed/20240416-01.html" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Eurostat - Energy Database (European Commission Statistical Integration):
                <a href="https://ec.europa.eu/eurostat/web/energy/database" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ELCOM (Federal Electricity Commission) - Market Monitoring and Pricing Data:
                <a href="https://www.elcom.admin.ch/elcom/en/home.html" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Alpiq/Axpo - Technical Data for Nant de Drance and Limmern Pumped Storage Plants:
                <a href="https://www.nant-de-drance.ch/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(CY) Cyprus-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Cypriot matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                TSOC (Cyprus Transmission System Operator) Real-time Data:
                <a href="https://tsoc.org.cy/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                CERA (Cyprus Energy Regulatory Authority) Annual Reports:
                <a href="https://www.cera.org.cy/en-gb/home" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                EAC (Electricity Authority of Cyprus) Generation Portfolio:
                <a href="https://www.eac.com.cy/EN/EAC/Sustainability/Pages/ElectricityProduction.aspx" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Energy, Commerce and Industry - NECP 2021-2030:
                <a href="https://meci.gov.cy/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Cyprus Statistical Service (CYSTAT) Energy Statistics:
                <a href="https://www.cystat.gov.cy/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                IRENA Renewable Energy Roadmap for the Republic of Cyprus:
                <a href="https://www.irena.org/publications" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Cyprus University of Technology (CUT) Energy Planning Models:
                <a href="https://www.cut.ac.cy/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(CZ) Czech Republic-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Czech matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                ČEPS (Transmission System Operator) - Transparency Data Portal:
                <a href="https://www.ceps.cz/en/all-data" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ERO (Energy Regulatory Office) - Yearly Operation Reports:
                <a href="https://eru.gov.cz/en/reports-on-operation" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                OTE, a.s. (Market Operator) - Monthly/Yearly Energy Balances:
                <a href="https://www.ote-cr.cz/en/statistics" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Industry and Trade - State Energy Policy (ASEP):
                <a href="https://www.mpo.cz/en/energy/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Czech Statistical Office (CZSO) - Energy Statistics:
                <a href="https://www.czso.cz/csu/czso/energy_stat" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                CEZ Group - Nuclear and Thermal Generation Portfolio:
                <a href="https://www.cez.cz/en/investors/results-and-reports" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                IEA - Czechia 2025 Energy Policy Review:
                <a href="https://www.iea.org/countries/czechia" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(DE) Germany-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The German matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                BMWK - Energiedaten: Gesamtausgabe [Complete Energy Data]:
                <a href="https://www.bmwk.de/Redaktion/DE/Artikel/Energie/energiedaten-gesamtausgabe.html" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Umweltbundesamt (UBA) - Erneuerbare Energien in Zahlen:
                <a href="https://www.umweltbundesamt.de/themen/klima-energie/erneuerbare-energien/erneuerbare-energien-in-zahlen" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Fraunhofer ISE - Energy Charts (Real-time Generation Data):
                <a href="https://www.energy-charts.info/index.html?l=en&c=DE" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                International Energy Agency (IEA) - Germany Policy Review:
                <a href="https://www.iea.org/reports/germany-2025" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Eurostat - Energy Database (European Commission):
                <a href="https://ec.europa.eu/eurostat/web/energy/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                SMARD.de - Bundesnetzagentur Electricity Market Data:
                <a href="https://www.smard.de/home" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                AG Energiebilanzen e.V. (AGEB) - Detailed Balance Sheets:
                <a href="https://ag-energiebilanzen.de/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(DK) Denmark-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Danish matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                Energinet - DataHub and Grid Transparency:
                <a href="https://energinet.dk/en/electricity/datahub" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Danish Energy Agency - Annual and Monthly Statistics:
                <a href="https://ens.dk/en/analyses-and-statistics/annual-and-monthly-statistics" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Statistics Denmark - Energy and Environment:
                <a href="https://www.dst.dk/en/Statistik/emner/energi-og-miljoe/energi" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                International Energy Agency (IEA) - Denmark Policy Review:
                <a href="https://www.iea.org/reports/denmark-2023" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Danish District Heating Association - Statistical Data:
                <a href="https://www.danskfjernvarme.dk/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Eurostat - Energy Balances for Denmark:
                <a href="https://ec.europa.eu/eurostat/web/energy/data" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Wind Denmark - Offshore and Onshore Capacity Stats:
                <a href="https://winddenmark.dk/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(EE) Estonia-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Estonian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                Elering - Estfeed Datahub (Electricity and Gas Transparency):
                <a href="https://datahub.elering.ee/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Statistics Estonia - Energy Balance and Production:
                <a href="https://stat.ee/en/find-statistics/statistics-theme/energy-and-transport/energy" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Estonian Competition Authority - Annual Market Reports:
                <a href="https://www.konkurentsiamet.ee/en/office-news-contacts/reports-analyses-assessments/estonian-electricity-and-gas-market-reports" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Climate - Energy Sector Development Plan (ENMAK 2035):
                <a href="https://kliimaministeerium.ee/en/energy-sector-development-plan" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                IEA - Estonia Energy Profile and Policy Review:
                <a href="https://www.iea.org/countries/estonia" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Eesti Energia (AS Eesti Energia) - Operational Reports:
                <a href="https://www.energia.ee/en/investor/reports" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Environmental Board - Oil Shale Mining and Use Statistics:
                <a href="https://keskkonnaamet.ee/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(EL) Greece-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Greek matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                ADMIE (IPTO) - Independent Power Transmission Operator Monthly Energy Reports (2025):
                <a href="https://www.admie.gr/en/market/reports/monthly-energy-reports" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                RAAEY (Regulatory Authority for Energy, Waste and Water) - Annual Market Monitoring:
                <a href="https://www.rae.gr/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                HEnEx (Hellenic Energy Exchange) - Day-Ahead and Intraday Market Statistics:
                <a href="https://www.enexgroup.gr/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Environment and Energy - Updated National Energy and Climate Plan (NECP 2025):
                <a href="https://ypen.gov.gr/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                The Green Tank - Trends in Greek Electricity Production (ADMIE Analysis 2025):
                <a href="https://thegreentank.gr/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ELSTAT (Hellenic Statistical Authority) - Energy Balance Surveys:
                <a href="https://www.statistics.gr/en/statistics/-/publication/SRE04/-" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                DAPEEP - Renewable Energy Special Account Bulletins:
                <a href="https://www.dapeep.gr/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(ES) Spain-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Spanish matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                Red Eléctrica de España (REE) - El sistema eléctrico español:
                <a href="https://www.ree.es/es/datos/publicaciones" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Red Eléctrica de España (REE) - Series estadísticas nacionales (Data-hub):
                <a href="https://www.ree.es/es/datos/aldia" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                International Energy Agency (IEA) - Spain Energy Policy Review:
                <a href="https://www.iea.org/reports/spain-2021" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                MITECO - Plan Nacional Integrado de Energía y Clima (PNIEC) 2021-2030:
                <a href="https://www.miteco.gob.es/es/prensa/pniec.aspx" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                OMIE - Iberian Electricity Market Operator (Spot Price and Volume Data):
                <a href="https://www.omie.es/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                CNMC - Comisión Nacional de los Mercados y la Competencia (Energy Supervision Reports):
                <a href="https://www.cnmc.es/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Enagás - National Gas System and Hydrogen Backbone Development Data:
                <a href="https://www.enagas.es/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(FI) Finland-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Finnish matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                Business Finland - Hydrogen Economy Growth:
                <a href="https://www.businessfinland.fi/4961e3/globalassets/julkaisut/future-watch-growth-opportunities-in-the-hydrogen-economy.pdf" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Finnish Bioenergy Association - Tietopankki:
                <a href="https://www.bioenergia.fi/tietopankki/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Finnish Energy - Electricity Generation Data:
                <a href="https://energia.fi/en/energy-sector-in-finland/energy-production/electricity-generation/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Renewables Finland - Wind and Solar Power Expansion:
                <a href="https://suomenuusiutuvat.fi/en/finnish-wind-power-association-expands-its-activities-to-solar-power/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                International Energy Agency (IEA) - Finland Heat Pumps:
                <a href="https://www.iea.org/energy-system/buildings/heat-pumps" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Statistics Finland - Production of Heat Surveys:
                <a href="https://stat.fi/en/surveys/ene" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Statistics Finland - Energy Supply and Consumption:
                <a href="https://stat.fi/en/statistics/ehk" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Fingrid - Electricity Market Transparency:
                <a href="https://www.fingrid.fi/en/electricity-market/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(FR) France-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The French matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                RTE - Eco2mix: Detailed Electricity Generation by Technology:
                <a href="https://www.rte-france.com/en/eco2mix" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministère de la Transition Énergétique - Chiffres clés de l'énergie:
                <a href="https://www.statistiques.developpement-durable.gouv.fr/energie" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                International Energy Agency (IEA) - France Country Review:
                <a href="https://www.iea.org/countries/france" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Eurostat - Energy Balances for France (European Commission):
                <a href="https://ec.europa.eu/eurostat/web/energy" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                EDF - Generation Portfolio and Nuclear Availability Reports:
                <a href="https://www.edf.fr/en/the-edf-group/industrial-performance/nuclear-generation" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                CRE (Commission de Régulation de l'Énergie) - Market Reports:
                <a href="https://www.cre.fr/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                SDES (Service de la donnée et des études statistiques) - Energy Data Portal:
                <a href="https://www.statistiques.developpement-durable.gouv.fr/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(GE) Georgia-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Georgian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                GSE (Georgian State Electrosystem) - Transmission and Dispatch Data:
                <a href="https://www.gse.com.ge/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ESCO (Electricity System Commercial Operator) - Market Statistics:
                <a href="https://esco.ge/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Geostat (National Statistics Office of Georgia) - Energy Balances:
                <a href="https://www.geostat.ge/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                GNERC (Georgian National Energy and Water Supply Regulatory Commission):
                <a href="https://gnerc.org/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Economy and Sustainable Development - Energy Department:
                <a href="http://www.economy.ge/?lang=en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                IEA - Georgia Energy Profile and Policy Review (2025/2026):
                <a href="https://www.iea.org/countries/georgia" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Energy Community Secretariat - Georgia Monitoring Reports:
                <a href="https://www.energy-community.org/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(HR) Croatia-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Croatian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                HOPS (Croatian Transmission System Operator) Data:
                <a href="https://www.hops.hr/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                HEP (Hrvatska elektroprivreda) Generation Portfolio:
                <a href="https://www.hep.hr/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                HERA (Croatian Energy Regulatory Agency) Reports:
                <a href="https://www.hera.hr/en/html/index.html" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                EIHP (Energy Institute Hrvoje Požar) - Energy in Croatia:
                <a href="https://www.eihp.hr/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                CROPEX (Croatian Energy Exchange) Market Statistics:
                <a href="https://www.cropex.hr/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                DZS (Croatian Bureau of Statistics) Energy Balance:
                <a href="https://podaci.dzs.hr/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Economy and Sustainable Development:
                <a href="https://mingor.gov.hr/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(HU) Hungary-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Hungarian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                MAVIR (Hungarian Independent Transmission Operator Company Ltd.) - System Adequacy and Market Data:
                <a href="https://www.mavir.hu/en/web/mavir-en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                MEKH (Hungarian Energy and Public Utility Regulatory Authority) - Annual and Quarterly Reports:
                <a href="http://www.mekh.hu/home" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Paks Nuclear Power Plant - Operational and Generation Statistics:
                <a href="https://atomeromu.mvm.hu/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                HUPX (Hungarian Power Exchange) - Market Price and Volume Indices:
                <a href="https://hupx.hu/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Energy - Updated National Energy and Climate Plan (NECP 2024-2025):
                <a href="https://kormany.hu/energiaugyi-miniszterium" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                International Energy Agency (IEA) - Hungary Energy Policy Review (2025):
                <a href="https://www.iea.org/countries/hungary" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                HCSO (Hungarian Central Statistical Office) - Energy Balances:
                <a href="https://www.ksh.hu/energy" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(IE) Ireland-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Irish matrix was constructed and refined using the following comprehensive localized sources:
        <ol>
            <li>
                EirGrid - Smart Grid Dashboard (Live Generation Data):
                <a href="https://www.smartgriddashboard.com/all/generation/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                SEAI - Energy in Ireland 2024 Report & Data:
                <a href="https://www.seai.ie/publications/Energy-in-Ireland-2024.pdf" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                EirGrid &amp; SONI - Ten-Year Generation Capacity Statement:
                <a href="https://cms.soni.ltd.uk/sites/default/files/media/documents/SONI-Generation-Capacity-Statement-2023-2032.pdf" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                DECC - National Hydrogen Strategy (2023):
                <a href="https://assets.gov.ie/static/documents/national-hydrogen-strategy.pdf" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Wind Energy Ireland - Monthly Operations Dashboard:
                <a href="https://windenergyireland.com/about-wind/more-resources/monthly-dashboard" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                EPA - Circular Economy and Waste Statistics:
                <a href="https://www.epa.ie/publications/monitoring--assessment/waste/national-waste-statistics/circular-economy-and-waste-statistics-highlights-report-2022.php" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ESB - Generation and Trading Portfolio:
                <a href="https://esb.ie/what-we-do/generation-and-trading" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                International Energy Agency (IEA) - Ireland Country Review:
                <a href="https://www.iea.org/countries/ireland" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                SEAI - Tidal and Current Energy Resources:
                <a href="https://www.seai.ie/publications/Tidal_Current_Energy_Resources_in_Ireland_Report.pdf" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                SEAI - Geothermal Energy Technologies:
                <a href="https://www.seai.ie/technologies/geothermal-energy/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(IS) Iceland-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Icelandic matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                Orkustofnun (National Energy Authority) - Energy Statistics and Data Library:
                <a href="https://orkustofnun.is/en/information/numerical_data/electricity" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Landsnet (Icelandic TSO) - Transmission System and Hourly Generation Data:
                <a href="https://www.landsnet.is/english/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Landsvirkjun (National Power Company) - Hydropower and Geothermal Capacity Reports:
                <a href="https://www.landsvirkjun.com/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Statistics Iceland (Hagstofa Íslands) - Physical Energy Flow Accounts (PEFA):
                <a href="https://statice.is/statistics/environment/energy/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                IEA - Iceland Country Profile and Energy Balances (2025/2026):
                <a href="https://www.iea.org/countries/iceland" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Environment and Energy Agency of Iceland - Energy Transition Progress Reports:
                <a href="https://orkustofnun.is/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                University of Iceland - Geothermal Power and District Heating Research:
                <a href="https://english.hi.is/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(IT) Italy-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Italian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                GSE - Rapporto Statistico: Energia da Fonti Rinnovabili in Italia:
                <a href="https://www.gse.it/dati-e-scenari/rapporti" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                International Energy Agency (IEA) - Italy Energy Policy Review:
                <a href="https://www.iea.org/reports/italy-2023" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Terna S.p.A. - Dati statistici sull'energia elettrica in Italia:
                <a href="https://www.terna.it/it/sistema-elettrico/statistiche" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                MASE - Piano Nazionale Integrato per l'Energia e il Clima (PNIEC):
                <a href="https://www.mase.gov.it/pagina/pniec-piano-nazionale-integrato-l-energia-e-il-clima" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ARERA - Annual Report on State of Services and Market Monitoring:
                <a href="https://www.arera.it/it/inglese/index.htm" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                GME - Gestore dei Mercati Energetici (Market Results and Price Statistics):
                <a href="https://www.mercatoelettrico.org/En/Default.aspx" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ENI - World Energy Review (Italian focus on Gas and Oil trends):
                <a href="https://www.eni.com/en-IT/operations/world-energy-review.html" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(LT) Lithuania-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Lithuanian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                Litgrid (Transmission System Operator) - Annual Power System Data and 10-Year Development Plan:
                <a href="https://www.litgrid.eu/index.php/power-system/power-system-information/national-electricity-demand-and-generation/3523" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Energy - National Energy Independence Strategy (NEIS) and Updated NECP (2024/2025):
                <a href="https://enmin.lrv.lt/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                IEA - Lithuania 2025 Energy Policy Review (July 2025):
                <a href="https://www.iea.org/reports/lithuania-2025" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ignitis Group - Strategic Plan 2025–2028 and Green Capacities Portfolio:
                <a href="https://ignitisgrupe.lt/en/about-us/strategic-plan" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                State Data Agency (Lietuvos statistika) - Energy Balance and Renewable Share Reports:
                <a href="https://stat.gov.lt/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                NERC (National Energy Regulatory Council) - Market Monitoring and Capacity Mechanism Data:
                <a href="https://www.regula.lt/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                LVEA (Lithuanian Wind Power Association) - 2025 Generation and Capacity Statistics:
                <a href="https://lvea.lt/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(LU) Luxembourg-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Luxembourgish matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                ILR (Luxembourg Regulatory Institute) - 2025/2026 Annual Energy Market Performance and Tariff Methodology Reports:
                <a href="https://www.ilr.lu/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                STATEC (National Institute of Statistics) - 2025 Energy Balance and Decoupling Metrics:
                <a href="https://statistiques.public.lu/en.html" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Creos Luxembourg - 2025 Network Development Plan and "Project 380" Progress:
                <a href="https://www.creos-net.lu/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                International Energy Agency (IEA) - Luxembourg Energy Policy Review and 2025 Update:
                <a href="https://www.iea.org/reports/luxembourg-2020" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of the Economy (Department of Energy) - Integrated National Energy and Climate Plan (NECP 2024/2025 Version):
                <a href="https://meco.gouvernement.lu/en.html" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                SEO (Société Électrique de l'Our) - Vianden Pumped Storage Plant Operational Parameters and Flexibility Data:
                <a href="https://www.seo.lu/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Encevo Group - Annual Results 2025 (Renewable Asset Expansion in Wind and Solar):
                <a href="https://www.encevo.eu/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(LV) Latvia-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Latvian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                AST (Augstsprieguma tīkls) - Transmission System Operator Reports:
                <a href="https://www.ast.lv/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Latvenergo - Generation Portfolio (Hydropower and Thermal):
                <a href="https://latvenergo.lv/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Central Statistical Bureau of Latvia (CSB) - Energy Statistics:
                <a href="https://stat.gov.lv/en/statistics-themes/business-sectors/energy" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Public Utilities Commission (PUC/SPRK) - Energy Market Monitoring:
                <a href="https://www.sprk.gov.lv/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Economics - National Energy and Climate Plan (NECP 2030):
                <a href="https://www.em.gov.lv/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Conexus Baltic Grid - Natural Gas Storage and Transmission Data:
                <a href="https://www.conexus.lv/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                International Energy Agency (IEA) - Latvia Energy Profile:
                <a href="https://www.iea.org/countries/latvia" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(MD) Moldova-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Moldovan matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                Moldelectrica (Transmission System Operator) - Ten-Year Network Development Plan (TYNDP 2025–2034):
                <a href="https://moldelectrica.md/en/activity/transport_network/network_development" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Energy - Integrated National Energy and Climate Plan (NECP) 2025-2030:
                <a href="https://energie.gov.md/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ANRE (National Agency for Energy Regulation) - Annual Monitoring Reports on the Electricity and Gas Markets:
                <a href="https://www.anre.md/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Energy Community Secretariat - Moldova 2025 Annual Implementation Report:
                <a href="https://www.energy-community.org/implementation/Moldova.html" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Energocom - Electricity Procurement and Market Diversification Reports (2024/2025):
                <a href="https://www.energocom.md/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                IEA - Moldova Energy Profile and System Integration of Renewables Roadmap:
                <a href="https://www.iea.org/countries/moldova" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                National Bureau of Statistics (BNS) - Annual Energy Balances of the Republic of Moldova:
                <a href="https://statistica.gov.md/en/statistics-by-domains/economy/energy" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(ME) Montenegro-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Montenegrin matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                CGES (Crnogorski elektroprenosni sistem) - Transmission System Operator Annual Reports (2024/2025):
                <a href="http://www.cges.me/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                EPCG (Elektroprivreda Crne Gore) - Generation Portfolio and Environmental Reconstruction Data:
                <a href="https://www.epcg.com/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                MONSTAT (Statistical Office of Montenegro) - Annual Energy Balances:
                <a href="https://www.monstat.org/eng/page.php?id=32&mid=64" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Energy Community Secretariat - Montenegro 2025 Implementation Report:
                <a href="https://www.energy-community.org/implementation/Montenegro.html" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Energy and Mining - National Energy and Climate Plan (NECP 2025-2030):
                <a href="https://www.gov.me/en/mue" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                TERNA S.p.A - Monita (Italy-Montenegro) Interconnector Operational Statistics:
                <a href="https://www.terna.it/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                REGAGEN (Energy and Water Regulatory Agency) - State of the Energy Market Reports:
                <a href="https://regagen.me/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(MK) North Macedonia-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The North Macedonian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                MEPSO (Transmission System Operator) - Annual Power System Reports and 10-Year Network Development Plan:
                <a href="https://www.mepso.com.mk/en-us/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ERC (Energy Regulatory Commission) - Annual Report on the Operations of the Energy and Water Services (2024/2025):
                <a href="https://www.erc.org.mk/page_en.aspx?id=342" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                State Statistical Office (SSO) - Energy Balances 2024 (Preliminary Data):
                <a href="https://www.stat.mk/en/stat/industry-energy-and-environment/energy/energy-balances/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Energy, Mining and Mineral Resources - National Energy and Climate Plan (NECP) and 2025 Energy Law Amendments:
                <a href="https://energie.gov.mk/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Energy Community Secretariat - North Macedonia 2025 Annual Implementation Report:
                <a href="https://www.energy-community.org/implementation/North_Macedonia.html" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ESM (Elektrani na Severna Makedonija) - Thermal and Hydro Production Statistics:
                <a href="https://www.esm.com.mk/?lang=en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                MEMO (National Electricity Market Operator) - Day-Ahead Market Trading Data:
                <a href="https://memo.mk/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(MT) Malta-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Maltese matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                NSO (National Statistics Office) Malta - Electricity Supply 2024/2025 News Releases:
                <a href="https://nso.gov.mt/electricity-supply-2024/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                REWS (Regulator for Energy and Water Services) - 2025 National Report on Electricity and Gas Markets:
                <a href="https://www.rews.org.mt/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Enemalta plc - Generation Portfolio and Interconnector Operational Data:
                <a href="https://www.enemalta.com.mt/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Energy and Water Agency (EWA) - Updated National Energy and Climate Plan (NECP 2021-2030, 2025 Update):
                <a href="https://energywateragency.gov.mt/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                European Commission - Assessment of the Final Updated NECP for Malta (2025):
                <a href="https://commission.europa.eu/publications/malta-final-updated-necp-2021-2030-submitted-2025_en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Interconnect Malta (ICM) - Second Malta-Italy Interconnector (IC2) Technical Specifications:
                <a href="https://interconnectmalta.com.mt/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Odyssee-Mure - Malta Energy Profile (January 2026):
                <a href="https://www.odyssee-mure.eu/publications/efficiency-trends-policies-profiles/malta-country-profile-english.pdf" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(NL) Netherlands-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Dutch matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                CBS - Energiebalans; aanbod, omzetting en verbruik:
                <a href="https://www.cbs.nl/en-gb/figures/detail/83109ENG" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                RVO (Netherlands Enterprise Agency) - Energy and Climate Reports:
                <a href="https://english.rvo.nl/topics/energy-agreement/energy-and-climate-reports" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                International Energy Agency (IEA) - Netherlands Policy Review:
                <a href="https://www.iea.org/reports/the-netherlands-2024" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Eurostat - Energy Database (European Commission):
                <a href="https://ec.europa.eu/eurostat/web/energy/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                TenneT - Network Development Plan and Market Data:
                <a href="https://www.tennet.eu/energy-system/transparency" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Energieopwek.nl - Real-time Renewable Generation Data:
                <a href="https://energieopwek.nl/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                PBL (Netherlands Environmental Assessment Agency) - Climate and Energy Outlook (KEV):
                <a href="https://www.pbl.nl/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(NO) Norway-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Norwegian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                Statistics Norway (SSB) - Energy Balance and Electricity Statistics:
                <a href="https://www.ssb.no/en/energi-og-industri/statistikker/energibalanse" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                NVE (Norwegian Water Resources and Energy Directorate) - Energy Data and Power Market Statistics:
                <a href="https://www.nve.no/search/?term=energy%20data%20and%20statistics/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                International Energy Agency (IEA) - Norway Energy Policy Review:
                <a href="https://www.iea.org/reports/norway-2022" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Norwegian Ministry of Petroleum and Energy - Long-term perspectives for the Norwegian energy sector:
                <a href="https://www.regjeringen.no/en/topics/energy/id437/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Statnett - Grid Development Plan and Nordic Market Insights:
                <a href="https://www.statnett.no/en/for-stakeholders-in-the-power-guide/plan-and-develop-the-grid/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Nord Pool - Day-Ahead and Intraday Market Data for the NO Price Areas:
                <a href="https://www.nordpoolgroup.com/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Equinor - Energy Perspectives and Offshore Wind Development Reports:
                <a href="https://www.equinor.com/sustainability/energy-perspectives" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(PL) Poland-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Polish matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                Ministerstwo Klimatu i Środowiska - Energy Policy of Poland until 2040 (PEP2040) &amp; 2025 Updates:
                <a href="https://www.gov.pl/web/klimat/polityka-energetyczna-polski" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Polskie Sieci Elektroenergetyczne (PSE) - 2025 System Operational Data and 10-Year Network Development Plan:
                <a href="https://www.pse.pl/dane-systemowe" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Główny Urząd Statystyczny (Statistics Poland) - 2025 Energy Statistics and Fuel Balance:
                <a href="https://stat.gov.pl/en/topics/environment-energy/energy/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                International Energy Agency (IEA) - Poland Energy Policy Review and Decarbonization Roadmap:
                <a href="https://www.iea.org/reports/poland-2022" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Eurostat - Energy Database (European Commission Statistical Integration 2026):
                <a href="https://ec.europa.eu/eurostat/web/energy/database" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ARE (Agencja Rynku Energii) - Statistical Bulletins on the Power Sector and Heat Market:
                <a href="https://www.are.waw.pl/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                TGE (Towarowa Giełda Energii) - Day-Ahead and Intraday Electricity Trading Data:
                <a href="https://tge.pl/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(PT) Portugal-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Portuguese matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                REN (Redes Energéticas Nacionais) - 2025 Annual Electricity Market Report and Transmission Data:
                <a href="https://www.ren.pt/en-gb/media/news/electricity-consumption-reaches-highest-ever-level-in-2025" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                DGEG (Direção-Geral de Energia e Geologia) - Energy Balance and Statistics (January 2026 Update):
                <a href="https://www.dgeg.gov.pt/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                APREN (Portuguese Renewable Energy Association) - Renewable Electricity Bulletin (2025 Year in Review):
                <a href="https://www.apren.pt/en/knowledge/statistical-data/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ADENE (Agency for Energy) - National Energy Balance and Efficiency Monitoring:
                <a href="https://www.adene.pt/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                IEA - Portugal 2025/2026 Energy Policy and System Integration Profile:
                <a href="https://www.iea.org/countries/portugal" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ERSE (Energy Services Regulatory Authority) - 2025 Electricity Market Monitoring Report:
                <a href="https://www.erse.pt/en/home/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                EDP (Energias de Portugal) - Hydroelectric and Wind Portfolio Technical Data:
                <a href="https://www.edp.com/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(RO) Romania-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Romanian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                Transelectrica (TSO) - 2025 Annual Operational Results and 10-Year Network Development Plan:
                <a href="https://www.transelectrica.ro/en/web/tel/raportari-periodice" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ANRE (National Energy Regulatory Authority) - Prosumer Statistics and 2025 Market Monitoring:
                <a href="https://www.anre.ro/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Energy - Final Updated National Energy and Climate Plan (NECP 2021-2030, 2024/2025 Version):
                <a href="https://energie.gov.md/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Nuclearelectrica (SNN) - Cernavodă Unit 1 Refurbishment Schedule and Unit 3&amp;4 Expansion Roadmap:
                <a href="https://www.nuclearelectrica.ro/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                National Institute of Statistics (INS) - Energy Balance and Resource Supply (2025 Preliminary):
                <a href="https://insse.ro/cms/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Energy Community Secretariat - Romania 2025 Implementation Review (Regional Integration Focus):
                <a href="https://www.energy-community.org/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                RPIA (Romanian Photovoltaic Industry Association) - 2025 Solar Growth and CfD Auction Analysis:
                <a href="https://rpia.ro/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(RS) Serbia-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Serbian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                EMS JSC (Elektromreža Srbije) - Transmission System Operator Annual Reports and ENTSO-E TYNDP Contributions (2025/2026):
                <a href="https://ems.rs/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                EPS (Elektroprivreda Srbije) - Power Generation Portfolio and "Green Road" Decarbonization Strategy:
                <a href="https://www.eps.rs/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                AERS (Energy Agency of the Republic of Serbia) - 2024/2025 Annual Reports on the Status of the Energy Sector:
                <a href="https://www.aers.rs/index.asp?l=2" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Mining and Energy - Integrated National Energy and Climate Plan (INECP) for the Period up to 2030:
                <a href="https://mre.gov.rs/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Energy Community Secretariat - Serbia 2025 Annual Implementation Report:
                <a href="https://www.energy-community.org/implementation/Serbia.html" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                SEEPEX (South East European Power Exchange) - Day-Ahead and Intraday Market Volumes and Price Data:
                <a href="https://seepex-ad.com/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Statistical Office of the Republic of Serbia (SORS) - Annual Energy Balances for Electricity and Heat:
                <a href="https://www.stat.gov.rs/en-us/oblasti/energetika/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(RU) Russian Federation-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Russian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                System Operator of the Unified Energy System (SO UPS) - 2025 Annual Operational Data and 2026 Dispatch Reports:
                <a href="https://www.so-ups.ru/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Energy (Minenergo) - Energy Strategy of the Russian Federation to 2050 (2025 Plenary Update):
                <a href="https://minenergo.gov.ru/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Rosenergoatom (Nuclear Utility) - 2025 Year-to-Date Generation and Operational Status of NPP Units:
                <a href="https://www.rosenergoatom.ru/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                RusHydro - 2025 Annual Production Results for Hydroelectric and Thermal (Far East) Facilities:
                <a href="https://eng.rushydro.ru/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Federal State Statistics Service (Rosstat) - 2025 Energy Balance and Fuel Consumption Statistics:
                <a href="https://rosstat.gov.ru/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Gazprom - Domestic Gas Supply and Power Sector Consumption Profiles (2025/2026):
                <a href="https://www.gazprom.com/projects/power-gen/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                IEA/Enerdata - Russia Energy Profile (2025 Analysis of Production and Demand Slumps):
                <a href="https://www.enerdata.net/publications/daily-energy-news/russias-power-consumption-drops-08-2025-generation-falls-12.html" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(SE) Sweden-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Swedish matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                Energimyndigheten - Energy in Sweden: Facts and Figures 2023/2024:
                <a href="https://www.energimyndigheten.se/en/news/2023/energy-in-sweden---facts-and-figures-2023/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                IEA - Sweden 2024 Analysis and Energy Policy Review:
                <a href="https://www.iea.org/reports/sweden-2024" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Svenska kraftnät (SVK) - Nordic Grid Development Perspective:
                <a href="https://www.svk.se/press-och-nyheter/nyheter/allmanna-nyheter/2023/ny-rapport-nordic-grid-development-perspective-2023/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Eurostat - Shedding Light on Energy in Europe &amp; Database (2024 Editions):
                <a href="https://ec.europa.eu/eurostat/web/energy/database" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                European Commission - EU Energy in Figures Statistical Data (Oct 2024):
                <a href="https://energy.ec.europa.eu/news/eu-energy-figures-statistical-data-eu-energy-sector-2024-10-14_en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Government of Sweden - Draft Updated National Energy and Climate Plan (NECP):
                <a href="https://www.government.se/contentassets/e731726022cd4e0b8ffa0f8229893115/swedens-draft-integrated-national-energy-and-climate-plan/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Swedish Government - Support for Ambitious Climate Plans (May 2025):
                <a href="https://www.government.se/articles/2025/05/the-swedish-offer-to-support-ambitious-climate-plans/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(SI) Slovenia-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Slovenian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                SURS (Statistical Office of the Republic of Slovenia) - Monthly and Annual Energy Statistics (2025/2026 Releases):
                <a href="https://www.stat.si/StatWeb/en/Field/Index/5/88" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of the Environment, Climate and Energy - Updated National Energy and Climate Plan (NECP 2025):
                <a href="https://www.gov.si/en/topics/national-energy-and-climate-plan/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ELES (Transmission System Operator) - National Grid Development Plan and 2026 Operational Data:
                <a href="https://www.eles.si/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                International Energy Agency (IEA) - Slovenia Country Profile and Energy Mix Analysis:
                <a href="https://www.iea.org/countries/slovenia/energy-mix" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                NEK (Krško Nuclear Power Plant) - Technical Parameters and Joint Production Statistics:
                <a href="https://www.nek.si/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Borzen - Slovenian Electricity Market Operator (Market Data and RES Support Schemes):
                <a href="https://www.borzen.si/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                EZS (Energy Association of Slovenia) - Energy in Slovenia and Worldwide 2025 Report:
                <a href="https://ezs.si/publications/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ELES (Combined Transmission &amp; Distribution System Operator) - 2025/2026 Network Development Plan and System Stability Reports:
                <a href="https://www.eles.si/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                SURS (Statistical Office of the Republic of Slovenia) - Monthly Energy Supply and Production Statistics (Dec 2025 Update):
                <a href="https://www.stat.si/StatWeb/en/Field/Index/5" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                EZS (Energy Association of Slovenia) - "Energy in Slovenia 2025" Comprehensive Overview and Second Edition:
                <a href="https://ezs.si/publications/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of the Environment, Climate and Energy - Final Updated National Energy and Climate Plan (NECP 2021-2030, Jan 2025):
                <a href="https://www.gov.si/en/topics/national-energy-and-climate-plan/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                IEA - Slovenia 2025 Energy Mix and Low-Carbon Power Data:
                <a href="https://www.iea.org/countries/slovenia" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                HSE (Holding Slovenske elektrarne) - Operational Data for Šoštanj Thermal and Drava/Soča Hydro Cascades:
                <a href="https://www.hse.si/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                NEK (Krško Nuclear Power Plant) - Annual Operational Performance and 2025 Availability Reports:
                <a href="https://www.nek.si/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(SK) Slovakia-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Slovak matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                SEPS (Slovenská elektrizačná prenosová sústava) - 2025 Yearly Operational Data and Power Balance:
                <a href="https://www.sepsas.sk/en/for-partners/control-centre/operational-data/yearly-operational-data/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Slovenské elektrárne (SE) - Nuclear and Hydro Generation Portfolio (Mochovce 3 &amp; 4 Progress):
                <a href="https://www.seas.sk/en/about-us/our-power-plants/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ÚRSO (Regulatory Office for Network Industries) - 2025 National Report on Energy Market Regulation:
                <a href="https://www.urso.gov.sk/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Statistical Office of the Slovak Republic - Statistical Yearbook 2025 (Energy Chapter 18):
                <a href="https://slovak.statistics.sk/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Economy - Final Updated National Energy and Climate Plan (NECP 2025 Assessment):
                <a href="https://www.mhsr.sk/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                IEA - Slovakia 2025/2026 Energy Policy Review and Nuclear Integration Analysis:
                <a href="https://www.iea.org/countries/slovakia" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                OKTE (Short-term Electricity Market Organizer) - Day-ahead and Intraday Market Statistics:
                <a href="https://www.okte.sk/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(TR) Turkey-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Turkish matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                International Energy Agency (IEA) - Turkey 2021: Energy Policy Review:
                <a href="https://www.iea.org/reports/turkey-2021" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Turkish Statistical Institute (TurkStat) - 2024/2025 Energy Statistics and National Balance:
                <a href="https://data.tuik.gov.tr/Kategori/GetKategori?p=cevre-ve-enerji-116" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Energy and Natural Resources - National Energy Plan and Strategic Policy Framework:
                <a href="https://enerji.gov.tr/bilgi-merkezi-enerji-politikalari" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Eurostat - Energy Database (Comparative European and Candidate Country Data):
                <a href="https://ec.europa.eu/eurostat/web/energy/data/database" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                TEİAŞ (Turkish Electricity Transmission Corporation) - 2025 Load and Capacity Statistics:
                <a href="https://www.teias.gov.tr/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                EPİAŞ (EXIST) - Energy Exchange Istanbul Market Results and Spot Prices:
                <a href="https://www.epias.com.tr/en/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                DSI (General Directorate of State Hydraulic Works) - Reservoir Levels and Hydroelectric Capability Data:
                <a href="https://www.dsi.gov.tr/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(UA) Ukraine-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Ukrainian matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                NPC Ukrenergo - 2026 Operational Data and Transmission Tariff Reports:
                <a href="https://www.pse.pl/dane-systemowe" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                IEA - Ukraine's Energy Security: Pre-Winter 2025/26 Assessment:
                <a href="https://www.iea.org/reports/ukraines-energy-security" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Green Deal Ukraina - Electricity and Gas Supply in Ukraine: Winter 2025/26 Outlook:
                <a href="https://greendealukraina.org/assets/images/reports/electricity-and-gas-supply-in-ukraine-winter-2025-26.pdf" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Cabinet of Ministers of Ukraine - Regional Energy Sustainability Plans (Feb 2026):
                <a href="https://www.kmu.gov.ua/en/news/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Energy Community Secretariat - Ukraine Energy Market Monitoring Report 2025:
                <a href="https://www.energy-community.org/dam/jcr:bf9d6221-9290-42a8-9c55-f2f4d82a2b5d/Ukraine_IR25CP.pdf" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ENTSO-E - Transparency Platform (Real-time Cross-border Flows SE/UKR):
                <a href="https://transparency.entsoe.eu/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Energy of Ukraine - Strategy for Decentralized Energy Resources 2030:
                <a href="https://mev.gov.ua/en" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(UK) United Kingdom-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The United Kingdom matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                Department for Energy Security and Net Zero - Digest of UK Energy Statistics (DUKES):
                <a href="https://www.gov.uk/government/collections/digest-of-uk-energy-statistics-dukes" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                The Crown Estate - Offshore Wind Operational and Pipeline Data:
                <a href="https://www.thecrownestate.co.uk/our-business/offshore-wind/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                National Grid ESO - Future Energy Scenarios (FES 2025/2026) and Live Generation Data:
                <a href="https://www.nationalgrideso.com/future-energy/future-energy-scenarios" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ofgem - State of the Energy Market 2025 Report:
                <a href="https://www.ofgem.gov.uk/publications/state-energy-market-2025" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                NESO (National Energy System Operator) - Winter Outlook and Interconnection Capacity Analysis:
                <a href="https://www.neso.energy/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Climate Change Committee (CCC) - Monitoring Progress in Reducing Emissions (2025 Report):
                <a href="https://www.theccc.org.uk/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                RenewableUK - Wind and Solar Capacity Statistics (January 2026):
                <a href="https://www.renewableuk.com/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        <b>(XK) Kosovo-Specific Data Sources & Literature Review</b>
    </span>
    <div style="border-top: 1px solid #000099; font-size: 10px; padding: 10px;">
        The Kosovo matrix was constructed and refined using the following localized and regional sources:
        <ol>
            <li>
                KOSTT (Transmission, System and Market Operator) - Transmission Development Plan 2023-2032:
                <a href="https://www.kostt.com" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ERO (Energy Regulatory Office) - Annual Report on the Energy Sector (2024/2025):
                <a href="https://www.ero-ks.org/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Ministry of Economy - Energy Strategy of the Republic of Kosovo 2022–2031:
                <a href="https://me.rks-gov.net/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                Energy Community Secretariat - Kosovo* Annual Implementation Report (2025):
                <a href="https://www.energy-community.org/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                ALPEX (Albanian Power Exchange) - Kosovo-Albania Market Coupling Data:
                <a href="https://alpex.al/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                KEK (Kosovo Energy Corporation) - Thermal Power Plant Operational Statistics:
                <a href="https://kek-energy.com/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
            <li>
                KAS (Kosovo Agency of Statistics) - Annual Energy Balance Bulletins:
                <a href="https://ask.rks-gov.net/" target="_blank" rel="noopener noreferrer" style="color: #000033;">
                    Link
                </a>
            </li>
        </ol>
    </div>
</div>

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        B-06.2. Technology Match by Country Dictionary
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        The individual data frames containing <b>Fuel type proportions by Technology</b> for each modeled country are consolidated into a unified master dictionary. <br>
        This structure acts as the primary lookup table for formatting process.
    <br>
        By joining these data frames, the model establishes a standardized interface for cross-border energy mapping, allowing the optimization engine to retrieve technology-specific fuel weights for any given territory within the simulated region.
    </div>
</div>

<div style= "background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099; " >
 <span style= "font-weight: bold; font-size: 16px; " >
Section C Overview: Raw Data Ingestion
 </span >
 <div style= "border-top: 1px solid #000099; padding: 10px; " >
This section handles the initial data extraction. <br>
     It reads the raw PyPSA optimization outputs (generators, links, and storage units) from CSV files for each selected zone and target year, loading them into structured Pandas DataFrames for subsequent processing.
 </div >
 </div >
 <div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        C-01. Raw Data Uploading
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        The nodal capacities generated by the <b>PyPSA</b> optimization outputs are extracted and converted into structured data frames. <br>
        This transformation is essential for the subsequent alignment of generation limits with the Dispa-SET technical parameters.
            <div style="margin-left: 2em; font-size: 12px">
        <b>Computational Workflow:</b><br> This stage involves parsing the <b>network.generators</b>, <b>network.links</b> and <b>network.storage_units</b> attributes to ensure that capacity values ($P_{nom}$) are correctly localized to their respective nodes.
        <br>
        By converting raw PyPSA outputs into managed data frames, the system enables efficient filtering, scaling, and reformatting of the energy infrastructure data before it is ingested by the Dispa-SET simulation engine.
            </div>
    </div>
</div>

In [15]:
# Initialize dictionaries to store raw data for generators, links, and storage units per zone
generators_pypsa_raw_data_dictionary = {}
links_pypsa_raw_data_dictionary = {}
storage_units_pypsa_raw_data_dictionary = {}
# Loop through each zone to load its respective CSV files into the dictionaries
for zone in zone_names:
    # Construct the base path for the current zone and target year
    base_path = (
        power_plants_pypsa_raw_data_folder_path
        / str(data_target_year)
        / zone
    )
    # Load generators, links, and storage units data for the current zone
    generators_pypsa_raw_data_dictionary[zone] = pd.read_csv(
        base_path / f"{zone}_generators.csv"
    )
    links_pypsa_raw_data_dictionary[zone] = pd.read_csv(
        base_path / f"{zone}_links.csv"
    )
    storage_units_pypsa_raw_data_dictionary[zone] = pd.read_csv(
        base_path / f"{zone}_storage_units.csv"
    )
# Print the shape (rows, columns) of each DataFrame for every zone
for zone in zone_names:
    print(
        f"{zone} | "
        f"Generators: {generators_pypsa_raw_data_dictionary[zone].shape} | "
        f"Links: {links_pypsa_raw_data_dictionary[zone].shape} | "
        f"Storage Units: {storage_units_pypsa_raw_data_dictionary[zone].shape}"
    )

BE | Generators: (56, 43) | Links: (100, 61) | Storage Units: (2, 40)
FR | Generators: (54, 43) | Links: (118, 61) | Storage Units: (2, 40)
DE | Generators: (59, 43) | Links: (134, 61) | Storage Units: (2, 40)
NL | Generators: (58, 43) | Links: (101, 61) | Storage Units: (0, 40)
UK | Generators: (115, 43) | Links: (203, 61) | Storage Units: (2, 40)


<div style= "background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099; " >
 <span style= "font-weight: bold; font-size: 16px; " >
Section D Overview: Core Transformation & Data Cleaning
 </span >
 <div style= "border-top: 1px solid #000099; padding: 10px; " >
This section executes the core data transformation pipeline. <br>It filters relevant carriers, maps PyPSA attributes to Dispa-SET templates, isolates power-sector buses from multi-bus links, calculates storage capacities, resolves multi-technology and multi-fuel ambiguities, sanitizes invalid negative values, and fills missing techno-economic parameters using reference dictionaries.
 </div >
 </div >
 <div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
      D-01. Power Units Carriers Configuration
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        The <b>power_units_carriers</b> list defines which PyPSA carrier types (technologies/fuels) are active in the model. Carriers are selectively enabled or disabled using comment markers (<b>###</b> for disabled, no marker for active).<br> This flexible approach allows for rapid scenario configuration without deleting code.<br>
<b>Configuration Logic:</b>
    <div style="margin-left: 2em; font-size: 12px">
        <b>Active Carriers</b>: Represent the core technologies included in the model (wind, solar, hydro, fossil, nuclear, storage, hydrogen).<br>
        <b>Comment Conventions</b>: <code>###"carrier"</code> = disabled, <code>"carrier"</code> = active.<br>
        <b>VPP Mode</b>: When <b>VPP = True</b>, additional resistive heater carriers are dynamically appended to the list.
        <br>
        This modular design enables quick switching between different energy system configurations, supporting scenarios that range from pure power-sector models to fully integrated multi-energy systems.
    </div>
    </div>
</div>

In [16]:
# BASE POWER UNIT CARRIERS
# =============================================================================
power_units_carriers = [
###"AC",
###"DC",
    "offwind-dc",
    "solar",
    "offwind-ac",
    "onwind",
    "hydro",
    "PHS",
    "ror",
    "lignite",
    "coal",
    "oil",
###"uranium",
###"none",
###"co2",
###"co2 stored",
###"co2 sequestered",
###"gas",
###"H2",
###"battery",
###"Li ion",
###"residential rural heat",
###"residential rural water tanks",
    #"residential rural solar thermal",
###"services rural heat",
###"services rural water tanks",
    #"services rural solar thermal",
###"residential urban decentral heat",
###"residential urban decentral water tanks",
    #"residential urban decentral solar thermal",
###"services urban decentral heat",
###"services urban decentral water tanks",
    #"services urban decentral solar thermal",
###"urban central heat",
###"urban central water tanks",
    #"urban central solar thermal",
###"biogas",
###"solid biomass",
###"NH3",
###"methanol",
###"ammonia store",
###"H2 Store",
    "urban central solid biomass CHP CC",
###"residential urban decentral water tanks discharger",
###"SMR CC",
###"biomass to liquid",
###"services urban decentral gas boiler",
    "OCGT",
    #"battery charger",
###"services rural water tanks discharger",
###"electricity distribution grid",
###"services urban decentral biomass boiler",
###"residential urban decentral gas boiler",
    #"residential rural ground heat pump",
    #"residential rural resistive heater",
    #"Haber-Bosch",
###"gas pipeline",
###"biogas to gas CC",
###"biogas to gas",
###"services rural water tanks charger",
    #"residential rural air heat pump",
    #"methanolisation",
    #"residential urban decentral resistive heater",
###"gas for industry",
###"urban central water tanks discharger",
###"residential urban decentral water tanks charger",
###"H2 pipeline retrofitted",
    #"EV charger",
###"services urban decentral water tanks charger",
###"H2 pipeline",
###"residential rural gas boiler",
###"residential urban decentral biomass boiler",
###"agriculture machinery oil",
###"services rural biomass boiler",
    #"DAC",
    "H2 Fuel Cell",
    #"urban central air heat pump",
###"residential rural biomass boiler",
    #"residential urban decentral air heat pump",
###"Sabatier",
    #"services urban decentral resistive heater",
###"process emissions CC",
    #"services rural resistive heater",
    #"services urban decentral air heat pump",
    #"urban central gas CHP",
###"kerosene for aviation",
    "nuclear",
###"coal for industry",
    "urban central solid biomass CHP",
###"urban central water tanks charger",
###"Fischer-Tropsch",
###"naphtha for industry",
    "urban central gas CHP CC",
###"solid biomass for industry CC",
###"BioSNG",
    #"services rural ground heat pump",
###"residential rural water tanks discharger",
###"gas pipeline new",
    #"services rural air heat pump",
    "CCGT",
###"shipping methanol",
###"land transport oil",
###"urban central gas boiler",
###"services urban decentral water tanks discharger",
###"gas for industry CC",
    #"V2G",
    #"battery discharger",
###"solid biomass for industry",
    #"H2 Electrolysis",
###"shipping oil",
###"solid biomass transport",
###"ammonia cracker",
###"residential rural water tanks charger",
    "H2 turbine",
###"SMR",
    #"urban central resistive heater",
###"services rural gas boiler",
###"process emissions",
    "solar rooftop",
###"low voltage",
###"low-temperature heat for industry",
###"agriculture electricity",
###"land transport EV",
###"industry electricity",
###"H2 for industry",
###"electricity",
###"agriculture heat",
###"H2 for shipping",
###"services rural oil boiler",
###"services urban decentral oil boiler",
###"residential urban decentral oil boiler",
###"residential rural oil boiler",
###"offwind",
###"load",
###"land transport fuel cell",
]
# =============================================================================
# VPP-SPECIFIC CARRIERS
# =============================================================================
vpp_resistive_heater_carriers = [
    "residential rural resistive heater",
    "residential urban decentral resistive heater",
    "services urban decentral resistive heater",
    "services rural resistive heater",
    "urban central resistive heater",
]
# =============================================================================
# ACTIVATE VPP CARRIERS WHEN VPP = True
# =============================================================================
if VPP:
    power_units_carriers.extend(
        vpp_resistive_heater_carriers
    )

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        D-02. Filtering PyPSA Components by Carrier Type
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        The raw PyPSA data is filtered to retain only the active carrier types defined in <b>power_units_carriers</b>. <br>
        This process extracts generators, links, and storage units for each zone, creating separate dictionaries for each component type.
    </div>
</div>

In [17]:
### Initialize dictionaries to store filtered power unit data for generators, links, and storage units per zone
power_units_generators_pypsa_dictionary = {}
power_units_links_pypsa_dictionary = {}
power_units_storage_units_pypsa_dictionary = {}
### Filter data for each zone to include only rows where the carrier is in the power_units_carriers list
for zone in zone_names:
    ### Retrieve raw data for the current zone
    generators_df = generators_pypsa_raw_data_dictionary[zone]
    links_df = links_pypsa_raw_data_dictionary[zone]
    storage_units_df = storage_units_pypsa_raw_data_dictionary[zone]
    ### Filter and store generators, links, and storage units data for power units
    power_units_generators_pypsa_dictionary[zone] = (
        generators_df[generators_df["carrier"].isin(power_units_carriers)].copy()
    )
    power_units_links_pypsa_dictionary[zone] = (
        links_df[links_df["carrier"].isin(power_units_carriers)].copy()
    )
    power_units_storage_units_pypsa_dictionary[zone] = (
        storage_units_df[storage_units_df["carrier"].isin(power_units_carriers)].copy()
    )
### Display the size of all filtered DataFrames
print("\n" + "=" * 80)
print("FILTERED POWER-UNIT DATAFRAME SIZES")
print("=" * 80)
for zone in zone_names:
    ### Retrieve the shape of each filtered DataFrame for the current zone
    generators_shape = power_units_generators_pypsa_dictionary[zone].shape
    links_shape = power_units_links_pypsa_dictionary[zone].shape
    storage_units_shape = power_units_storage_units_pypsa_dictionary[zone].shape
    ### Print the shapes for generators, links, and storage units
    print(
        f"{zone} | "
        f"Generators: {generators_shape} | "
        f"Links: {links_shape} | "
        f"Storage Units: {storage_units_shape}"
    )
print("=" * 80)


FILTERED POWER-UNIT DATAFRAME SIZES
BE | Generators: (16, 43) | Links: (25, 61) | Storage Units: (2, 40)
FR | Generators: (16, 43) | Links: (35, 61) | Storage Units: (2, 40)
DE | Generators: (18, 43) | Links: (51, 61) | Storage Units: (2, 40)
NL | Generators: (17, 43) | Links: (24, 61) | Storage Units: (0, 40)
UK | Generators: (31, 43) | Links: (53, 61) | Storage Units: (2, 40)


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        D-03. PyPSA to Dispa-SET Column Mapping
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This dictionary defines the mapping between <b>PyPSA</b> attribute names (right) and <b>Dispa-SET</b> parameter names (left). It standardizes the data structure across both modeling frameworks.
    </div>
</div>

In [18]:
### Mapping between Dispa-SET column names and corresponding PyPSA column names (or list of possible column names)
# =============================================================================
pypsa_dispaset_column_mapping = {
    "Unit"                  : ["name"                                                                                         ],
    "PowerCapacity"         : ["p_nom_opt"                                                                                    ],
    "Efficiency"            : ["efficiency"      , "efficiency2", "efficiency3", "efficiency4", "efficiency_dispatch"         ],
    "Zone"                  : ["bus"             , "bus0"	    , "bus1"	   , "bus2"	      , "bus3"                , "bus4"],
    "Technology"            : ["carrier"                                                                                      ],
    "MinUpTime"             : ["min_up_time"                                                                                  ],
    "MinDownTime"           : ["min_down_time"                                                                                ],
    "RampUpRate"            : ["ramp_limit_up"                                                                                ],
    "RampDownRate"          : ["ramp_limit_down"                                                                              ],
    "StartUpCost"           : ["start_up_cost"                                                                                ],
    "NoLoadCost_pu"         : ["stand_by_cost"                                                                                ],
    "PartLoadMin"           : ["p_min_pu"                                                                                     ],
    "STOCapacity"           : ["max_hours"                                                                                    ],
    "STOSelfDischarge"      : ["standing_loss"                                                                                ],
    "STOChargingEfficiency" : ["efficiency_store"                                                                             ],
}
# =============================================================================

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       D-04. Filtering PyPSA DataFrames by Relevant Columns
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        After filtering by carrier type, the DataFrames are further refined to retain only the columns required for Dispa-SET compatibility. <br>This step reduces data complexity and ensures only relevant parameters are passed forward.
    </div>
</div>

In [19]:
### Filter the columns of the PyPSA power-unit DataFrames
# _____________________________________________________________________________
### Extract all PyPSA column names defined in the mapping dictionary
relevant_pypsa_columns = set(
    column
    for columns in pypsa_dispaset_column_mapping.values()
    for column in columns
)
### Filter each DataFrame to retain only the columns that exist in the mapping
for zone in zone_names:
    ### Filter generators DataFrame to keep only relevant columns
    power_units_generators_pypsa_dictionary[zone] = (
        power_units_generators_pypsa_dictionary[zone]
        .loc[
            :,
            [
                column
                for column in power_units_generators_pypsa_dictionary[zone].columns
                if column in relevant_pypsa_columns
            ]
        ]
        .copy()
    )
    ### Filter links DataFrame to keep only relevant columns
    power_units_links_pypsa_dictionary[zone] = (
        power_units_links_pypsa_dictionary[zone]
        .loc[
            :,
            [
                column
                for column in power_units_links_pypsa_dictionary[zone].columns
                if column in relevant_pypsa_columns
            ]
        ]
        .copy()
    )
    ### Filter storage units DataFrame to keep only relevant columns
    power_units_storage_units_pypsa_dictionary[zone] = (
        power_units_storage_units_pypsa_dictionary[zone]
        .loc[
            :,
            [
                column
                for column in power_units_storage_units_pypsa_dictionary[zone].columns
                if column in relevant_pypsa_columns
            ]
        ]
        .copy()
    )
### Display the current size of all filtered DataFrames
print("\n" + "=" * 90)
print("PYPSA POWER-UNIT DATAFRAMES AFTER COLUMN FILTERING")
print("=" * 90)
for zone in zone_names:
    ### Retrieve the shape of each filtered DataFrame for the current zone
    generators_shape = (
        power_units_generators_pypsa_dictionary[zone].shape
    )
    links_shape = (
        power_units_links_pypsa_dictionary[zone].shape
    )
    storage_units_shape = (
        power_units_storage_units_pypsa_dictionary[zone].shape
    )
    ### Print the shapes for generators, links, and storage units
    print(
        f"{zone} | "
        f"Generators: {generators_shape} | "
        f"Links: {links_shape} | "
        f"Storage Units: {storage_units_shape}"
    )
print("=" * 90)
# Print a summary message confirming the completion of column filtering
print(
    f"\nColumn filtering completed successfully. "
    f"{len(relevant_pypsa_columns)} relevant PyPSA columns "
    f"were retained according to the Dispa-SET mapping."
)


PYPSA POWER-UNIT DATAFRAMES AFTER COLUMN FILTERING
BE | Generators: (16, 12) | Links: (25, 19) | Storage Units: (2, 9)
FR | Generators: (16, 12) | Links: (35, 19) | Storage Units: (2, 9)
DE | Generators: (18, 12) | Links: (51, 19) | Storage Units: (2, 9)
NL | Generators: (17, 12) | Links: (24, 19) | Storage Units: (0, 9)
UK | Generators: (31, 12) | Links: (53, 19) | Storage Units: (2, 9)

Column filtering completed successfully. 24 relevant PyPSA columns were retained according to the Dispa-SET mapping.


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       D-05. Power-Sector Bus Filtering for PyPSA Links
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        Links in PyPSA represent multi-bus components (e.g., CHP plants, converters, storage) with up to 5 buses (bus0–bus4). <br>
        This process identifies and retains only the <b>power-sector (electricity)</b> bus for each link, along with its associated efficiency, while dropping non-electricity buses (e.g., heat, H2, CO2).
    </div>
</div>

In [20]:
### Bus to efficiency mapping
bus_to_efficiency = {
    "bus0": "efficiency",
    "bus1": "efficiency",
    "bus2": "efficiency2",
    "bus3": "efficiency3",
    "bus4": "efficiency4",
}
bus_columns = ["bus0", "bus1", "bus2", "bus3", "bus4"]
efficiency_columns = [
    "efficiency",
    "efficiency2",
    "efficiency3",
    "efficiency4",
]
### PROCESS ORIGINAL NESTED DICTIONARY IN PLACE
for zone in zone_names:
    links_df = power_units_links_pypsa_dictionary[zone].copy()

    # Identify equivalent prefix (e.g. UK -> GB, EL -> GR, otherwise fallback to zone)
    alt_acronym = ""
    if zone in zone_names_equivalences_dict:
        alt_acronym = (
            zone_names_equivalences_dict[zone]["Acronym"][0].strip().upper()
        )
    ### Valid prefixes to check against bus names
    valid_prefixes = tuple(
        p for p in set([zone.upper(), alt_acronym]) if p != ""
    )
    ### COLUMNS THAT ARE NOT BUS/EFFICIENCY COLUMNS
    columns_to_keep = [
        column
        for column in links_df.columns
        if column not in bus_columns and column not in efficiency_columns
    ]
    selected_buses = []
    selected_efficiencies = []
    selected_indices = []
    ### PROCESS EACH LINK
    for link_name, link_row in links_df.iterrows():
        selected_bus = None
        selected_efficiency = None
        ### CHECK BUS0 ... BUS4
        for bus_col in bus_columns:
            if bus_col not in links_df.columns:
                continue
            bus_value = link_row[bus_col]
            if pd.isna(bus_value):
                continue
            bus_value = str(bus_value).strip()
            bus_parts = bus_value.split()
            is_power_sector_bus = False
            if len(bus_parts) >= 2:
                first_part = bus_parts[0]
                ### Check if the bus node starts with either the zone key ("UK") or acronym ("GB")
                if first_part.startswith(valid_prefixes):
                    ### Normal electricity bus (e.g., GB0 0)
                    ### or low-voltage electricity bus (e.g., GB0 0 low voltage)
                    if len(bus_parts) == 2 or (
                        len(bus_parts) == 4
                        and bus_parts[2].lower() == "low"
                        and bus_parts[3].lower() == "voltage"
                    ):
                        is_power_sector_bus = True
            if not is_power_sector_bus:
                continue
            ### STORE THE SELECTED BUS
            selected_bus = bus_value
            ### FIND ASSOCIATED EFFICIENCY
            efficiency_col = bus_to_efficiency.get(bus_col)
            if (
                efficiency_col is not None
                and efficiency_col in links_df.columns
            ):
                selected_efficiency = link_row[efficiency_col]
            break
        ### KEEP ONLY LINKS WITH A POWER-SECTOR BUS
        if selected_bus is not None:
            selected_indices.append(link_name)
            selected_buses.append(selected_bus)
            selected_efficiencies.append(selected_efficiency)
    ### BUILD FILTERED DATAFRAME
    filtered_links_df = links_df.loc[
        selected_indices, columns_to_keep
    ].copy()
    filtered_links_df["bus"] = selected_buses
    filtered_links_df["efficiency"] = selected_efficiencies
    ### REORDER COLUMNS
    filtered_links_df = filtered_links_df[
        columns_to_keep + ["bus", "efficiency"]
    ]
    ### REPLACE ORIGINAL DATAFRAME IN THE NESTED DICTIONARY
    power_units_links_pypsa_dictionary[zone] = filtered_links_df
### DISPLAY RESULTS
print("\n" + "=" * 90)
print("ORIGINAL PYPSA LINKS DICTIONARY AFTER POWER-SECTOR FILTERING")
print("=" * 90)
for zone in zone_names:
    df = power_units_links_pypsa_dictionary[zone]
    print(
        f"{zone:4s} | "
        f"Rows: {df.shape[0]:4d} | "
        f"Columns: {df.shape[1]:2d}"
    )
print("=" * 90)


ORIGINAL PYPSA LINKS DICTIONARY AFTER POWER-SECTOR FILTERING
BE   | Rows:   25 | Columns: 12
FR   | Rows:   35 | Columns: 12
DE   | Rows:   51 | Columns: 12
NL   | Rows:   24 | Columns: 12
UK   | Rows:   53 | Columns: 12


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       D-06. PyPSA to Dispa-SET DataFrame Transformation
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process transforms the filtered PyPSA generator, link, and storage unit DataFrames into the standardized <b>Dispa-SET format</b>.  <br>A template CSV provides the column structure, while the mapping dictionary defines how PyPSA attributes are translated to Dispa-SET parameters.
    </div>
</div>

In [21]:
### CREATE DISPA-SET DATAFRAMES FROM PYPSA DATAFRAMES
# _____________________________________________________________________________
### PATH TO THE REPRESENTATIVE DISPA-SET CSV
dispaset_example_csv_path = vpp_power_plants_base_data_folder_path
### READ DISPA-SET CSV HEADER
dispaset_template_df = pd.read_csv(
    dispaset_example_csv_path,
    nrows=0
)
dispaset_columns = dispaset_template_df.columns.tolist()
### SOURCE PYPSA NESTED DICTIONARIES
pypsa_source_dictionaries = [
    power_units_generators_pypsa_dictionary,
    power_units_links_pypsa_dictionary,
    power_units_storage_units_pypsa_dictionary
]
### CREATE NEW NESTED DICTIONARY
power_units_dispaset_dictionary = {}
### PROCESS EACH ZONE
for zone in zone_names:
    zone_dataframes = []
    ### PROCESS GENERATORS, LINKS AND STORAGE UNITS
    for source_dictionary in pypsa_source_dictionaries:
        if zone not in source_dictionary:
            continue
        pypsa_df = source_dictionary[zone]
        if pypsa_df.empty:
            continue
        ### CREATE EMPTY DATAFRAME USING DISPA-SET TEMPLATE COLUMNS
        dispaset_df = pd.DataFrame(
            index=pypsa_df.index,
            columns=dispaset_columns
        )
        ### MAP PYPSA COLUMNS TO DISPA-SET COLUMNS
        for dispaset_column, pypsa_columns in (
            pypsa_dispaset_column_mapping.items()
        ):
            ### Ignore mappings for columns not present in the template
            if dispaset_column not in dispaset_columns:
                continue
            ### Find the first available PyPSA column
            selected_pypsa_column = None
            for pypsa_column in pypsa_columns:
                if pypsa_column in pypsa_df.columns:
                    selected_pypsa_column = pypsa_column
                    break
            ### Copy data when a corresponding PyPSA column exists
            if selected_pypsa_column is not None:
                dispaset_df[dispaset_column] = (
                    pypsa_df[selected_pypsa_column].values
                )
        ### STORE TRANSFORMED DATAFRAME
        zone_dataframes.append(dispaset_df)
    ### COMBINE GENERATORS + LINKS + STORAGE UNITS
    if zone_dataframes:
        power_units_dispaset_dictionary[zone] = pd.concat(
            zone_dataframes,
            axis=0,
            ignore_index=True
        )
    else:
        power_units_dispaset_dictionary[zone] = pd.DataFrame(
            columns=dispaset_columns
        )
### DISPLAY RESULTS
print("\n" + "=" * 100)
print("PYPSA → DISPA-SET TRANSFORMATION COMPLETED")
print("=" * 100)
for zone in zone_names:
    df = power_units_dispaset_dictionary[zone]
    print(
        f"{zone} | "
        f"Rows: {df.shape[0]} | "
        f"Columns: {df.shape[1]}"
    )
print("=" * 100)


PYPSA → DISPA-SET TRANSFORMATION COMPLETED
BE | Rows: 43 | Columns: 40
FR | Rows: 53 | Columns: 40
DE | Rows: 71 | Columns: 40
NL | Rows: 41 | Columns: 40
UK | Rows: 86 | Columns: 40


/tmp/ipykernel_2803280/3615253091.py:56: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  power_units_dispaset_dictionary[zone] = pd.concat(
/tmp/ipykernel_2803280/3615253091.py:56: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  power_units_dispaset_dictionary[zone] = pd.concat(
/tmp/ipykernel_2803280/3615253091.py:56: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determ

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       D-07. STOCapacity Calculation for Storage Technologies
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        For storage technologies, the <b>STOCapacity</b> (energy capacity in MWh) is calculated by multiplying <b>PowerCapacity</b> (MW) by the storage duration in hours.<br> At this stage, the raw <b>max_hours</b> value from PyPSA is temporarily stored in the STOCapacity column during the initial transformation.
    </div>
    <div style="text-align: center;font-size: 14px;">
        <b>STOCapacity [MWh] = PowerCapacity [MW] × max_hours [h]</b>
    </div>
</div>

In [22]:
### CALCULATE DISPA-SET STOCapacity FROM PYPSA max_hours
# _____________________________________________________________________________
### Storage technologies
storage_technologies = [
    "PHS",
    "hydro"
]
### PROCESS EACH ZONE
for zone in zone_names:
    df = power_units_dispaset_dictionary[zone]
    ### IDENTIFY STORAGE UNITS
    storage_mask = df["Technology"].isin(
        storage_technologies
    )
    ### CALCULATE STOCapacity
    ### STOCapacity [MWh] =
    ### PowerCapacity [MW] × max_hours [h]
    ### At this stage, max_hours was temporarily stored in STOCapacity
    ### during the PyPSA → Dispa-SET transformation.
    df.loc[storage_mask, "STOCapacity"] = (
        df.loc[storage_mask, "PowerCapacity"]
        *
        df.loc[storage_mask, "STOCapacity"]
    )
### DISPLAY RESULTS
print("\n" + "=" * 100)
print("STOCapacity CALCULATION COMPLETED")
print("=" * 100)
for zone in zone_names:
    df = power_units_dispaset_dictionary[zone]
    storage_df = df[
        df["Technology"].isin(storage_technologies)
    ][
        [
            "Unit",
            "Technology",
            "PowerCapacity",
            "STOCapacity"
        ]
    ]
    print(f"\n{zone}")
    if storage_df.empty:
        print("  No storage units found.")
    else:
        print(storage_df.to_string(index=False))
print("\n" + "=" * 100)


STOCapacity CALCULATION COMPLETED

BE
       Unit Technology  PowerCapacity  STOCapacity
  BE1 0 PHS        PHS    1308.000000       5710.0
BE1 0 hydro      hydro      12.735669     200000.0

FR
       Unit Technology  PowerCapacity  STOCapacity
  FR1 0 PHS        PHS    5236.300000 8.411215e+04
FR1 0 hydro      hydro    8573.492994 9.800000e+06

DE
       Unit Technology  PowerCapacity   STOCapacity
  DE1 0 PHS        PHS        7526.22  42151.314312
DE1 0 hydro      hydro         289.50 300000.000000

NL
  No storage units found.

UK
       Unit Technology  PowerCapacity  STOCapacity
  GB0 0 PHS        PHS          440.0       7000.0
GB0 0 hydro      hydro          221.5     200000.0



<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       D-08. Converting PyPSA Bus Names to Dispa-SET Zone Names
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        PyPSA bus names contain zone identifiers followed by bus numbers and optional suffixes (e.g., <b>"BE1 0 low voltage"</b>). <br>
        This process extracts and converts these bus names to the standardized <b>Dispa-SET zone codes</b> (e.g., "BE", "FR", "DE") used throughout the model.
    </div>
</div>

In [23]:
### 1. Map target zones to their valid prefixes (Target Name + Acronym)
zone_prefixes_dict = {}
for code in zone_names:
    prefixes = [code]
    if code in raw_countries:
        acronym = raw_countries[code][1].strip()
        if acronym:
            prefixes.append(acronym)
    zone_prefixes_dict[code] = tuple(prefixes)
### 2. Perform Zone Conversion
for zone in zone_names:
    df = power_units_dispaset_dictionary[zone]
    valid_prefixes = zone_prefixes_dict[zone]  # e.g., ('UK', 'GB') for UK
    ### Match rows that start with either the zone name or its acronym
    mask = (
        df["Zone"]
        .fillna("")
        .astype(str)
        .str.startswith(valid_prefixes)
    )
    df.loc[mask, "Zone"] = zone
### 3. Display Results
print("\n" + "=" * 100)
print("ZONE CONVERSION COMPLETED")
print("=" * 100)
for zone in zone_names:
    df = power_units_dispaset_dictionary[zone]
    print(f"\n{zone}")
    print(df["Zone"].value_counts(dropna=False).to_string())
print("\n" + "=" * 100)


ZONE CONVERSION COMPLETED

BE
Zone
BE    43

FR
Zone
FR    53

DE
Zone
DE    71

NL
Zone
NL    41

UK
Zone
UK    86



<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       D-09. Mapping PyPSA Technologies to Dispa-SET Attributes
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This function parses PyPSA technology keys (carrier names) and maps them to the corresponding <b>Dispa-SET Technology and Fuel codes</b>. It uses an equivalence dictionary to translate raw PyPSA carrier strings into standardized Dispa-SET attributes.
        <br>
        <span style="font-weight: bold;">Mapping Logic:</span>
    <div style="margin-left: 2em; font-size: 12px">
        For each zone and row, the function extracts the raw technology string (from the <b>Technology</b> column, falling back to <b>Unit</b> if empty).<br> It then searches for the longest matching key in the equivalence dictionary.<br> When a match is found, it populates the <b>Technology</b> and <b>Fuel</b> columns with comma-separated lists of Dispa-SET codes.
    </div>
</div>

In [24]:
def map_dispa_set_attributes_dict(units_dict, equivalences_dict):
    """
    Iterates through a dictionary of country DataFrames, parses PyPSA technology keys,
    populates 'Technology' and 'Fuel' columns with comma-separated Dispa-SET codes,
    and prints a summary message at the end.
    """
    total_processed_rows = 0
    total_mapped_rows = 0
    for zone, df in units_dict.items():
        ### Ensure Technology and Fuel columns exist
        if 'Technology' not in df.columns:
            df['Technology'] = None
        if 'Fuel' not in df.columns:
            df['Fuel'] = None
        for index, row in df.iterrows():
            total_processed_rows += 1
            ### Extract string from Technology or Unit column
            raw_tech = str(row['Technology']).strip() if pd.notna(row['Technology']) else str(row['Unit'])
            ### Match against the longest key in equivalences_dict
            matched_key = None
            for key in sorted(equivalences_dict.keys(), key=len, reverse=True):
                if key in raw_tech:
                    matched_key = key
                    break
            ### Join all equivalent tech/fuel codes with comma separation
            if matched_key:
                tech_list = equivalences_dict[matched_key]['tech']
                fuel_list = equivalences_dict[matched_key]['fuel']
                
                df.at[index, 'Technology'] = ", ".join(tech_list)
                df.at[index, 'Fuel'] = ", ".join(fuel_list)
                total_mapped_rows += 1
    ### End notification message
    print(
        f"Mapping complete!\n"
        f"• Zones processed: {len(units_dict)}\n"
        f"• Total units mapped: {total_mapped_rows} out of {total_processed_rows} rows."
    )
    return units_dict
### Apply to your dictionary of DataFrames
power_units_dispaset_dictionary = map_dispa_set_attributes_dict(
    power_units_dispaset_dictionary, 
    tech_equivalences_dict
)

Mapping complete!
• Zones processed: 5
• Total units mapped: 294 out of 294 rows.


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       D-10. Resolving Multi-Technology Units to Single Dispa-SET Codes
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        After the initial mapping, some rows may contain <b>multiple technology codes</b> (e.g., "STUR, COMC, GTUR"). This function resolves these ambiguities by selecting a single technology based on the unit's <b>PowerCapacity</b> and predefined business rules, with an updated rule for coal technologies.<br>
<b>Resolution Logic:</b>
    <div style="margin-left: 2em; font-size: 12px">
        The function checks if a row contains multiple technology codes (comma-separated). It then applies a series of rules to select the appropriate technology based on capacity thresholds:
        <br>
        <b>Oil Technologies</b> (STUR, GTUR, ICEN): STUR if ≥100 MW, GTUR if ≥20 MW, else ICEN<br>
        <b>Coal Technologies</b> (STUR, COMC, GTUR): COMC if ≥500 MW (IGCC), STUR if ≥100 MW (pulverized coal), else GTUR (syngas)<br>
        <b>H2 Fuel Cell</b> (GTUR, ICEN): GTUR if ≥50 MW, else ICEN<br>
        <b>Urban Central Gas CHP</b> (STUR, ICEN): STUR if ≥50 MW, else ICEN
    </div>
</div>

In [25]:
def resolve_technology(row: pd.Series) -> str:
    """
    Selects a single technology string based strictly on PowerCapacity
    when a row contains multiple candidate technologies.
    """
    tech = str(row['Technology'])
    ### Return immediately if it is already a single technology code
    if ',' not in tech:
        return tech
    options = [t.strip() for t in tech.split(',')]
    capacity = float(row.get('PowerCapacity', 0))
    ### 1. Oil technologies ("STUR", "GTUR", "ICEN")
    if set(options) == {"STUR", "GTUR", "ICEN"}:
        if capacity >= 100:
            return "STUR"
        elif capacity >= 20:
            return "GTUR"
        else:
            return "ICEN"
    ### 2. Coal technologies ("STUR", "COMC", "GTUR")
    if set(options) == {"STUR", "COMC", "GTUR"}:
        if capacity >= 500:
            return "COMC"  # Large modern IGCC combined cycle plant
        elif capacity >= 100:
            return "STUR"  # Standard large pulverized coal steam turbine
        else:
            return "GTUR"  # Small/niche syngas gas turbine unit
    ### 3. H2 Fuel Cell ("GTUR", "ICEN")
    if set(options) == {"GTUR", "ICEN"}:
        return "GTUR" if capacity >= 50 else "ICEN"
    ### 4. Urban Central Gas CHP ("STUR", "ICEN")
    if set(options) == {"STUR", "ICEN"}:
        return "STUR" if capacity >= 50 else "ICEN"
    ### Fallback to the first technology if no rule matches
    return options[0]
# _____________________________________________________________________________
def clean_power_units_dictionary(power_dict: dict) -> dict:
    """
    Applies the single-technology selection to each DataFrame in the dictionary
    without modifying Efficiency or any other columns.
    """
    cleaned_dict = {}
    total_resolved = 0
    for country, df in power_dict.items():
        df_copy = df.copy()
        ### Count how many multi-technology entries exist before cleaning
        multi_tech_mask = df_copy['Technology'].astype(str).str.contains(',')
        resolved_count = multi_tech_mask.sum()
        total_resolved += resolved_count
        ### Resolve technology strings
        df_copy['Technology'] = df_copy.apply(resolve_technology, axis=1)
        cleaned_dict[country] = df_copy
        print(f"[{country}] Resolved {resolved_count} multi-technology units to single Dispa-SET codes.")
    print("\n" + "=" * 60)
    print(f"SUCCESS: Cleaning complete! Processed {len(cleaned_dict)} countries and resolved a total of {total_resolved} technology definitions.")
    print("Efficiency values and all other unit parameters were left strictly untouched.")
    print("=" * 60)
    return cleaned_dict
### Run the update
power_units_dispaset_dictionary = clean_power_units_dictionary(power_units_dispaset_dictionary)

[BE] Resolved 3 multi-technology units to single Dispa-SET codes.
[FR] Resolved 4 multi-technology units to single Dispa-SET codes.
[DE] Resolved 16 multi-technology units to single Dispa-SET codes.
[NL] Resolved 4 multi-technology units to single Dispa-SET codes.
[UK] Resolved 6 multi-technology units to single Dispa-SET codes.

SUCCESS: Cleaning complete! Processed 5 countries and resolved a total of 33 technology definitions.
Efficiency values and all other unit parameters were left strictly untouched.


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       D-11. Splitting Multi-Fuel Units in Dispa-SET Dictionary
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        Some Dispa-SET units have multiple fuels (e.g., <b>"Gas, Oil"</b> or <b>"Coal, Biomass"</b>). <br>
        This process splits such multi-fuel units into separate single-fuel units, assigning capacity proportions based on the <b>fuel–technology mapping</b> data, while preserving all other attributes like Efficiency.<br>
<b>Splitting Logic:</b>
    <div style="margin-left: 2em; font-size: 12px">
        For each zone, the script retrieves the country-specific fuel mapping (or falls back to DEFAULT). It then iterates through all units, identifies those with multiple fuels (comma or space-separated), and splits them using the appropriate technology multipliers from the mapping.<br> Each new unit is named with a suffix (<b>_1</b>, <b>_2</b>, etc.), receives its proportional capacity, and is assigned a single fuel type while copying all other attributes (Efficiency, etc.).
    </div>
</div>

In [26]:
### Split Multi-Fuel Power Plant Units for power_units_dispaset_dictionary
for zone, df in power_units_dispaset_dictionary.items():
    ### 1. Select Country Mapping
    country_fuel_df = fuel_technologies_match_dict.get(zone)
    if country_fuel_df is None:
        print(f"⚠️ No country-specific fuel mapping found for '{zone}'. Using DEFAULT mapping.")
        country_fuel_df = overall_fuel_technologies_match_df
    new_rows = []
    rows_to_drop = []
    ### 2. Iterate Through Units
    for idx, row in df.iterrows():
        tech = str(row["Technology"]).strip()
        fuel_str = str(row["Fuel"]).strip()
        ### Handle both comma-separated and space-separated fuels
        fuels = [f.strip() for f in fuel_str.replace(",", " ").split()]
        ### Skip single-fuel units
        if len(fuels) <= 1:
            continue
        ### Check if technology is mapped
        if tech not in country_fuel_df.index and tech not in overall_fuel_technologies_match_df.index:
            print(f"⚠️ Technology '{tech}' not found in mappings. Skipping unit '{row['Unit']}'.")
            rows_to_drop.append(idx)
            continue
        original_capacity = row["PowerCapacity"]
        new_capacities = []
        new_rows_for_this_row = []
        ### 3. Split Unit by Fuel
        for i, fuel in enumerate(fuels, start=1):
            multiplier = None
            ### Look up in country-specific mapping first
            if tech in country_fuel_df.index and fuel in country_fuel_df.columns:
                multiplier = country_fuel_df.loc[tech, fuel]
            ### Fallback to DEFAULT mapping
            if pd.isna(multiplier):
                if tech in overall_fuel_technologies_match_df.index and fuel in overall_fuel_technologies_match_df.columns:
                    multiplier = overall_fuel_technologies_match_df.loc[tech, fuel]
            ### Skip if no valid multiplier exists
            if pd.isna(multiplier):
                print(f"⚠️ No valid multiplier for Tech='{tech}', Fuel='{fuel}', Zone='{zone}'. Skipping fuel.")
                continue
            ### Create fuel-specific unit (all other attributes like Efficiency are strictly copied)
            new_row = row.copy()
            new_row["Unit"] = f"{row['Unit']}_{i}"
            new_row["Fuel"] = fuel
            new_row["PowerCapacity"] = original_capacity * multiplier
            new_rows_for_this_row.append(new_row)
            new_capacities.append(new_row["PowerCapacity"])
        ### 4. Capacity Balancing
        if not new_rows_for_this_row:
            rows_to_drop.append(idx)
            continue
        sum_new_capacities = sum(new_capacities)
        if sum_new_capacities < original_capacity:
            max_cap_idx = max(range(len(new_capacities)), key=lambda k: new_capacities[k])
            new_rows_for_this_row[max_cap_idx]["PowerCapacity"] += (original_capacity - sum_new_capacities)
        new_rows.extend(new_rows_for_this_row)
        rows_to_drop.append(idx)
    ### 5. Apply Changes to DataFrame in-place inside the dictionary
    df_cleaned = df.drop(rows_to_drop, errors="ignore")
    if new_rows:
        df_cleaned = pd.concat([df_cleaned, pd.DataFrame(new_rows)], ignore_index=True)
    power_units_dispaset_dictionary[zone] = df_cleaned
    print(f"[{zone}] Split completed: {len(rows_to_drop)} multi-fuel units converted.")
print("\n" + "=" * 60)
print("✅ Success! 'power_units_dispaset_dictionary' updated in-place.")
print("=" * 60)

[BE] Split completed: 0 multi-fuel units converted.
[FR] Split completed: 0 multi-fuel units converted.
[DE] Split completed: 8 multi-fuel units converted.
[NL] Split completed: 1 multi-fuel units converted.
[UK] Split completed: 0 multi-fuel units converted.

✅ Success! 'power_units_dispaset_dictionary' updated in-place.


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       D-12. Removing Zero-Capacity Units
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        After the multi-fuel splitting and capacity calculations, some units may have <b>zero PowerCapacity</b>.
        <br>These units are not meaningful for the optimization and are removed from the dictionary.
    </div>
</div>

In [27]:
### Remove zero-capacity units from power_units_dispaset_dictionary
for country in list(power_units_dispaset_dictionary.keys()):
    df = power_units_dispaset_dictionary[country]
    power_units_dispaset_dictionary[country] = df[df['PowerCapacity'] != 0].reset_index(drop=True)
print("✅ Removed zero-capacity units from 'power_units_dispaset_dictionary'.")

✅ Removed zero-capacity units from 'power_units_dispaset_dictionary'.


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       D-13. Setting Nunits Column to 1
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        The <b>Nunits</b> column in Dispa-SET represents the number of identical units aggregated in each row.<br> Since the PyPSA data has already been processed into individual unit rows, this column is set to <b>1</b> for all units.
    </div>
</div>

In [28]:
### FILL Nunits WITH 1
for zone in zone_names:
    df = power_units_dispaset_dictionary[zone]
    df["Nunits"] = 1
### DISPLAY RESULTS
print("\n" + "=" * 100)
print("Nunits COLUMN FILLED WITH 1")
print("=" * 100)
for zone in zone_names:
    df = power_units_dispaset_dictionary[zone]
    print(
        f"{zone} | "
        f"Rows: {len(df)} | "
        f"Unique Nunits values: {df['Nunits'].unique().tolist()}"
    )
print("=" * 100)


Nunits COLUMN FILLED WITH 1
BE | Rows: 41 | Unique Nunits values: [1]
FR | Rows: 51 | Unique Nunits values: [1]
DE | Rows: 69 | Unique Nunits values: [1]
NL | Rows: 39 | Unique Nunits values: [1]
UK | Rows: 82 | Unique Nunits values: [1]


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
      D-14.  Sanitizing Negative Values in Dispa-SET Parameters
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        Some Dispa-SET parameters may contain <b>negative values</b> due to data inconsistencies or missing placeholders. <br>These values are invalid for the optimization and are replaced with <b>0</b> to ensure model stability.
    </div>
</div>

In [29]:
### Target columns to check and sanitize
features_to_update = [
    'MinUpTime', 'MinDownTime', 'RampUpRate', 'RampDownRate',
    'StartUpCost', 'NoLoadCost_pu', 'RampingCost', 'PartLoadMin',
    'MinEfficiency', 'StartUpTime', 'CO2Intensity'
]
total_negative_replacements = 0
### Create a cleaned dictionary
power_units_dispaset_sanitized = {}
for country, df in power_units_dispaset_dictionary.items():
    df_clean = df.copy()
    country_replacements = 0
    for feat in features_to_update:
        if feat in df_clean.columns:
            ### Coerce feature column to numeric in case strings/placeholders exist
            numeric_col = pd.to_numeric(df_clean[feat], errors='coerce')
            ### Find mask of negative values
            neg_mask = numeric_col < 0
            neg_count = neg_mask.sum()
            if neg_count > 0:
                country_replacements += neg_count
                ### Optional print statement to inspect replaced negative values
                neg_techs = df_clean.loc[neg_mask, 'Technology'].unique()
                print(f"[{country}] Replaced {neg_count} negative value(s) in '{feat}' for tech: {list(neg_techs)}")
                ### Replace negative values with 0
                df_clean[feat] = np.where(neg_mask, 0, numeric_col)
    total_negative_replacements += country_replacements
    power_units_dispaset_dictionary[country] = df_clean
print(
        f"[{country}] Completed imputation. Total values updated: {country_replacements}"
    )
print("\n--- Imputation Cleanup Summary ---")
print(
    f"Total missing/zero values updated across all countries: {total_negative_replacements}"
)

[BE] Replaced 1 negative value(s) in 'PartLoadMin' for tech: ['HPHS']
[FR] Replaced 1 negative value(s) in 'PartLoadMin' for tech: ['HPHS']
[DE] Replaced 1 negative value(s) in 'PartLoadMin' for tech: ['HPHS']
[UK] Replaced 1 negative value(s) in 'PartLoadMin' for tech: ['HPHS']
[UK] Completed imputation. Total values updated: 1

--- Imputation Cleanup Summary ---
Total missing/zero values updated across all countries: 4


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       D-15. Techno-Economic Features Dictionary File Paths
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        The <b>techno-economic features dictionaries</b> are CSV files that define the default technical and economic parameters for each technology type. <br>These files serve as the baseline reference when country-specific data is not available.
    </div>
</div>

In [30]:
technoeconomic_features_dictionary_1_file_path = (
    Path(dispaSET_unleash_folder_path)
    / "scripts"
    / "Unleash_PyPSA_DispaSET_Raw_Data_Processing"
    / "Coefficients_Sources"
    / "TechnoEconomic_Features_Dictionary_1.csv"
)
# _____________________________________________________________________________
technoeconomic_features_dictionary_2_file_path = (
    Path(dispaSET_unleash_folder_path)
    / "scripts"
    / "Unleash_PyPSA_DispaSET_Raw_Data_Processing"
    / "Coefficients_Sources"
    / "TechnoEconomic_Features_Dictionary_2.csv"
)

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
      D-16.  Updating Techno-Economic Features from Reference Dictionaries
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process fills missing or zero values in the power unit DataFrames using two <b>techno-economic reference dictionaries</b>. These dictionaries contain default technical and economic parameters for each technology, organized by <b>Fuel + Technology</b> and <b>PowerCapacity</b> ranges.
        <br><b>Update Logic:</b> 
            <div style="margin-left: 2em; font-size: 12px">
        For each unit, the script searches for the best matching reference row using:
            <div style="margin-left: 2em;">
        <b>1.</b> Match on <b>Fuel + Technology</b> (most specific)<br>
        <b>2.</b> If no match, fall back to <b>Technology only</b><br>
        <b>3.</b> Find the row with the <b>nearest PowerCapacity</b> value<br>
        <b>4.</b> Update columns only if the current value is <b>NaN</b> or <b>0</b> (existing non-zero values are preserved)
        </div>
        Two reference tables are applied sequentially: <b>Dictionary 1</b> first, then <b>Dictionary 2</b> for any remaining missing values.
    <br>
        This ensures that every power unit has a complete set of techno-economic parameters (e.g., <b>MinUpTime</b>, <b>RampUpRate</b>, <b>StartUpCost</b>) for the Dispa-SET optimization.
            </div>
    </div>
</div>

In [31]:
### COLUMNS TO UPDATE
columns_to_update = [
    'MinUpTime',
    'MinDownTime',
    'RampUpRate',
    'RampDownRate',
    'StartUpCost',
    'NoLoadCost_pu',
    'RampingCost',
    'PartLoadMin',
    'MinEfficiency',
    'StartUpTime',
    'CO2Intensity'
]
### LOAD THE TWO TECHNO-ECONOMIC REFERENCE TABLES
technoeconomic_df_1 = pd.read_csv(
    technoeconomic_features_dictionary_1_file_path
)
technoeconomic_df_2 = pd.read_csv(
    technoeconomic_features_dictionary_2_file_path
)
### PREPARE REFERENCE TABLES
for reference_df in [
    technoeconomic_df_1,
    technoeconomic_df_2
]:
    reference_df['PowerCapacity'] = pd.to_numeric(
        reference_df['PowerCapacity'],
        errors='coerce'
    )
    for column in columns_to_update:
        reference_df[column] = pd.to_numeric(
            reference_df[column],
            errors='coerce'
        )
### FUNCTION: FIND THE BEST MATCH
def find_best_reference_row(unit_row, reference_df):
    fuel = unit_row['Fuel']
    technology = unit_row['Technology']
    power_capacity = pd.to_numeric(
        unit_row['PowerCapacity'],
        errors='coerce'
    )
    ### 1. MATCH Fuel + Technology
    candidates = reference_df[
        (reference_df['Fuel'] == fuel) &
        (reference_df['Technology'] == technology)
    ].copy()
    ### 2. IF NO MATCH, FALL BACK TO Technology ONLY
    if candidates.empty:
        candidates = reference_df[
            reference_df['Technology'] == technology
        ].copy()
    ### 3. IF STILL NO MATCH, RETURN NONE
    if candidates.empty:
        return None
    ### 4. ONLY CONSIDER REFERENCE ROWS WITH PowerCapacity
    candidates = candidates.dropna(
        subset=['PowerCapacity']
    )
    if candidates.empty:
        return None
    if pd.isna(power_capacity):
        return None
    ### 5. FIND THE NEAREST PowerCapacity
    candidates['PowerCapacityDifference'] = (
        candidates['PowerCapacity'] - power_capacity
    ).abs()
    best_index = candidates[
        'PowerCapacityDifference'
    ].idxmin()
    return candidates.loc[best_index]
### FUNCTION: UPDATE FROM ONE REFERENCE TABLE
def update_from_reference_table(units_df, reference_df):
    for index in units_df.index:
        unit_row = units_df.loc[index]
        best_reference_row = find_best_reference_row(
            unit_row,
            reference_df
        )
        if best_reference_row is None:
            continue
        ### UPDATE ONLY NaN OR 0
        for column in columns_to_update:
            current_value = units_df.at[index, column]
            reference_value = best_reference_row[column]
            ### Do not copy NaN from reference table
            if pd.isna(reference_value):
                continue
            ### Fill empty field
            if pd.isna(current_value):
                units_df.at[index, column] = reference_value
            ### Replace zero
            elif current_value == 0:
                units_df.at[index, column] = reference_value
            ### Existing non-zero value → KEEP IT
    return units_df
### FIRST LOOKUP: TECHNO-ECONOMIC DICTIONARY 1
for country, units_df in power_units_dispaset_dictionary.items():
    print(f'\nProcessing {country} - Dictionary 1')
    power_units_dispaset_dictionary[country] = (
        update_from_reference_table(
            units_df,
            technoeconomic_df_1
        )
    )
### SECOND LOOKUP: TECHNO-ECONOMIC DICTIONARY 2
for country, units_df in power_units_dispaset_dictionary.items():
    print(f'Processing {country} - Dictionary 2')
    power_units_dispaset_dictionary[country] = (
        update_from_reference_table(
            units_df,
            technoeconomic_df_2
        )
    )
print('\nAll countries processed successfully.')


Processing BE - Dictionary 1

Processing FR - Dictionary 1

Processing DE - Dictionary 1

Processing NL - Dictionary 1

Processing UK - Dictionary 1
Processing BE - Dictionary 2


/tmp/ipykernel_2803280/3177621735.py:95: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3.125' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  units_df.at[index, column] = reference_value
/tmp/ipykernel_2803280/3177621735.py:95: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3.25' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  units_df.at[index, column] = reference_value


Processing FR - Dictionary 2
Processing DE - Dictionary 2
Processing NL - Dictionary 2
Processing UK - Dictionary 2

All countries processed successfully.


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        D-17. Capping Efficiency Values at 1 in Power Units Dispa-SET Dictionary
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process identifies and caps any <b>Efficiency</b> values greater than <b>1</b> in the <b>power_units_dispaset_dictionary</b>.<br> Values above 1 are physically unrealistic for most conversion technologies and are capped at 1 to ensure model consistency.<br>
    <b>Capping Logic:</b>
            <div style="margin-left: 2em; font-size: 12px">
        For each zone in the dictionary, the script checks if the <b>Efficiency</b> column exists. <br>It identifies rows where Efficiency > 1, prints them for inspection, and then caps those values at <b>1</b> using <b>df.loc[df['Efficiency'] > 1, 'Efficiency'] = 1</b>. <br>The updated DataFrame is stored back into the dictionary.<br>
        This ensures that all efficiency values remain within the physically valid range (0 to 1), preventing unrealistic fuel consumption or generation calculations in the Dispa-SET optimization.
            </div>
    </div>
</div>

In [32]:
for key, df in power_units_dispaset_dictionary.items():
    if 'Efficiency' in df.columns:
        # Find rows where Efficiency > 1
        rows_to_cap = df[df['Efficiency'] > 1]
        if not rows_to_cap.empty:
            print(f"Key: {key}")
            print(f"Rows with Efficiency > 1 (before capping):")
            print(rows_to_cap[['Unit', 'Efficiency']])
            print("\n")

            # Cap the values at 1
            df.loc[df['Efficiency'] > 1, 'Efficiency'] = 1
            power_units_dispaset_dictionary[key] = df  # Update the dictionary
        else:
            print(f"Key: {key} - No rows with Efficiency > 1 found.")

Key: BE
Rows with Efficiency > 1 (before capping):
                Unit  Efficiency
2  BE1 0 onwind-2030         1.0


Key: FR - No rows with Efficiency > 1 found.
Key: DE - No rows with Efficiency > 1 found.
Key: NL
Rows with Efficiency > 1 (before capping):
                    Unit  Efficiency
0  NL1 0 offwind-ac-2030         1.0


Key: UK - No rows with Efficiency > 1 found.


<div style= "background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099; " >
 <span style= "font-weight: bold; font-size: 16px; " >
Section E Overview: VPP & Thermal Storage Processing
 </span >
 <div style= "border-top: 1px solid #000099; padding: 10px; " >
This section focuses on sector-coupling and Virtual Power Plant (VPP) modeling. <br>It loads water-tank thermal storage data, allocates storage capacity to individual assets based on heat load fractions, updates resistive heater parameters, and aggregates distributed heaters first by category and then into a single consolidated VPP unit per zone to reduce optimization complexity.
 </div >
 </div >
 <div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       E-01. VPP Resistive Heater Parameter Updates
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        When <b>VPP mode</b> is enabled, this function identifies resistive heater units and updates their Dispa-SET parameters to correctly represent their <b>storage and charging characteristics</b>. This ensures that resistive heaters are properly modeled as flexible load units with thermal storage.<br>
    <b>Update Logic:</b>
            <div style="margin-left: 2em; font-size: 12px">
        For each country, the function searches for units whose <b>Unit</b> name contains any of the <b>vpp_resistive_heater_carriers</b> patterns (e.g., "residential rural resistive heater"). <br>For each matching unit, it sets:
            <div style="margin-left: 2em;">
        <b>STOMaxChargingPower</b> = PowerCapacity (maximum charge rate)<br>
        <b>EfficiencySector1</b> = Efficiency (discharge efficiency)<br>
        <b>ChargingEfficiencySector1</b> = Efficiency (charge efficiency)<br>
        <b>Sector1</b> = <b>"DHW_"</b> + acronym from Unit name
            </div>
        The acronym is generated by stripping the year suffix (e.g., "-2030"), removing the country/node prefix (e.g., "GB0 0"), and taking the first letter of each remaining word.
            </div>
    </div>
</div>

In [33]:
if VPP:
# _____________________________________________________________________________
    def extract_vpp_acronym(unit_name):
        ### 1. Strip year suffix if present (e.g., "-2030")
        base = str(unit_name).rsplit("-", 1)[0]
        ### 2. Strip country/node prefix (e.g., "GB0 0 ")
        parts = base.split()
        if len(parts) >= 2 and parts[1].isdigit():
            base = " ".join(parts[2:])
        ### 3. Take first letter of each word to form acronym
        words = base.split()
        acronym = "".join([w[0].upper() for w in words if w.isalnum()])
        return f"DHW_{acronym}"
    # _____________________________________________________________________________
    def update_vpp_resistive_heaters(
        df, vpp_resistive_heater_carriers, zone_name=""
    ):
        ### Create regex pattern to match any carrier in the list
        pattern = "|".join(vpp_resistive_heater_carriers)
        ### Identify matching rows where Unit contains any carrier string
        mask = df["Unit"].str.contains(pattern, case=False, na=False)
        updated_indices = df[mask].index
        num_updated = len(updated_indices)
        if num_updated > 0:
            ### Perform updates
            df.loc[mask, "STOMaxChargingPower"] = df.loc[mask, "PowerCapacity"]
            df.loc[mask, "EfficiencySector1"] = df.loc[mask, "Efficiency"]
            df.loc[mask, "ChargingEfficiencySector1"] = df.loc[
                mask, "Efficiency"
            ]
            ### Generate 'DHW_' + acronym from Unit name
            df.loc[mask, "Sector1"] = df.loc[mask, "Unit"].apply(
                extract_vpp_acronym
            )
            ### Print message showing which rows and values were updated
            print(f"\n[{zone_name}] Successfully updated {num_updated} row(s):")
            for idx in updated_indices:
                unit_name = df.at[idx, "Unit"]
                power_cap = df.at[idx, "PowerCapacity"]
                eff = df.at[idx, "Efficiency"]
                sec1 = df.at[idx, "Sector1"]
                print(f"  • Row {idx} | Unit: '{unit_name}'")
                print(
                    f"    └── STOMaxChargingPower={power_cap} | EfficiencySector1={eff} | ChargingEfficiencySector1={eff} | Sector1='{sec1}'"
                )
        else:
            print(f"\n[{zone_name}] No matching resistive heater rows found.")
        return df
    ### Execute across dictionary
    total_rows_updated = 0
    for country, units_df in power_units_dispaset_dictionary.items():
        power_units_dispaset_dictionary[country] = update_vpp_resistive_heaters(
            units_df, vpp_resistive_heater_carriers, zone_name=country
        )
    print("\n" + "=" * 80)
    print("Resistive heater VPP processing complete.")
    print("=" * 80)


[BE] Successfully updated 11 row(s):
  • Row 21 | Unit: 'BE1 0 residential rural resistive heater-2030'
    └── STOMaxChargingPower=0.042167827513539 | EfficiencySector1=0.9 | ChargingEfficiencySector1=0.9 | Sector1='DHW_RRRH'
  • Row 22 | Unit: 'BE1 0 services rural resistive heater-2030'
    └── STOMaxChargingPower=0.1861093147296933 | EfficiencySector1=0.9 | ChargingEfficiencySector1=0.9 | Sector1='DHW_SRRH'
  • Row 23 | Unit: 'BE1 0 residential urban decentral resistive heater-2030'
    └── STOMaxChargingPower=0.0516930416594733 | EfficiencySector1=0.9 | ChargingEfficiencySector1=0.9 | Sector1='DHW_RUDRH'
  • Row 24 | Unit: 'BE1 0 services urban decentral resistive heater-2030'
    └── STOMaxChargingPower=0.0766590051989181 | EfficiencySector1=0.9 | ChargingEfficiencySector1=0.9 | Sector1='DHW_SUDRH'
  • Row 25 | Unit: 'BE1 0 urban central resistive heater-2030'
    └── STOMaxChargingPower=2574.250216682336 | EfficiencySector1=0.99 | ChargingEfficiencySector1=0.99 | Sector1='DHW_U

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
      E-02. Loading Water-Tank Store Data
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process loads the <b>water-tank store</b> DataFrames for each country from the exported CSV files. <br>These files contain thermal storage parameters (e.g., <b>e_nom_opt</b>, <b>standing_loss</b>) that are essential for modeling district heating and thermal flexibility.<br>
    <b>Loading Logic:</b>
            <div style="margin-left: 2em; font-size: 12px">
        For each country, the script constructs the expected file path: <b>{base_path}/{year}/{country}/pypsa_water_tanks_{country}_{year}.csv</b>. <br>If the file exists, it is loaded with the first column set as the index. The DataFrame is then stored in a dictionary keyed by the country code.
    </div>
</div>

In [34]:
### CREATE WATER-TANK STORE DICTIONARY
water_tank_stores = {}
for country in zone_names:
    ### Construct filename
    file_name = f"pypsa_water_tanks_{country}_{data_target_year}.csv"
    ### Construct full path:
    ### power_plants_pypsa_raw_data_folder_path/
    ###     data_target_year/
    ###         country/
    ###             file_name
    file_path = os.path.join(
        power_plants_pypsa_raw_data_folder_path,
        str(data_target_year),
        country,
        file_name
    )
    ### Check that the file exists
    if not os.path.isfile(file_path):
        print(
            f"[Warning] Water-tank file not found:\n"
            f"  {file_path}"
        )
        continue
    ### Read CSV
    water_tank_df = pd.read_csv(
        file_path,
        index_col=0
    )
    ### Store complete DataFrame under country
    water_tank_stores[country] = water_tank_df
    print(
        f"{country}: "
        f"{len(water_tank_df)} water-tank stores loaded"
    )
### SUMMARY
print("\n==============================================================")
print("Water-tank store dictionary created")
print("==============================================================")
print(
    f"Countries loaded: {list(water_tank_stores.keys())}"
)
for country, df in water_tank_stores.items():
    print(f"\n{country}:")
    print(f"  Rows: {len(df)}")
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Nodes: {df['node'].nunique()}")
    print(f"  Heat loads: {df['heat_load'].nunique()}")
    print(
        df[
            [
                "bus",
                "e_nom_opt",
                "standing_loss",
                "country",
                "node",
                "heat_load"
            ]
        ].head()
    )

BE: 5 water-tank stores loaded
FR: 5 water-tank stores loaded
DE: 5 water-tank stores loaded
NL: 5 water-tank stores loaded
UK: 10 water-tank stores loaded

Water-tank store dictionary created
Countries loaded: ['BE', 'FR', 'DE', 'NL', 'UK']

BE:
  Rows: 5
  Columns: ['bus', 'type', 'carrier', 'e_nom', 'e_nom_mod', 'e_nom_extendable', 'e_nom_min', 'e_nom_max', 'e_nom_set', 'e_min_pu', 'e_max_pu', 'e_initial', 'e_initial_per_period', 'e_cyclic', 'e_cyclic_per_period', 'p_set', 'q_set', 'e_set', 'sign', 'marginal_cost', 'marginal_cost_quadratic', 'marginal_cost_storage', 'capital_cost', 'overnight_cost', 'discount_rate', 'fom_cost', 'standing_loss', 'active', 'build_year', 'lifetime', 'e_nom_opt', 'country', 'node', 'heat_load']
  Nodes: 1
  Heat loads: 5
                                                                                              bus  \
BE1 0 residential rural water tanks-2030                      BE1 0 residential rural water tanks   
BE1 0 services rural water tanks-2

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
      E-03. Loading Power-Plant DataFrames
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process loads the previously exported <b>power-plant DataFrames</b> for each country from CSV files.<br> These files contain the complete PyPSA power-plant data, including all units, technologies, capacities, and technical parameters.<br>
    <b>Loading Logic:</b>
            <div style="margin-left: 2em; font-size: 12px">
        For each country, the script constructs the expected file path: <b>{base_path}/{year}/{country}/pypsa_power_plants_{country}_{year}.csv</b>.<br> If the file exists, it is loaded with the first column set as the index. The DataFrame is then stored in a dictionary keyed by the country code.
            </div>
    </div>
</div>

In [35]:
### Define all possible heat load options to loop through
# =============================================================================
all_heat_loads = [
    'urban central heat'               ,
    'services urban decentral heat'    ,
    'services rural heat'              ,
    'residential rural heat'           ,
    'residential urban decentral heat'
]
### =============================================================================
### CREATE POWER-PLANT DATAFRAME DICTIONARY
### Structure:
### country_dataframes[country] = complete PyPSA power-plants DataFrame
### Example:
### country_dataframes["BE"]
### country_dataframes["FR"]
country_dataframes = {}
for country in zone_names:
    ### Construct filename
    file_name = (
        f"pypsa_power_plants_{country}_{data_target_year}.csv"
    )
    ### Construct full path
    ###
    ### power_plants_pypsa_raw_data_folder_path/
    ###     data_target_year/
    ###         country/
    ###             file_name
    file_path = os.path.join(
        power_plants_pypsa_raw_data_folder_path,
        str(data_target_year),
        country,
        file_name
    )
    ### Check that the file exists
    if not os.path.isfile(file_path):
        print(
            f"[Warning] Power-plant file not found:\n"
            f"  {file_path}"
        )
        continue
    ### Read CSV
    power_plants_df = pd.read_csv(
        file_path,
        index_col=0
    )
    ### Store complete DataFrame under country
    country_dataframes[country] = power_plants_df
    print(
        f"{country}: "
        f"{len(power_plants_df)} power-plant units loaded"
    )
### SUMMARY
print("\n==============================================================")
print("Power-plant country dictionary created")
print("==============================================================")
print(
    f"Countries loaded: {list(country_dataframes.keys())}"
)
for country, df in country_dataframes.items():
    print(f"\n{country}:")
    print(f"  Rows: {len(df)}")
    print(f"  Columns: {df.columns.tolist()}")
    if "node" in df.columns:
        print(f"  Nodes: {df['node'].nunique()}")
    if "carrier" in df.columns:
        print(f"  Carriers: {df['carrier'].nunique()}")
    print(df.head())

BE: 158 power-plant units loaded
FR: 174 power-plant units loaded
DE: 195 power-plant units loaded
NL: 159 power-plant units loaded
UK: 320 power-plant units loaded

Power-plant country dictionary created
Countries loaded: ['BE', 'FR', 'DE', 'NL', 'UK']

BE:
  Rows: 158
  Columns: ['bus', 'bus0', 'bus1', 'bus2', 'bus3', 'bus4', 'efficiency', 'efficiency2', 'efficiency3', 'efficiency4', 'carrier', 'start_up_cost', 'shut_down_cost', 'stand_by_cost', 'min_up_time', 'min_down_time', 'up_time_before', 'down_time_before', 'ramp_limit_up', 'ramp_limit_down', 'ramp_limit_start_up', 'ramp_limit_shut_down', 'p_nom_opt', 'urban central heat_fraction', 'services urban decentral heat_fraction', 'services rural heat_fraction', 'residential rural heat_fraction', 'residential urban decentral heat_fraction']
  Carriers: 91
                         bus bus0 bus1 bus2 bus3 bus4  efficiency  \
name                                                                
BE1 0 offwind-ac-2030  BE1 0  NaN  NaN  NaN 

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       E-04. Allocating Water-Tank Storage to Power-Plant Assets
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process <b>allocates water-tank storage capacity</b> to individual power-plant assets based on their heat load fractions.<br> It combines the water-tank store DataFrames with the power-plant DataFrames to create a comprehensive mapping of thermal storage per asset.<br>
    <b>Allocation Logic:</b>
            <div style="margin-left: 2em; font-size: 12px">
        For each country, the script identifies active assets with a positive heat load fraction. <br>It extracts the <b>node</b> from the asset name using regex, finds the corresponding water-tank store for that node and heat load, and allocates storage capacity proportionally:</div>
        <div style="text-align: center; font-size: 15px;">
        <b>allocated_e_nom_opt = node_e_nom_opt × fraction_value</b>.
    </div>
</div>

In [36]:
### CREATE LOOKUP MAP FOR COUNTRY ACRONYMS
code_to_canonical = {}
for canonical_key, info in selected_zone_names_equivalences_dict.items():
    code_to_canonical[canonical_key] = canonical_key
    for alt in info.get("Acronym", []):
        if alt.strip():
            code_to_canonical[alt.strip()] = canonical_key
### CREATE ALLOCATED COUNTRY STORES
### Input:
### water_tank_stores[country] = DataFrame
### country_dataframes[country] = power-plant DataFrame
### Output:
### allocated_country_stores[country] = DataFrame
### Columns:
### asset_name
### node
### heat_category
### fraction
### e_nom_opt
### standing_loss
allocated_country_stores = {}
### LOOP THROUGH COUNTRIES
for country in water_tank_stores:
    print(
        f"\n--- Allocating storage for Country: {country} ---"
    )
    ### 1. RESOLVE COUNTRY KEY
    target_df_key = (
        country
        if country in country_dataframes
        else code_to_canonical.get(country)
    )
    if not target_df_key or target_df_key not in country_dataframes:
        print(
            f"  [Warning] Country '{country}' metadata not found "
            f"in country_dataframes. Skipping."
        )
        continue
    ### 2. GET POWER-PLANT DATAFRAME
    master_df = country_dataframes[target_df_key]
    ### 3. GET WATER-TANK DATAFRAME
    water_tank_df = water_tank_stores[country]
    ### 4. INITIALIZE LIST OF ALLOCATED ROWS
    allocated_rows = []
    ### 5. LOOP THROUGH ALL HEAT LOADS
    for current_heat_load in all_heat_loads:
        fraction_col = f"{current_heat_load}_fraction"
        ### Check that the fraction column exists
        if fraction_col not in master_df.columns:
            print(
                f"  [Warning] Column '{fraction_col}' not found "
                f"for country {target_df_key}."
            )
            continue
        ### Select active power-plant assets
        active_assets = master_df[
            master_df[fraction_col] > 0
        ]
        ### 6. LOOP THROUGH ACTIVE POWER-PLANT ASSETS
        for asset_name, row in active_assets.iterrows():
            # Get heat-load fraction
            fraction_value = float(
                row[fraction_col]
            )
            # Extract sub-node from asset name
            # Example:
            ### BE1 0 urban central resistive heater-2030
            ### becomes:
            ### BE1 0
            node_match = re.match(
                r"^([A-Z]{2}\d*\s*\d*)",
                str(asset_name)
            )
            if node_match:
                asset_node = node_match.group(1).strip()
            else:
                asset_node = str(asset_name).split(" ")[0]
            ### 7. FIND CORRESPONDING WATER-TANK ROW
            matching_tanks = water_tank_df[
                (water_tank_df["node"] == asset_node)
                &
                (water_tank_df["heat_load"] == current_heat_load)
            ]
            ### 8. NO CORRESPONDING WATER TANK
            if matching_tanks.empty:
                continue
            ### 9. GET WATER-TANK PARAMETERS
            node_store_row = matching_tanks.iloc[0]
            node_e_nom_opt = float(
                node_store_row["e_nom_opt"]
            )
            standing_loss = float(
                node_store_row["standing_loss"]
            )
            ### 10. ALLOCATE STORAGE CAPACITY
            allocated_e_nom_opt = (
                node_e_nom_opt * fraction_value
            )
            ### 11. ADD ALLOCATED ASSET
            allocated_rows.append(
                {
                    "asset_name": asset_name,
                    "node": asset_node,
                    "heat_category": current_heat_load,
                    "fraction": fraction_value,
                    "e_nom_opt": allocated_e_nom_opt,
                    "standing_loss": standing_loss
                }
            )
    ### 12. CREATE COUNTRY DATAFRAME
    if allocated_rows:
        allocated_country_stores[target_df_key] = pd.DataFrame(
            allocated_rows
        )
        print(
            f"  ✅ Disaggregated storage across "
            f"{len(allocated_rows)} assets."
        )
    else:
        print(
            f"  [Warning] No assets were allocated for "
            f"country {target_df_key}."
        )
### FINAL SUMMARY
print("\n==============================================================")
print("Allocated country stores dictionary created")
print("==============================================================")
for country, df in allocated_country_stores.items():
    print(
        f"\nCountry: {country}"
    )
    print(
        f"  Number of assets: {len(df)}"
    )
    print(
        f"  Number of nodes: {df['node'].nunique()}"
    )
    print(
        f"  Columns: {df.columns.tolist()}"
    )
    print(
        "\nFirst 3 rows:"
    )
    print(
        df.head(3).to_string(index=False)
    )


--- Allocating storage for Country: BE ---
  ✅ Disaggregated storage across 21 assets.

--- Allocating storage for Country: FR ---
  ✅ Disaggregated storage across 31 assets.

--- Allocating storage for Country: DE ---
  ✅ Disaggregated storage across 23 assets.

--- Allocating storage for Country: NL ---
  ✅ Disaggregated storage across 25 assets.

--- Allocating storage for Country: UK ---
  ✅ Disaggregated storage across 48 assets.

Allocated country stores dictionary created

Country: BE
  Number of assets: 21
  Number of nodes: 1
  Columns: ['asset_name', 'node', 'heat_category', 'fraction', 'e_nom_opt', 'standing_loss']

First 3 rows:
                               asset_name  node      heat_category  fraction    e_nom_opt  standing_loss
                  BE1 0 H2 Fuel Cell-2030 BE1 0 urban central heat  0.000559   151.566112       0.000231
   BE1 0 urban central air heat pump-2030 BE1 0 urban central heat  0.000007     2.021415       0.000231
BE1 0 urban central resistive heate

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       E-05. Copying Storage Parameters to Resistive Heater Units
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        When <b>VPP mode</b> is enabled, this process copies storage parameters (<b>STOCapacity</b> and <b>STOSelfDischarge</b>) from the allocated water-tank stores to the corresponding <b>resistive heater</b> units in the Dispa-SET dictionary.<br> This ensures that resistive heaters have the correct thermal storage characteristics for the optimization.<br>
    <b>Copy Logic:</b>
            <div style="margin-left: 2em; font-size: 12px">
        For each zone, the script iterates through all power units and identifies those containing <b>"resistive"</b> in their Unit name. <br>For each matching unit, it searches the allocated country stores for an asset with the same name and copies the <b>e_nom_opt</b> (to <b>STOCapacity</b>) and <b>standing_loss</b> (to <b>STOSelfDischarge</b>) values.
    </div>
</div>

In [37]:
if VPP:
# _________________________________________________________________________________
    ### Initialize a dictionary to store summary information for each zone
    updates_summary = {}
    for zone in zone_names:
        ### Get the DataFrames for the current zone
        power_df = power_units_dispaset_dictionary[zone]
        allocated_df = allocated_country_stores[zone]
        ### Initialize counters for the current zone
        resistive_assets = 0
        updated_rows = 0
        ### Iterate over the rows of the power_df
        for index_power, row_power in power_df.iterrows():
            unit = row_power['Unit']
            ### Check if 'resistive' is in the Unit name
            if 'resistive' in unit.lower():
                resistive_assets += 1
                ### Find the matching row in allocated_df
                for index_alloc, row_alloc in allocated_df.iterrows():
                    asset_name = row_alloc['asset_name']   
                    if unit == asset_name:
                        ### Copy the values
                        power_df.at[index_power, 'STOCapacity'] = row_alloc['e_nom_opt']
                        power_df.at[index_power, 'STOSelfDischarge'] = row_alloc['standing_loss']
                        updated_rows += 1
                        break  # Exit the inner loop once the match is found
        ### Store the summary for the current zone
        updates_summary[zone] = {
            "resistive_assets": resistive_assets,
            "updated_rows": updated_rows
        }
    ### SUMMARY
    print("\n==============================================================")
    print("Storage parameters copied to DispaSET power units")
    print("==============================================================")
    print(f"Zones processed: {len(zone_names)}")
    print(f"Zone names: {zone_names}")
    total_updates = sum(info["updated_rows"] for info in updates_summary.values())
    print(f"Total updated rows: {total_updates}")
    for zone, info in updates_summary.items():
        print(f"\n{zone}:")
        print(f"  Resistive assets found: {info['resistive_assets']}")
        print(f"  Matching units updated: {info['updated_rows']}")
        ### Optional preview
        updated_df = power_units_dispaset_dictionary[zone]
        print("\n  Preview of updated data:")
        print(updated_df[["Unit", "STOCapacity", "STOSelfDischarge"]].head())


Storage parameters copied to DispaSET power units
Zones processed: 5
Zone names: ['BE', 'FR', 'DE', 'NL', 'UK']
Total updated rows: 60

BE:
  Resistive assets found: 11
  Matching units updated: 11

  Preview of updated data:
                    Unit  STOCapacity  STOSelfDischarge
0  BE1 0 offwind-ac-2030          NaN               NaN
1  BE1 0 offwind-dc-2030          NaN               NaN
2      BE1 0 onwind-2030          NaN               NaN
3              BE1 0 ror          NaN               NaN
4       BE1 0 solar-2030          NaN               NaN

FR:
  Resistive assets found: 13
  Matching units updated: 13

  Preview of updated data:
                    Unit  STOCapacity  STOSelfDischarge
0  FR1 0 offwind-ac-2030          NaN               NaN
1  FR1 0 offwind-dc-2030          NaN               NaN
2      FR1 0 onwind-2030          NaN               NaN
3              FR1 0 ror          NaN               NaN
4       FR1 0 solar-2030          NaN               NaN

DE:
  Res

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       E-06. Virtual Power Plant (VPP) Aggregation
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        When <b>VPP mode</b> is enabled, individual resistive heater units are aggregated into <b>Virtual Power Plant blocks</b>.<br> This consolidates many small, distributed units into larger, more manageable entities for the optimization while preserving their aggregate technical characteristics.<br>
    <b>Aggregation Logic:</b>
            <div style="margin-left: 2em; font-size: 12px">
        For each VPP carrier pattern (e.g., <b>residential rural resistive heater</b>), the script identifies all matching units. It then:
            <div style="margin-left: 2em;">
        <b>1.</b> Sums the <b>PowerCapacity</b> of all matching units<br>
        <b>2.</b> Calculates <b>weighted averages</b> for numeric parameters (e.g., Efficiency, MinUpTime) using PowerCapacity as weights<br>
        <b>3.</b> Creates a single aggregated row with <b>_VPP</b> suffix in the Unit name<br>
        <b>4.</b> Sets <b>Nunits = 1</b> for the consolidated block<br>
        <b>5.</b> Removes the original individual units and replaces them with the aggregated row
            </div>
            </div>
    </div>
</div>

In [38]:
power_units_dispaset_dictionary_vpp = {}
### Define capacity columns to sum up
cols_to_sum = ["PowerCapacity", "STOCapacity", "STOMaxChargingPower"]
print("=" * 80)
print("STARTING VPP AGGREGATION PROCESS")
print("=" * 80)
for country, df in power_units_dispaset_dictionary.items():
    df_copy = df.copy()
    ### Identify numeric columns for averaging, excluding summed capacity columns and Nunits
    all_numeric_cols = df_copy.select_dtypes(
        include=[np.number]
    ).columns.tolist()
    cols_to_average = [
        col
        for col in all_numeric_cols
        if col not in cols_to_sum and col != "Nunits"
    ]
    vpp_rows = []
    indices_to_drop = []
    print(f"\n--- Processing Zone / Country: {country} ---")
    ### Process each VPP category
    for pattern in vpp_resistive_heater_carriers:
        mask = df_copy["Unit"].str.contains(pattern, case=False, na=False)
        matching_df = df_copy[mask]
        if not matching_df.empty:
            indices_to_drop.extend(matching_df.index.tolist())
            ### Initialize base aggregated row from first match
            agg_row = matching_df.iloc[0].copy()
            agg_row["Unit"] = f"{pattern}_VPP"
            ### 1. SUM CAPACITIES (PowerCapacity, STOCapacity, STOMaxChargingPower)
            for col in cols_to_sum:
                if col in matching_df.columns:
                    agg_row[col] = pd.to_numeric(
                        matching_df[col], errors="coerce"
                    ).sum()
            total_capacity = agg_row["PowerCapacity"]
            ### 2. WEIGHTED AVERAGE FOR OTHER NUMERIC COLUMNS
            ### (STOSelfDischarge, EfficiencySector1, ChargingEfficiencySector1, etc.)
            for col in cols_to_average:
                col_values = pd.to_numeric(
                    matching_df[col], errors="coerce"
                ).fillna(0)
                if total_capacity > 0:
                    weighted_val = (
                        col_values * matching_df["PowerCapacity"]
                    ).sum() / total_capacity
                else:
                    weighted_val = col_values.mean()
                agg_row[col] = weighted_val
            ### 3. PRESERVE COMMON SECTOR1 VALUE
            if "Sector1" in matching_df.columns:
                valid_sectors = matching_df["Sector1"].dropna()
                agg_row["Sector1"] = (
                    valid_sectors.iloc[0] if not valid_sectors.empty else np.nan
                )
            ### Reset unit count to 1
            if "Nunits" in agg_row:
                agg_row["Nunits"] = 1
            vpp_rows.append(agg_row)
            ### --- DETAILED CONSOLE REPORTING ---
            print(
                f"  • Aggregated {len(matching_df)} unit(s) into '{agg_row['Unit']}':"
            )
            print(f"    - Constituent Units: {matching_df['Unit'].tolist()}")
            print(
                f"    - Total PowerCapacity:        {agg_row['PowerCapacity']:.2f} MW"
            )
            print(
                f"    - Total STOCapacity:         {agg_row.get('STOCapacity', 0):.2f} MWh"
            )
            print(
                f"    - Total STOMaxChargingPower: {agg_row.get('STOMaxChargingPower', 0):.2f} MW"
            )
            print(
                f"    - Weighted Efficiency:       {agg_row.get('Efficiency', np.nan):.4f}"
            )
            print(
                f"    - Weighted STOSelfDischarge: {agg_row.get('STOSelfDischarge', np.nan):.4f}"
            )
            print(
                f"    - Sector1 Value:             '{agg_row.get('Sector1', 'N/A')}'"
            )
        else:
            print(f"  • No units found matching pattern: '{pattern}'")
    ### Remove individual constituent rows from original DataFrame
    df_remaining = df_copy.drop(index=indices_to_drop)
    ### Append new VPP aggregated rows
    if vpp_rows:
        df_vpp_added = pd.DataFrame(vpp_rows)
        df_final = pd.concat([df_remaining, df_vpp_added], ignore_index=True)
    else:
        df_final = df_remaining
    power_units_dispaset_dictionary_vpp[country] = df_final
    print(
        f"  [Row Count Summary] Original: {len(df)} | Removed: {len(indices_to_drop)} | Final: {len(df_final)}"
    )
print("\n" + "=" * 80)
print("VPP AGGREGATION COMPLETE ACROSS ALL ZONES")
print("=" * 80)

STARTING VPP AGGREGATION PROCESS

--- Processing Zone / Country: BE ---
  • Aggregated 3 unit(s) into 'residential rural resistive heater_VPP':
    - Constituent Units: ['BE1 0 residential rural resistive heater-2030', 'BE1 0 residential rural resistive heater-2015', 'BE1 0 residential rural resistive heater-2019']
    - Total PowerCapacity:        56.10 MW
    - Total STOCapacity:         0.06 MWh
    - Total STOMaxChargingPower: 56.10 MW
    - Weighted Efficiency:       0.9000
    - Weighted STOSelfDischarge: 0.0138
    - Sector1 Value:             'DHW_RRRH'
  • Aggregated 3 unit(s) into 'residential urban decentral resistive heater_VPP':
    - Constituent Units: ['BE1 0 residential urban decentral resistive heater-2030', 'BE1 0 residential urban decentral resistive heater-2015', 'BE1 0 residential urban decentral resistive heater-2019']
    - Total PowerCapacity:        1204.02 MW
    - Total STOCapacity:         0.06 MWh
    - Total STOMaxChargingPower: 1204.02 MW
    - Weighted E

/tmp/ipykernel_2803280/4052447070.py:90: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_remaining, df_vpp_added], ignore_index=True)
/tmp/ipykernel_2803280/4052447070.py:90: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_remaining, df_vpp_added], ignore_index=True)
/tmp/ipykernel_2803280/4052447070.py:90: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclu

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       E-07. Single Virtual Power Plant (VPP) Consolidation
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        After the initial VPP aggregation, multiple VPP blocks (e.g., residential, urban, services) are further consolidated into a <b>single aggregated VPP unit</b>. <br>This reduces the model complexity to a single representative entity for all resistive heaters.<br>
    <b>Consolidation Logic:</b>
            <div style="margin-left: 2em; font-size: 12px">
        The script identifies all units ending with <b>_VPP</b> (created in the previous step). It then:
            <div style="margin-left: 2em;">
        <b>1.</b> Sums the total <b>PowerCapacity</b> across all VPP blocks<br>
        <b>2.</b> Calculates <b>weighted averages</b> for numeric parameters using PowerCapacity as weights<br>
        <b>3.</b> Creates a single consolidated row with <b>Unit = "Resistive_Heaters_VPP"</b><br>
        <b>4.</b> Sets <b>Nunits = 1</b><br>
        <b>5.</b> Removes the individual VPP blocks and replaces them with the single consolidated row.
            </div>
            </div>
    </div>
</div>

In [39]:
power_units_dispaset_dictionary_vpp_single = {}
### Define capacity columns to sum up across sub-VPP units
cols_to_sum = ["PowerCapacity", "STOCapacity", "STOMaxChargingPower"]
print("=" * 80)
print("STARTING SINGLE VPP CONSOLIDATION PROCESS")
print("=" * 80)
for country, df in power_units_dispaset_dictionary_vpp.items():
    df_copy = df.copy()
    ### Identify numeric columns for weighted averaging (excluding capacity sums and Nunits)
    all_numeric_cols = df_copy.select_dtypes(
        include=[np.number]
    ).columns.tolist()
    cols_to_average = [
        col
        for col in all_numeric_cols
        if col not in cols_to_sum and col != "Nunits"
    ]
    ### Match all category-level VPP rows ending with '_VPP'
    vpp_mask = df_copy["Unit"].str.endswith("_VPP", na=False)
    vpp_df = df_copy[vpp_mask]
    print(f"\n--- Processing Zone / Country: {country} ---")
    if not vpp_df.empty:
        ### Separate non-VPP units to keep them untouched
        df_remaining = df_copy[~vpp_mask].copy()
        ### Initialize single master VPP row from the first VPP match
        agg_row = vpp_df.iloc[0].copy()
        agg_row["Unit"] = "Resistive_Heaters_VPP"
        ### 1. SUM CAPACITIES (PowerCapacity, STOCapacity, STOMaxChargingPower)
        for col in cols_to_sum:
            if col in vpp_df.columns:
                agg_row[col] = pd.to_numeric(
                    vpp_df[col], errors="coerce"
                ).sum()
        total_capacity = agg_row["PowerCapacity"]
        ### 2. CAPACITY-WEIGHTED AVERAGE FOR OTHER NUMERIC COLUMNS
        ### (Efficiency, STOSelfDischarge, EfficiencySector1, ChargingEfficiencySector1, etc.)
        for col in cols_to_average:
            col_values = pd.to_numeric(vpp_df[col], errors="coerce").fillna(0)
            if total_capacity > 0:
                weighted_val = (
                    col_values * vpp_df["PowerCapacity"]
                ).sum() / total_capacity
            else:
                weighted_val = col_values.mean()
            agg_row[col] = weighted_val
        ### 3. ASSIGN UNIFIED SECTOR1 VALUE FOR THE SINGLE VPP ROW
        if "Sector1" in vpp_df.columns:
            agg_row["Sector1"] = "DHW_VPP"
        ### Reset Nunits to 1
        if "Nunits" in agg_row:
            agg_row["Nunits"] = 1
        ### Combine non-VPP units with the new single consolidated VPP row
        df_final = pd.concat(
            [df_remaining, pd.DataFrame([agg_row])], ignore_index=True
        )
        ### --- CONSOLE REPORTING ---
        print(
            f"  • Consolidated {len(vpp_df)} sub-VPP unit(s) into single 'Resistive_Heaters_VPP':"
        )
        print(f"    - Merged Units:              {vpp_df['Unit'].tolist()}")
        print(
            f"    - Total PowerCapacity:        {agg_row['PowerCapacity']:.2f} MW"
        )
        print(
            f"    - Total STOCapacity:          {agg_row.get('STOCapacity', 0):.2f} MWh"
        )
        print(
            f"    - Total STOMaxChargingPower: {agg_row.get('STOMaxChargingPower', 0):.2f} MW"
        )
        print(
            f"    - Weighted Efficiency:        {agg_row.get('Efficiency', np.nan):.4f}"
        )
        print(
            f"    - Weighted STOSelfDischarge: {agg_row.get('STOSelfDischarge', np.nan):.4f}"
        )
        print(
            f"    - Sector1 Value:              '{agg_row.get('Sector1', 'N/A')}'"
        )
    else:
        df_final = df_copy
        print("  • No '_VPP' units found to consolidate.")
    power_units_dispaset_dictionary_vpp_single[country] = df_final
    print(
        f"  [Row Count Summary] Before: {len(df)} | After Single VPP Consolidation: {len(df_final)}"
    )
print("\n" + "=" * 80)
print("SINGLE VPP CONSOLIDATION COMPLETE ACROSS ALL ZONES")
print("=" * 80)

STARTING SINGLE VPP CONSOLIDATION PROCESS

--- Processing Zone / Country: BE ---
  • Consolidated 5 sub-VPP unit(s) into single 'Resistive_Heaters_VPP':
    - Merged Units:              ['residential rural resistive heater_VPP', 'residential urban decentral resistive heater_VPP', 'services urban decentral resistive heater_VPP', 'services rural resistive heater_VPP', 'urban central resistive heater_VPP']
    - Total PowerCapacity:        4130.82 MW
    - Total STOCapacity:          89256.28 MWh
    - Total STOMaxChargingPower: 4130.82 MW
    - Weighted Efficiency:        0.9561
    - Weighted STOSelfDischarge: 0.0053
    - Sector1 Value:              'DHW_VPP'
  [Row Count Summary] Before: 35 | After Single VPP Consolidation: 31

--- Processing Zone / Country: FR ---
  • Consolidated 5 sub-VPP unit(s) into single 'Resistive_Heaters_VPP':
    - Merged Units:              ['residential rural resistive heater_VPP', 'residential urban decentral resistive heater_VPP', 'services urban decentr

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        E-08. Replacing Zero Values with NaN in Sector X-Related Columns
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process replaces <b>zero values</b> with <b>NaN</b> in all <b>Sector-related columns</b> of the <b>power_units_dispaset_dictionary_vpp_single</b>. <br>This is necessary because Dispa-SET interprets zero values as valid sector identifiers, which can lead to incorrect sector assignments. <br>Replacing them with NaN ensures that empty sectors are properly ignored.<br>
    <b>Replacement Logic:</b>
            <div style="margin-left: 2em; font-size: 12px">
        For each zone in the dictionary, the script iterates through the target columns (<b>Sector1</b>, <b>EfficiencySector1</b>, <b>ChargingEfficiencySector1</b>, <b>Sector2</b>, <b>EfficiencySector2</b>, <b>ChargingEfficiencySector2</b>). <br>If the column exists, all zero values are replaced with <b>NaN</b> using <b>df[column].replace(0, np.nan)</b>. The updated DataFrame is then stored back into the dictionary.<br>
        This ensures that sector-related columns contain only valid sector identifiers or NaN, preventing Dispa-SET from misinterpreting zero values as active sectors.
            </div>
    </div>
</div>

In [40]:
columns_to_check = [
    'Sector1',
    'EfficiencySector1',
    'ChargingEfficiencySector1',
    'Sector2',
    'EfficiencySector2',
    'ChargingEfficiencySector2'
]
for key, df in power_units_dispaset_dictionary_vpp_single.items():
    for column in columns_to_check:
        if column in df.columns:
            # Count the number of 0s before replacement
            zeros_before = (df[column] == 0).sum()
            # Replace 0 with NA
            df[column] = df[column].replace(0, np.nan)
            # Count the number of NAs after replacement
            na_after = df[column].isna().sum()
            # Update the dictionary
            power_units_dispaset_dictionary[key] = df

            if zeros_before > 0:
                print(f"Key: {key}, Column: {column} - Replaced {zeros_before} zeros with NA.")
# Summary message
print("\nReplacement complete. All 0 values in the specified columns have been replaced with NA.")

Key: BE, Column: Sector2 - Replaced 1 zeros with NA.
Key: BE, Column: EfficiencySector2 - Replaced 1 zeros with NA.
Key: BE, Column: ChargingEfficiencySector2 - Replaced 1 zeros with NA.
Key: FR, Column: Sector2 - Replaced 1 zeros with NA.
Key: FR, Column: EfficiencySector2 - Replaced 1 zeros with NA.
Key: FR, Column: ChargingEfficiencySector2 - Replaced 1 zeros with NA.
Key: DE, Column: Sector2 - Replaced 1 zeros with NA.
Key: DE, Column: EfficiencySector2 - Replaced 1 zeros with NA.
Key: DE, Column: ChargingEfficiencySector2 - Replaced 1 zeros with NA.
Key: NL, Column: Sector2 - Replaced 1 zeros with NA.
Key: NL, Column: EfficiencySector2 - Replaced 1 zeros with NA.
Key: NL, Column: ChargingEfficiencySector2 - Replaced 1 zeros with NA.
Key: UK, Column: Sector2 - Replaced 1 zeros with NA.
Key: UK, Column: EfficiencySector2 - Replaced 1 zeros with NA.
Key: UK, Column: ChargingEfficiencySector2 - Replaced 1 zeros with NA.

Replacement complete. All 0 values in the specified columns have

<div style= "background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099; " >
 <span style= "font-weight: bold; font-size: 16px; " >
Section F Overview: Power Plants Export & Routing
 </span >
 <div style= "border-top: 1px solid #000099; padding: 10px; " >
This section manages the export of the finalized power plant datasets.<br> It saves the processed DataFrames to CSV files and applies conditional routing logic, directing VPP-aggregated files to the VPP directory for specific zones while routing standard files for the remaining zones.
 </div >
 </div >
 <div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       F-01. Exporting Dispa-SET Datasets to CSV Files
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        After all processing steps (cleaning, splitting, sanitizing, and filling techno-economic features), the finalized Dispa-SET datasets are exported as <b>CSV files</b>. <br>Each country/zone is saved in its own subfolder with the target year as the filename.
        <br>
<b>Export Logic:</b>
            <div style="margin-left: 2em; font-size: 12px">
        The script iterates through all zones in the <b>power_units_dispaset_dictionary</b>. <br>For each zone, it creates a subfolder named after the zone and saves the DataFrame as <b>{data_target_year}.csv</b>. <br>The export is only performed for zones present in the <b>zone_names</b> whitelist.
            </div>
    </div>
</div>

In [41]:
### 1. Select dictionary and filename suffix based on VPP flag
if VPP:  # Accepts True or "Y" (depending on your flag type)
# _________________________________________________________________________________
    target_dict = power_units_dispaset_dictionary_vpp_single
    file_suffix = "_vpp"
else:
    target_dict = power_units_dispaset_dictionary
    file_suffix = ""
### 2. Define base export path
base_path = power_plants_pypsa_formated_data_folder_path
os.makedirs(base_path, exist_ok=True)
power_plants_pypsa_formated_data_folder_path
### 3. Process and export DataFrames
for zone, df in target_dict.items():
    if zone in zone_names:
        subfolder_path = os.path.join(base_path, zone)
        os.makedirs(subfolder_path, exist_ok=True)
        ### Construct file name with conditional suffix (e.g., '2030_vpp.csv' or '2030.csv')
        filename = f"{data_target_year}{file_suffix}.csv"
        filepath = os.path.join(subfolder_path, filename)
        df.to_csv(filepath, index=False)
        print(f"Saved Dispa-SET dataset for '{zone}' to: {filepath}")
    else:
        print(f"Zone '{zone}' not found in zone_names whitelist. Skipping.")

Saved Dispa-SET dataset for 'BE' to: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Sufficiency_Scenario/PowerPlants/BE/2030_vpp.csv
Saved Dispa-SET dataset for 'FR' to: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Sufficiency_Scenario/PowerPlants/FR/2030_vpp.csv
Saved Dispa-SET dataset for 'DE' to: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Sufficiency_Scenario/PowerPlants/DE/2030_vpp.csv
Saved Dispa-SET dataset for 'NL' to: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Sufficiency_Scenario/PowerPlants/NL/2030_vpp.csv
Saved Dispa-SET dataset for 'UK' to: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Sufficiency_Scenario/PowerPlants/UK/2030_vpp.csv


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       F-02. Conditional VPP File Copying
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        When <b>VPP mode</b> is enabled, the script copies the finalized power plant CSV files from the source directory to the destination directory, selecting between <b>VPP</b> or <b>Standard</b> files based on the zone's configuration.
    <br>
    <b>Conditional Logic:</b>
            <div style="margin-left: 2em; font-size: 12px">
        The copy operation is executed <b>only when VPP = True</b>. For each zone, the script checks if the zone is in the <b>vpp_zone_names</b> list:
        <br>
        <b>VPP Zones</b> → Copy <code>{data_target_year}_vpp.csv</code> (VPP-aggregated data)<br>
        <b>Standard Zones</b> → Copy <code>{data_target_year}.csv</code> (regular Dispa-SET data)
            </div>
    </div>
</div>

In [42]:
if VPP:
# _________________________________________________________________________________
    ### Define source and destination root paths for the data
    src_root = Path(power_plants_pypsa_formated_data_folder_path)
    dst_root = Path(vpp_power_plants_pypsa_formated_data_folder_path)
    ### Initialize a list to track copied files
    copied_files = []
    ### Copy files for each zone, differentiating between VPP and standard zones
    for zone in zone_names:
        ### Determine source file and type based on whether the zone is in VPP zones
        if zone in vpp_zone_names:
            src_file = src_root / zone / f"{data_target_year}_vpp.csv"
            source_type = "VPP"
        else:
            src_file = src_root / zone / f"{data_target_year}.csv"
            source_type = "Standard"
        ### Define destination file path
        dst_file = dst_root / zone / f"{data_target_year}.csv"
        ### Copy the file from source to destination
        shutil.copy2(src_file, dst_file)
        ### Record the copied file details
        copied_files.append(
            f"{zone}: {src_file.name} -> {dst_file.name} ({source_type})"
        )
    ### Print a summary of all copied files
    print("\nCopy Summary")
    print("-" * 50)
    for item in copied_files:
        print(item)
    print(f"\nTotal files copied: {len(copied_files)}")


Copy Summary
--------------------------------------------------
BE: 2030_vpp.csv -> 2030.csv (VPP)
FR: 2030.csv -> 2030.csv (Standard)
DE: 2030.csv -> 2030.csv (Standard)
NL: 2030.csv -> 2030.csv (Standard)
UK: 2030.csv -> 2030.csv (Standard)

Total files copied: 5


<div style= "background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099; " >
 <span style= "font-weight: bold; font-size: 16px; " >
Section G Overview: VPP Features Extraction & Assignment
 </span >
 <div style= "border-top: 1px solid #000099; padding: 10px; " >
This section extracts the specific boundary features required for the VPP optimization. <br>It populates a VPP features dictionary with aggregated storage parameters (capacity, self-discharge, max power) and assigns critical boundary conditions, such as the minimum state of charge (<i>STOMinSOC</i>) and the cost of energy not served.
 </div >
 </div >
 <div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       G-01. Creating Empty VPP Features DataFrames
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process creates <b>empty VPP features DataFrames</b> for each zone using the column structure from a template CSV file.<br> These DataFrames will later be populated with VPP-specific parameters for the optimization.
    </div>
</div>

In [43]:
### CREATE EMPTY VPP FEATURES DATAFRAMES USING CSV HEADERS
# _____________________________________________________________________________
### READ ONLY THE HEADERS FROM THE CSV
vpp_features_template_df = pd.read_csv(
    vpp_features_base_data_folder_path,
    nrows=0
)
vpp_features_columns = (
    vpp_features_template_df.columns.tolist()
)
### CREATE NEW NESTED DICTIONARY
vpp_features_dictionary = {}
### CREATE AN EMPTY DATAFRAME FOR EACH ZONE
for zone in zone_names:
    vpp_features_dictionary[zone] = pd.DataFrame(
        columns=vpp_features_columns
    )
### DISPLAY RESULTS
print("\n" + "=" * 100)
print("EMPTY VPP FEATURES DICTIONARY CREATED")
print("=" * 100)
print(
    f"Template CSV: "
    f"{vpp_features_base_data_folder_path}"
)
print(
    f"Number of columns: "
    f"{len(vpp_features_columns)}"
)
print(
    f"Columns:\n{vpp_features_columns}"
)
print("\nDataFrames:")
for zone in zone_names:
    df = vpp_features_dictionary[zone]
    print(
        f"{zone} | "
        f"Rows: {df.shape[0]} | "
        f"Columns: {df.shape[1]}"
    )
print("=" * 100)


EMPTY VPP FEATURES DICTIONARY CREATED
Template CSV: /home/ray/Dispa-SET_Unleash/Database/Boundary_Sector/BoundarySectorData/BE/2023.csv
Number of columns: 9
Columns:
['Unnamed: 0', 'Sector', 'STOCapacity', 'STOSelfDischarge', 'STOMinSOC', 'MaxFlexDemand', 'MaxFlexSupply', 'CostXNotServed', 'STOMaxPower']

DataFrames:
BE | Rows: 0 | Columns: 9
FR | Rows: 0 | Columns: 9
DE | Rows: 0 | Columns: 9
NL | Rows: 0 | Columns: 9
UK | Rows: 0 | Columns: 9


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
      G-02.  Copying VPP Storage Features from Dispa-SET Data
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process extracts <b>storage features</b> (Sector, STOCapacity, STOSelfDischarge, STOMaxPower) from the aggregated Dispa-SET DataFrames and populates the <b>VPP features dictionary</b>.<br> It handles both Sector1 and Sector2 columns, creating separate rows for each sector where present.<br>
    <b>Copy Logic:</b>
            <div style="margin-left: 2em; font-size: 12px;">
        For each zone, the script iterates through all rows in the source DataFrame (<b>power_units_dispaset_dictionary_vpp_single</b>). <br>For each row, it checks if <b>Sector1</b> and/or <b>Sector2</b> are present and valid (not NaN and not 0).<br> For each valid sector, it creates a new row in the target DataFrame with the sector name and associated storage parameters.
            </div>
    </div>
</div>

In [44]:
### COPY VPP STORAGE FEATURES FROM DISPA-SET DATA
for zone in zone_names:
    ### SOURCE DATAFRAME
    source_df = power_units_dispaset_dictionary_vpp_single[zone]
    ### TARGET DATAFRAME
    target_df = vpp_features_dictionary[zone]
    ### List to store the new rows
    new_rows = []
    ### PROCESS EACH ROW
    for _, row in source_df.iterrows():
        ### SECTOR 1
        if (
            "Sector1" in source_df.columns
            and pd.notna(row["Sector1"])
            and row["Sector1"] != 0
        ):
            new_rows.append({
                "Sector": row["Sector1"],
                "STOCapacity": row["STOCapacity"],
                "STOSelfDischarge": row["STOSelfDischarge"],
                "STOMaxPower": row["STOMaxChargingPower"]
            })
        ### SECTOR 2
        if (
            "Sector2" in source_df.columns
            and pd.notna(row["Sector2"])
            and row["Sector2"] != 0
        ):
            new_rows.append({
                "Sector": row["Sector2"],
                "STOCapacity": row["STOCapacity"],
                "STOSelfDischarge": row["STOSelfDischarge"],
                "STOMaxPower": row["STOMaxChargingPower"]
            })
    ### ADD NEW ROWS TO TARGET DATAFRAME
    if new_rows:
        new_rows_df = pd.DataFrame(new_rows)
        vpp_features_dictionary[zone] = pd.concat(
            [
                target_df,
                new_rows_df
            ],
            ignore_index=True
        )
    else:
        ### Keep the existing empty dataframe
        vpp_features_dictionary[zone] = target_df
### DISPLAY RESULTS
print("\n" + "=" * 100)
print("VPP FEATURES DICTIONARY UPDATED")
print("=" * 100)
for zone in zone_names:
    df = vpp_features_dictionary[zone]
    print(
        f"{zone} | "
        f"Rows: {df.shape[0]} | "
        f"Columns: {df.shape[1]}"
    )
    if not df.empty:
        print(
            df[
                [
                    "Sector",
                    "STOCapacity",
                    "STOSelfDischarge",
                    "STOMaxPower"
                ]
            ].head()
        )
    print("-" * 100)


VPP FEATURES DICTIONARY UPDATED
BE | Rows: 1 | Columns: 9
    Sector   STOCapacity  STOSelfDischarge  STOMaxPower
0  DHW_VPP  89256.281612          0.005342  4130.816433
----------------------------------------------------------------------------------------------------
FR | Rows: 1 | Columns: 9
    Sector   STOCapacity  STOSelfDischarge   STOMaxPower
0  DHW_VPP  16146.835682          0.013219  45546.719611
----------------------------------------------------------------------------------------------------
DE | Rows: 1 | Columns: 9
    Sector  STOCapacity  STOSelfDischarge  STOMaxPower
0  DHW_VPP  96902.73552          0.004764   15601.5829
----------------------------------------------------------------------------------------------------
NL | Rows: 1 | Columns: 9
    Sector   STOCapacity  STOSelfDischarge  STOMaxPower
0  DHW_VPP  1.328649e+06          0.013793  11424.40078
----------------------------------------------------------------------------------------------------
UK | Rows: 

/tmp/ipykernel_2803280/3520365252.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  vpp_features_dictionary[zone] = pd.concat(
/tmp/ipykernel_2803280/3520365252.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  vpp_features_dictionary[zone] = pd.concat(
/tmp/ipykernel_2803280/3520365252.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
      G-03.  Manually Assigning VPP Feature Values
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process assigns <b>constant values</b> to two critical VPP features: <b>STOMinSOC</b> (minimum state of charge) and <b>CostXNotServed</b> (cost of energy not served). These values are applied uniformly across all zones in the VPP features dictionary.<br>
    <b>Assignment Logic:</b>
            <div style="margin-left: 2em; font-size: 12px;">
        The script defines two constant values: <b>STOMinSOC_value = 0.2</b> (20% minimum storage level) and <b>CostXNotServed_value = 10000</b> (penalty cost).<br> For each zone, it retrieves the DataFrame and assigns these values to the <b>STOMinSOC</b> and <b>CostXNotServed</b> columns across all rows.
    </div>
</div>

In [45]:
### MANUALLY SET VPP FEATURE VALUES
# _____________________________________________________________________________
### VALUES TO ASSIGN
# =============================================================================
STOMinSOC_value      = 0.2
CostXNotServed_value = 10000
# =============================================================================
### APPLY VALUES TO ALL ZONES
for zone in zone_names:
    df = vpp_features_dictionary[zone]
    df["STOMinSOC"] = STOMinSOC_value
    df["CostXNotServed"] = CostXNotServed_value
### DISPLAY RESULTS
print("\n" + "=" * 100)
print("VPP FEATURES MANUALLY ASSIGNED")
print("=" * 100)
for zone in zone_names:
    df = vpp_features_dictionary[zone]
    print(
        f"{zone} | "
        f"Rows: {df.shape[0]} | "
        f"STOMinSOC: {df['STOMinSOC'].iloc[0] if not df.empty else 'No rows'} | "
        f"CostXNotServed: {df['CostXNotServed'].iloc[0] if not df.empty else 'No rows'}"
    )
print("=" * 100)


VPP FEATURES MANUALLY ASSIGNED
BE | Rows: 1 | STOMinSOC: 0.2 | CostXNotServed: 10000
FR | Rows: 1 | STOMinSOC: 0.2 | CostXNotServed: 10000
DE | Rows: 1 | STOMinSOC: 0.2 | CostXNotServed: 10000
NL | Rows: 1 | STOMinSOC: 0.2 | CostXNotServed: 10000
UK | Rows: 1 | STOMinSOC: 0.2 | CostXNotServed: 10000


<div style= "background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099; " >
 <span style= "font-weight: bold; font-size: 16px; " >
Section H Overview: VPP Features Export
 </span >
 <div style= "border-top: 1px solid #000099; padding: 10px; " >
This section finalizes the VPP workflow by exporting the populated VPP features DataFrames to CSV files. <br>The files are systematically organized into dedicated subfolders for each zone within the Dispa-SET Unleash directory structure, making them fully ready for the optimization engine.
 </div >
 </div >
 <div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       H-01. Exporting VPP Features DataFrames to CSV
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        When <b>VPP mode</b> is enabled, this process exports the populated VPP features DataFrames to <b>CSV files</b>. <br>Each zone's DataFrame is saved in a dedicated subfolder with the target year as the filename, following the same organizational structure as other exported data.<br>
    <b>Export Logic:</b>
            <div style="margin-left: 2em; font-size: 12px;">
        The script creates a base output directory (<b>VPP_features/</b>) and, for each zone, creates a subfolder named after the zone.<br> The DataFrame is then saved as <b>{data_target_year}.csv</b> with <b>index=False</b> to ensure a clean CSV format.
            </div>
    </div>
</div>

In [46]:
### EXPORT VPP FEATURES DATAFRAMES TO CSV    
if VPP:
# _________________________________________________________________________________
    ### CREATE BASE VPP FEATURES DIRECTORY
    vpp_features_output_folder_path = (
        vpp_data_pypsa_formated_data_folder_path
        / "VPP_features"
    )
    ### EXPORT ONE CSV PER ZONE    
    for zone in zone_names:
        ### CREATE ZONE DIRECTORY    
        zone_output_folder_path = (
            vpp_features_output_folder_path
            / zone
        )
        zone_output_folder_path.mkdir(
            parents=True,
            exist_ok=True
        )
        ### GET DATAFRAME    
        df = vpp_features_dictionary[zone]
        ### DEFINE CSV FILE PATH    
        csv_output_path = (
            zone_output_folder_path
            / f"{data_target_year}.csv"
        )
        ### EXPORT DATAFRAME    
        df.to_csv(
            csv_output_path,
            index=False
        )
        print(
            f"{zone} | "
            f"Rows: {df.shape[0]} | "
            f"Columns: {df.shape[1]} | "
            f"Saved: {csv_output_path}"
        )
    ### FINAL MESSAGE    
    print("\n" + "=" * 100)
    print("VPP FEATURES EXPORT COMPLETED")
    print("=" * 100)
    print(
        f"Output directory: "
        f"{vpp_features_output_folder_path}"
    )
    print(
        f"Target year: {data_target_year}"
    )
    print(
        f"Zones exported: {len(zone_names)}"
    )
    print("=" * 100)

BE | Rows: 1 | Columns: 9 | Saved: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Sufficiency_Scenario________VPP/VPP_data/VPP_features/BE/2030.csv
FR | Rows: 1 | Columns: 9 | Saved: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Sufficiency_Scenario________VPP/VPP_data/VPP_features/FR/2030.csv
DE | Rows: 1 | Columns: 9 | Saved: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Sufficiency_Scenario________VPP/VPP_data/VPP_features/DE/2030.csv
NL | Rows: 1 | Columns: 9 | Saved: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Sufficiency_Scenario________VPP/VPP_data/VPP_features/NL/2030.csv
UK | Rows: 1 | Columns: 9 | Saved: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Sufficiency_Scenario________VPP/VPP_data/VPP_features/UK/2030.csv

VPP FEATURES EXPORT COMPLETED
Output directory: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Sufficiency_Scenario________VPP/VPP_data/VPP_features
Target year: 2030
Zones exported: 5
